In [44]:
%matplotlib inline
import os
import glob
import urllib.request
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import mpl_toolkits.mplot3d.art3d as art3d
import ipywidgets as widgets
from IPython.display import display
from dataclasses import dataclass
from typing import List, Tuple, Optional, Dict
from scipy.interpolate import RectBivariateSpline

# ==============================================================================
# 1. AIRFOIL SCRAPING & 2D POLAR INTERPOLATOR
# ==============================================================================
def to_airfoiltools_slug(name: str) -> str:
    clean = name.lower().strip().replace(" ", "").replace("-", "").replace("_", "").replace("(", "").replace(")", "")
    known_mappings = {
        'clarky': 'clarky-il', 
        'clarkyh': 'clarkyh-il', 
        'goe398': 'goe398-il',
        'goe535': 'goe535-il', 
        'usa35b': 'usa35b-il',
        'boeingvertolvr12': 'vr12-il',
        'nasasc(2)0010': 'sc20010-il',
        'nasasc(2)0012': 'sc20012-il',
        'nasasc(2)0410': 'sc20410-il',
        'nasasc(2)0412': 'sc20412-il',
        'naca64a010': 'naca64a010-il',
        'naca64a012': 'n64012a-il',
        'oneraoa209': 'oa209-il',
        'oneraoa212': 'oa212-il',
        'sikorskysc1095': 'sc1095-il'
    }
    
    
    if clean in known_mappings:
        return known_mappings[clean]
    if clean.startswith("naca"):
        return f"n{clean.replace('naca', '')}-il"
    return f"{clean}-il" if not (clean.endswith("il") or clean.endswith("sa")) else clean

def ensure_airfoil_data(airfoil_name: str, airfoil_dir: str = "airfoils"):
    os.makedirs(airfoil_dir, exist_ok=True)
    slug = to_airfoiltools_slug(airfoil_name)
    target_dir = os.path.join(airfoil_dir, slug)
    os.makedirs(target_dir, exist_ok=True)

    dat_path = os.path.join(target_dir, f"{slug}.dat")
    if not os.path.exists(dat_path):
        try:
            url_dat = f"http://airfoiltools.com/airfoil/seligdatfile?airfoil={slug}"
            req = urllib.request.Request(url_dat, headers={'User-Agent': 'Mozilla/5.0'})
            with urllib.request.urlopen(req, timeout=3) as resp:
                content = resp.read().decode('utf-8')
                if len(content) > 100 and "<html>" not in content.lower():
                    with open(dat_path, 'w') as f:
                        f.write(content)
        except Exception:
            pass

    for re_v in [50000, 100000, 200000, 500000, 1000000]:
        for nc_str, nc_tag in [("", ""), ("-n5", "-n5")]:
            csv_path = os.path.join(target_dir, f"xf-{slug}-{re_v}{nc_tag}.csv")
            if not os.path.exists(csv_path):
                try:
                    url_csv = f"http://airfoiltools.com/polar/csv?polar=xf-{slug}-{re_v}{nc_str}"
                    req = urllib.request.Request(url_csv, headers={'User-Agent': 'Mozilla/5.0'})
                    with urllib.request.urlopen(req, timeout=3) as resp:
                        content = resp.read().decode('utf-8')
                        if "alpha" in content.lower() and "cl" in content.lower():
                            with open(csv_path, 'w') as f:
                                f.write(content)
                except Exception:
                    pass

def load_airfoil_coords(airfoil_name: str) -> Tuple[np.ndarray, np.ndarray]:
    ensure_airfoil_data(airfoil_name)
    slug = to_airfoiltools_slug(airfoil_name)
    for d in ["airfoils", "."]:
        for fpath in glob.glob(os.path.join(d, "**", "*.dat"), recursive=True):
            if slug.replace("-il", "") in os.path.basename(fpath).lower():
                raw_x, raw_y = [], []
                with open(fpath, 'r') as f:
                    for line in f.readlines()[1:]:
                        p = line.strip().split()
                        if len(p) == 2:
                            try:
                                raw_x.append(float(p[0]))
                                raw_y.append(float(p[1]))
                            except ValueError:
                                continue
                if len(raw_x) > 10:
                    return np.array(raw_x), np.array(raw_y)
    beta = np.linspace(0, np.pi, 30)
    x = 0.5 * (1.0 - np.cos(beta))
    yt = 5.0 * 0.12 * (0.2969 * np.sqrt(x) - 0.1260 * x - 0.3516 * (x**2) + 0.2843 * (x**3) - 0.1015 * (x**4))
    return np.concatenate([x[::-1], x[1:]]), np.concatenate([yt[::-1], -yt[1:]])

class AirfoilModel:
    def __init__(self, airfoil_name: str = "NACA 0012", ncrit: int = 9):
        self.airfoil_name = airfoil_name
        self.ncrit = ncrit
        self.has_polars = False
        self.re_list = []
        self.raw_data = {}
        ensure_airfoil_data(airfoil_name)
        self._build_polar_interpolators()

    def _build_polar_interpolators(self):
        slug = to_airfoiltools_slug(self.airfoil_name).replace("-il", "")
        for fpath in glob.glob(os.path.join("airfoils", "**", "*.csv"), recursive=True):
            if slug in os.path.basename(fpath).lower():
                try:
                    df = pd.read_csv(fpath, skiprows=10)
                    df.columns = [c.strip().lower() for c in df.columns]
                    if 'alpha' in df.columns and 'cl' in df.columns and 'cd' in df.columns:
                        re_val = float(os.path.basename(fpath).split("-")[-1].replace(".csv", "").replace("n5", ""))
                        self.raw_data[re_val] = df[['alpha', 'cl', 'cd']].rename(columns={'cl': 'CL', 'cd': 'CD'})
                except Exception:
                    pass

        if self.raw_data:
            self.re_list = sorted(self.raw_data.keys())
            alpha_common = np.linspace(np.radians(-15.0), np.radians(20.0), 71)
            cl_grid = np.zeros((len(alpha_common), len(self.re_list)))
            cd_grid = np.zeros((len(alpha_common), len(self.re_list)))
            for j, re_v in enumerate(self.re_list):
                df = self.raw_data[re_v]
                a_rad = np.radians(df['alpha'].values.astype(float))
                cl_grid[:, j] = np.interp(alpha_common, a_rad, df['CL'].values.astype(float))
                cd_grid[:, j] = np.interp(alpha_common, a_rad, df['CD'].values.astype(float))
            self.spline_cl = RectBivariateSpline(alpha_common, np.log10(self.re_list), cl_grid, kx=2, ky=min(len(self.re_list)-1, 2))
            self.spline_cd = RectBivariateSpline(alpha_common, np.log10(self.re_list), cd_grid, kx=2, ky=min(len(self.re_list)-1, 2))
            self.has_polars = True

    def evaluate(self, alpha_rad: np.ndarray, re_arr: np.ndarray, mach_arr: np.ndarray) -> Tuple[np.ndarray, np.ndarray, np.ndarray]:
        if self.has_polars:
            # Clamp Re to polar data range to prevent wild extrapolation
            re_lo = np.log10(min(self.re_list))
            re_hi = np.log10(max(self.re_list))
            log_re = np.clip(np.log10(np.clip(re_arr, 1e4, 1e8)), re_lo, re_hi)
            cl = self.spline_cl.ev(alpha_rad, log_re)
            cd = self.spline_cd.ev(alpha_rad, log_re)
            # Prandtl-Glauert compressibility correction for Cd (turbulent skin friction increases at high Re)
            # Add a Re-scaled Cd floor so high-Re blades don't have unrealistically low drag
            re_actual = np.clip(re_arr, 1e5, 2e7)
            cd_turbulent_floor = 0.074 / (re_actual ** 0.2) * (np.pi * 1.0)  # flat plate approximation
            cd = np.maximum(cd, cd_turbulent_floor * 0.15)
        else:
            cl = 5.75 * alpha_rad
            cd = 0.011 + 1.25 * (alpha_rad ** 2)
        beta = np.sqrt(np.maximum(1e-4, 1.0 - np.clip(mach_arr, 0.0, 0.95)**2))
        return cl / beta, cd / beta, np.abs(alpha_rad) > np.radians(14.0)

# ==============================================================================
# 2. BEMT AEROMECHANICAL SOLVER
# ==============================================================================
@dataclass
class BEMTResult:
    thrust_N: float
    torque_Nm: float
    power_kW: float
    CT: float
    CP: float
    FM: float
    eta_prop: float
    r_stations: np.ndarray
    chords: np.ndarray
    thetas_deg: np.ndarray
    phi_deg: np.ndarray
    alpha_deg: np.ndarray
    lambda_i: np.ndarray
    dCT_dr: np.ndarray
    dCP_dr: np.ndarray
    lambda_tot: np.ndarray = None
    cl: np.ndarray = None
    cd: np.ndarray = None
    re_r: np.ndarray = None
    dt_dr: np.ndarray = None
    dp_dr: np.ndarray = None
    dq_dr: np.ndarray = None
    mach_helical: np.ndarray = None

def run_bemt_solver(radius: float, root_cutout: float, num_blades: int, c_root: float, taper: float,
                    pitch_75_deg: float, twist_deg: float, rpm: float, v_axial_ms: float,
                    airfoil: AirfoilModel, rho: float = 1.225, a_sound: float = 340.3, mu: float = 1.789e-5, num_elements: int = 35) -> BEMTResult:
    r_edges = np.linspace(root_cutout, radius, num_elements + 1)
    r_stations = 0.5 * (r_edges[:-1] + r_edges[1:])
    dr = np.diff(r_edges)
    r_norm = r_stations / radius

    chords = c_root + (c_root * taper - c_root) * ((r_stations - root_cutout) / max(1e-4, radius - root_cutout))
    thetas = np.radians(pitch_75_deg + twist_deg * (r_norm - 0.75))

    omega = rpm * 2.0 * np.pi / 60.0
    v_tip = omega * radius
    lambda_c = v_axial_ms / max(v_tip, 1e-4)
    sigma_r = (num_blades * chords) / (np.pi * radius)
    mach_r = (omega * r_stations) / a_sound

    re_mid = (
        rho
        * (omega * 0.75 * radius)
        * (c_root * (1.0 + taper) * 0.5)
        / max(mu, 1e-6)
    )
    cl_slope = (
        airfoil.get_cl_slope(re_mid)
        if hasattr(airfoil, "get_cl_slope")
        else 5.73
    )
    
    # Correct initial induction guess: BEMT hover solution (no sign(theta) error)
    lambda_i_init = np.sqrt(np.maximum(1e-8, (sigma_r * cl_slope / 16.0)**2 + sigma_r * cl_slope * np.maximum(thetas, 0) * r_norm / 8.0)) - (sigma_r * cl_slope / 16.0)
    lambda_tot = lambda_c + np.clip(lambda_i_init, 0.0, 0.5)

    phi = np.zeros_like(r_stations)
    alpha = np.zeros_like(r_stations)

    for _ in range(60):
        u_t = omega * r_stations
        u_p = lambda_tot * v_tip
        phi = np.arctan2(u_p, u_t)
        alpha = thetas - phi
        f = (num_blades / 2.0) * (1.0 - r_norm) / np.maximum(np.abs(lambda_tot), 1e-4)
        f_loss = np.maximum((2.0 / np.pi) * np.arccos(np.exp(-np.clip(f, 0.0, 35.0))), 1e-3)
        # Correct BEMT: solve lambda_i from dT equation using actual alpha via Cl(alpha)
        term1 = (sigma_r * cl_slope) / (16.0 * f_loss)
        # Use actual geometric pitch (thetas) corrected for local inflow
        lambda_new = (np.sqrt(np.maximum(1e-8, term1**2 + sigma_r * cl_slope * thetas * r_norm / (8.0 * f_loss))) - term1) + lambda_c
        # Clamp to prevent runaway on negatively-pitched elements (windmilling region)
        lambda_tot = 0.5 * lambda_tot + 0.5 * np.clip(lambda_new, lambda_c - 0.08, 0.55)

    u_res_sq = (omega * r_stations)**2 + (lambda_tot * v_tip)**2
    re_r = (rho * np.sqrt(u_res_sq) * chords) / max(mu, 1e-6)
    cl, cd, _ = airfoil.evaluate(alpha, re_r, mach_r)

    dt_dr = num_blades * 0.5 * rho * u_res_sq * chords * (cl * np.cos(phi) - cd * np.sin(phi))
    dfx_dr = num_blades * 0.5 * rho * u_res_sq * chords * (cd * np.cos(phi) + cl * np.sin(phi))
    thrust = np.sum(dt_dr * dr)
    torque = np.sum(r_stations * dfx_dr * dr)
    power = omega * torque

    disk_area = np.pi * (radius ** 2)
    ct = thrust / max(rho * disk_area * (v_tip ** 2), 1e-6)
    cp = power / max(rho * disk_area * (v_tip ** 3), 1e-6)
    
    # --- PHYSICAL HOVER FM & CRUISE PROPULSIVE EFFICIENCY AUDIT ---
    if ct > 1e-4 and cp > 1e-5 and v_axial_ms == 0.0:
        # Ideal momentum theory power vs actual shaft power
        p_ideal_w = (max(thrust, 0.0) ** 1.5) / np.sqrt(2.0 * rho * disk_area)
        fm_raw = p_ideal_w / max(power, 1e-3)
        # Bounded by theoretical limit (real-world maximum for proprotors is ~0.82)
        fm = float(np.clip(fm_raw, 0.0, 0.85))
    else:
        fm = 0.0

    eta = float(np.clip((thrust * v_axial_ms) / (power + 1e-6), 0.0, 0.92)) if v_axial_ms > 5.0 else 0.0

    return BEMTResult(
        thrust_N=thrust, torque_Nm=torque, power_kW=power * 1e-3, CT=ct, CP=cp, FM=fm, eta_prop=eta,
        r_stations=r_stations, chords=chords, thetas_deg=np.degrees(thetas), phi_deg=np.degrees(phi),
        alpha_deg=np.degrees(alpha), lambda_i=(lambda_tot - lambda_c), lambda_tot=lambda_tot,
        cl=cl, cd=cd, re_r=re_r, dt_dr=dt_dr, dp_dr=(omega * r_stations * dfx_dr), dq_dr=(r_stations * dfx_dr),
        dCT_dr=dt_dr / (rho * (omega**2) * (radius**4) * np.pi * dr / radius),
        dCP_dr=(omega * r_stations * dfx_dr) / (rho * (omega**3) * (radius**5) * np.pi * dr / radius),
        mach_helical=mach_r
    )
   

print("CELL 1 EXECUTED: Airfoil Model & BEMT Solver Engine Ready.")

CELL 1 EXECUTED: Airfoil Model & BEMT Solver Engine Ready.


In [45]:

# # ==============================================================================
# # CELL 2: TILTROTOR DESIGNER, 4-PANEL HEATMAPS, 3D BEMT ANALYZER & REGRESSIONS
# # ==============================================================================
# import os
# import glob
# import urllib.request
# from dataclasses import dataclass
# from typing import Dict, List, Optional, Tuple

# import numpy as np
# import pandas as pd
# import matplotlib.pyplot as plt
# import mpl_toolkits.mplot3d.art3d as art3d
# import ipywidgets as widgets
# from IPython.display import display, HTML
# from scipy.interpolate import RectBivariateSpline
# from scipy.optimize import curve_fit
# from matplotlib.lines import Line2D
# from matplotlib.patches import Patch

# # ------------------------------------------------------------------------------
# # 1. ATMOSPHERE & HISTORICAL BENCHMARK REGRESSIONS
# # ------------------------------------------------------------------------------
# FUSE_LEN = 15.20
# FUSE_W = 2.15
# FUSE_H = 2.05
# PAYLOAD_FIXED = 1400.0
# N_ROTORS = 2

# d_eq = np.sqrt((4.0 / np.pi) * FUSE_W * FUSE_H)
# fineness = max(FUSE_LEN / d_eq, 3.0)
# S_WET_FUSE = np.pi * d_eq * FUSE_LEN * (1.0 - 2.0 / fineness) ** (2 / 3) * (1.0 + 1.0 / (fineness**2))

# def isa_atmosphere(altitude_m: float) -> tuple[float, float, float, float]:
#   alt = np.clip(altitude_m, 0.0, 11000.0)
#   T = 288.15 - 0.0065 * alt
#   P = 101325.0 * ((T / 288.15) ** 5.2561)
#   rho = P / (287.058 * T)
#   a = np.sqrt(1.4 * 287.058 * T)
#   # Sutherland's Law for dynamic viscosity mu [Pa*s]
#   mu = 1.789e-5 * ((T / 288.15) ** 1.5) * ((288.15 + 110.4) / (T + 110.4))
#   return rho, a, T, mu

# benchmarks = pd.DataFrame({
#     "Aircraft": ["Bell XV-3", "Curtiss-Wright X-19", "CL-84 Dynavert", "Bell XV-15", "Leonardo AW609", "Bell V-280 Valor", "LTV XC-142", "Bell Boeing V-22"],
#     "MTOW_kg": [2177.0, 6196.0, 5715.0, 6000.0, 8165.0, 14000.0, 20185.0, 23982.0],
#     "Empty_kg": [1648.0, 4527.0, 3818.0, 4570.0, 4765.0, 8200.0, 10250.0, 15032.0],
#     "Disc_Loading": [32.5, 145.0, 102.0, 73.2, 88.4, 95.0, 135.0, 102.5],
#     "Solidity": [0.053, 0.115, 0.102, 0.089, 0.096, 0.100, 0.110, 0.105],
#     "Installed_PW": [0.154, 0.410, 0.440, 0.385, 0.354, 0.533, 0.450, 0.380],
#     "Service_Ceiling_m": [3600.0, 5330.0, 4880.0, 8840.0, 7620.0, 8500.0, 7600.0, 7620.0],
#     "Wing_tc": [15.0, 19.0, 16.0, 23.0, 21.0, 20.0, 18.0, 23.0]
# })
# benchmarks["We_W0"] = benchmarks["Empty_kg"] / benchmarks["MTOW_kg"]

# def power_law(x, a, b):
#     return a * (x ** b)

# def linear_law(x, a, b):
#     return a * x + b

# popt_we, _ = curve_fit(power_law, benchmarks["MTOW_kg"].values, benchmarks["We_W0"].values, p0=[1.5, -0.09])
# popt_pw, _ = curve_fit(power_law, benchmarks["Disc_Loading"].values, benchmarks["Installed_PW"].values, p0=[0.05, 0.5])
# popt_sol, _ = curve_fit(linear_law, benchmarks["Disc_Loading"].values, benchmarks["Solidity"].values, p0=[0.001, 0.02])
# popt_ceil, _ = curve_fit(linear_law, benchmarks["Service_Ceiling_m"].values, benchmarks["Installed_PW"].values, p0=[5e-5, 0.1])

# # ------------------------------------------------------------------------------
# # 2. AIRFOIL SCRAPER & HIGH-FIDELITY BEMT SOLVER
# # ------------------------------------------------------------------------------
# KNOWN_MAPPINGS = {
#     'clarky': 'clarky-il',
#     'clarkyh': 'clarkyh-il',
#     'goe398': 'goe398-il',
#     'goe535': 'goe535-il',
#     'usa35b': 'usa35b-il',
#     's1223rtl': 's1223rtl-il',
#     'fx63137': 'fx63137-il',
#     'fx74cl5140': 'fx74cl5140-il',
#     'vr12': 'vr12-il',
#     'boeingvertolvr12': 'vr12-il',
#     'boeingvertolvr12airfoil': 'vr12-il',
#     'naca64a010': 'naca64a010-il',
#     '64a010': 'naca64a010-il',
#     'naca64a012': 'n64012a-il',
#     '64a012': 'n64012a-il',
#     'naca23012': 'n23012-il',
# }

# def to_airfoiltools_slug(name: str) -> str:
#     clean = name.lower().strip().replace(" ", "").replace("-", "").replace("_", "")
#     if clean in KNOWN_MAPPINGS:
#         return KNOWN_MAPPINGS[clean]
#     if clean.startswith("naca"):
#         return f"n{clean.replace('naca', '')}-il"
#     return f"{clean}-il" if not (clean.endswith("il") or clean.endswith("sa")) else clean

# def ensure_airfoil_data(airfoil_name: str, airfoil_dir: str = "airfoils"):
#     os.makedirs(airfoil_dir, exist_ok=True)
#     slug = to_airfoiltools_slug(airfoil_name)
#     target_dir = os.path.join(airfoil_dir, slug)
#     os.makedirs(target_dir, exist_ok=True)
#     dat_path = os.path.join(target_dir, f"{slug}.dat")
#     if not os.path.exists(dat_path):
#         try:
#             url_dat = f"http://airfoiltools.com/airfoil/seligdatfile?airfoil={slug}"
#             req = urllib.request.Request(url_dat, headers={'User-Agent': 'Mozilla/5.0'})
#             with urllib.request.urlopen(req, timeout=3) as resp:
#                 content = resp.read().decode('utf-8')
#                 if len(content) > 100 and "<html>" not in content.lower():
#                     with open(dat_path, 'w') as f:
#                         f.write(content)
#         except Exception:
#             pass

#     for re_v in [50000, 100000, 200000, 500000, 1000000]:
#         for nc_str, nc_tag in [("", ""), ("-n5", "-n5")]:
#             csv_path = os.path.join(target_dir, f"xf-{slug}-{re_v}{nc_tag}.csv")
#             if not os.path.exists(csv_path):
#                 try:
#                     url_csv = f"http://airfoiltools.com/polar/csv?polar=xf-{slug}-{re_v}{nc_str}"
#                     req = urllib.request.Request(url_csv, headers={'User-Agent': 'Mozilla/5.0'})
#                     with urllib.request.urlopen(req, timeout=3) as resp:
#                         content = resp.read().decode('utf-8')
#                         if "alpha" in content.lower() and "cl" in content.lower():
#                             with open(csv_path, 'w') as f:
#                                 f.write(content)
#                 except Exception:
#                     pass

# def load_airfoil_coords(airfoil_name: str) -> Tuple[np.ndarray, np.ndarray]:
#     ensure_airfoil_data(airfoil_name)
#     slug = to_airfoiltools_slug(airfoil_name)
#     for d in ["airfoils", "."]:
#         for fpath in glob.glob(os.path.join(d, "**", "*.dat"), recursive=True):
#             if slug.replace("-il", "") in os.path.basename(fpath).lower():
#                 raw_x, raw_y = [], []
#                 with open(fpath, 'r') as f:
#                     for line in f.readlines()[1:]:
#                         p = line.strip().split()
#                         if len(p) == 2:
#                             try:
#                                 raw_x.append(float(p[0]))
#                                 raw_y.append(float(p[1]))
#                             except ValueError:
#                                 continue
#                 if len(raw_x) > 10:
#                     return np.array(raw_x), np.array(raw_y)
#     beta = np.linspace(0, np.pi, 30)
#     x = 0.5 * (1.0 - np.cos(beta))
#     yt = 5.0 * 0.12 * (0.2969 * np.sqrt(x) - 0.1260 * x - 0.3516 * (x**2) + 0.2843 * (x**3) - 0.1015 * (x**4))
#     return np.concatenate([x[::-1], x[1:]]), np.concatenate([yt[::-1], -yt[1:]])

# class AirfoilModel:
#     def __init__(self, airfoil_name: str = "NACA 0012", ncrit: int = 9):
#         self.airfoil_name = airfoil_name
#         self.ncrit = ncrit
#         self.has_polars = False
#         self.raw_data = {}
#         ensure_airfoil_data(airfoil_name)
#         self._build_polar_interpolators()

#     def _build_polar_interpolators(self):
#         slug = to_airfoiltools_slug(self.airfoil_name).replace("-il", "")
#         for fpath in glob.glob(os.path.join("airfoils", "**", "*.csv"), recursive=True):
#             if slug in os.path.basename(fpath).lower():
#                 try:
#                     df = pd.read_csv(fpath, skiprows=10)
#                     df.columns = [c.strip().lower() for c in df.columns]
#                     if 'alpha' in df.columns and 'cl' in df.columns and 'cd' in df.columns:
#                         re_val = float(os.path.basename(fpath).split("-")[-1].replace(".csv", "").replace("n5", ""))
#                         self.raw_data[re_val] = df[['alpha', 'cl', 'cd']].rename(columns={'cl': 'CL', 'cd': 'CD'})
#                 except Exception:
#                     pass

#         if self.raw_data:
#             self.re_list = sorted(self.raw_data.keys())
#             alpha_common = np.linspace(np.radians(-15.0), np.radians(20.0), 71)
#             cl_grid = np.zeros((len(alpha_common), len(self.re_list)))
#             cd_grid = np.zeros((len(alpha_common), len(self.re_list)))
#             for j, re_v in enumerate(self.re_list):
#                 df = self.raw_data[re_v]
#                 a_rad = np.radians(df['alpha'].values.astype(float))
#                 cl_grid[:, j] = np.interp(alpha_common, a_rad, df['CL'].values.astype(float))
#                 cd_grid[:, j] = np.interp(alpha_common, a_rad, df['CD'].values.astype(float))
#             self.spline_cl = RectBivariateSpline(
#                 alpha_common,
#                 np.log10(self.re_list),
#                 cl_grid,
#                 kx=2,
#                 ky=min(len(self.re_list) - 1, 2),
#             )
#             self.spline_cd = RectBivariateSpline(
#                 alpha_common,
#                 np.log10(self.re_list),
#                 cd_grid,
#                 kx=2,
#                 ky=min(len(self.re_list) - 1, 2),
#             )
#             self.has_polars = True

#     def get_cl_slope(self, re_val: float = 500000.0) -> float:
#       if self.has_polars:
#         log_re = np.log10(np.clip(re_val, 1e4, 1e7))
#         a1, a2 = np.radians(-2.0), np.radians(4.0)
#         cl1 = float(self.spline_cl.ev(a1, log_re))
#         cl2 = float(self.spline_cl.ev(a2, log_re))
#         slope = (cl2 - cl1) / (a2 - a1)
#         return float(np.clip(slope, 4.5, 6.8))
#       return 5.73

#     def evaluate(self, alpha_rad: np.ndarray, re_arr: np.ndarray, mach_arr: np.ndarray) -> Tuple[np.ndarray, np.ndarray, np.ndarray]:
#         if self.has_polars:
#             log_re = np.log10(np.clip(re_arr, 1e4, 1e7))
#             cl = self.spline_cl.ev(alpha_rad, log_re)
#             cd = self.spline_cd.ev(alpha_rad, log_re)
#         else:
#             cl = 5.75 * alpha_rad
#             cd = 0.011 + 1.25 * (alpha_rad ** 2)
#         beta = np.sqrt(np.maximum(1e-4, 1.0 - np.clip(mach_arr, 0.0, 0.95)**2))
#         return cl / beta, cd / beta, np.abs(alpha_rad) > np.radians(14.0)

# @dataclass
# class BEMTResult:
#     thrust_N: float
#     torque_Nm: float
#     power_kW: float
#     CT: float
#     CP: float
#     FM: float
#     eta_prop: float
#     r_stations: np.ndarray
#     chords: np.ndarray
#     thetas_deg: np.ndarray
#     phi_deg: np.ndarray
#     alpha_deg: np.ndarray
#     lambda_i: np.ndarray
#     lambda_tot: np.ndarray
#     cl: np.ndarray
#     cd: np.ndarray
#     re_r: np.ndarray
#     dt_dr: np.ndarray
#     dp_dr: np.ndarray
#     dq_dr: np.ndarray
#     dCT_dr: np.ndarray
#     dCP_dr: np.ndarray
#     mach_helical: np.ndarray

# def run_bemt_solver(radius: float, root_cutout: float, num_blades: int, c_root: float, taper: float,
#                     pitch_75_deg: float, twist_deg: float, rpm: float, v_axial_ms: float,
#                     airfoil: AirfoilModel, rho: float = 1.225, a_sound: float = 340.3,mu: float = 1.789e-5, num_elements: int = 24) -> BEMTResult:
#     r_edges = np.linspace(root_cutout, radius, num_elements + 1)
#     r_stations = 0.5 * (r_edges[:-1] + r_edges[1:])
#     dr = np.diff(r_edges)
#     r_norm = r_stations / radius

#     chords = c_root + (c_root * taper - c_root) * ((r_stations - root_cutout) / max(1e-4, radius - root_cutout))
#     thetas = np.radians(pitch_75_deg + twist_deg * (r_norm - 0.75))

#     omega = rpm * (np.pi / 30.0)
#     v_tip = max(omega * radius, 1e-4)
#     lambda_c = v_axial_ms / v_tip
#     sigma_r = (num_blades * chords) / (np.pi * radius)

#     re_mid = (
#         rho
#         * (omega * 0.75 * radius)
#         * (c_root * (1.0 + taper) * 0.5)
#         / max(mu, 1e-6)
#     )
#     cl_slope = (
#         airfoil.get_cl_slope(re_mid)
#         if hasattr(airfoil, 'get_cl_slope')
#         else 5.73
#     )
#     term_ind = sigma_r * cl_slope / 16.0
#     lambda_i = np.sign(thetas) * np.sqrt(np.abs(term_ind**2 + (sigma_r * cl_slope * thetas * r_norm / 8.0))) - term_ind
#     lambda_tot = lambda_c + np.clip(lambda_i, -0.2, 0.5)

#     u_t = omega * r_stations
#     for _ in range(16):
#         f = (num_blades * 0.5) * (1.0 - r_norm) / np.maximum(np.abs(lambda_tot), 1e-4)
#         f_loss = np.maximum((2.0 / np.pi) * np.arccos(np.exp(-np.clip(f, 0.0, 18.0))), 1e-3)
#         t1 = (sigma_r * cl_slope) / (16.0 * f_loss)
#         t2 = (sigma_r * cl_slope * thetas * r_norm) / (8.0 * f_loss)
#         lambda_new = np.sign(thetas) * (np.sqrt(np.maximum(1e-6, t1**2 + np.abs(t2))) - t1) + lambda_c
#         lambda_tot = 0.6 * lambda_tot + 0.4 * lambda_new

#     u_p = lambda_tot * v_tip
#     phi = np.arctan2(u_p, u_t)
#     alpha = thetas - phi

#     u_res_sq = u_t**2 + u_p**2
#     mach_r = np.sqrt(u_res_sq) / a_sound
#     re_r = (rho * np.sqrt(u_res_sq) * chords) / mu
#     cl, cd, _ = airfoil.evaluate(alpha, re_r, mach_r)

#     q_blade = num_blades * 0.5 * rho * u_res_sq * chords
#     cos_p, sin_p = np.cos(phi), np.sin(phi)
#     dt_dr = q_blade * (cl * cos_p - cd * sin_p)
#     dfx_dr = q_blade * (cd * cos_p + cl * sin_p)
#     thrust = np.sum(dt_dr * dr)
#     torque = np.sum(r_stations * dfx_dr * dr)
#     power = omega * torque

#     disk_area = np.pi * (radius ** 2)
#     ct = thrust / max(rho * disk_area * (v_tip ** 2), 1e-6)
#     cp = power / max(rho * disk_area * (v_tip ** 3), 1e-6)
    
#     # --- PHYSICAL HOVER FM & CRUISE PROPULSIVE EFFICIENCY AUDIT ---
#     if ct > 1e-4 and cp > 1e-5 and v_axial_ms == 0.0:
#         # Ideal momentum theory power vs actual shaft power
#         p_ideal_w = (max(thrust, 0.0) ** 1.5) / np.sqrt(2.0 * rho * disk_area)
#         fm_raw = p_ideal_w / max(power, 1e-3)
#         # Bounded by theoretical limit (real-world maximum for proprotors is ~0.82)
#         fm = float(np.clip(fm_raw, 0.0, 0.85))
#     else:
#         fm = 0.0

#     eta = float(np.clip((thrust * v_axial_ms) / (power + 1e-6), 0.0, 0.92)) if v_axial_ms > 5.0 else 0.0

#     return BEMTResult(
#         thrust_N=thrust, torque_Nm=torque, power_kW=power * 1e-3, CT=ct, CP=cp, FM=fm, eta_prop=eta,
#         r_stations=r_stations, chords=chords, thetas_deg=np.degrees(thetas), phi_deg=np.degrees(phi),
#         alpha_deg=np.degrees(alpha), lambda_i=(lambda_tot - lambda_c), lambda_tot=lambda_tot,
#         cl=cl, cd=cd, re_r=re_r, dt_dr=dt_dr, dp_dr=(omega * r_stations * dfx_dr), dq_dr=(r_stations * dfx_dr),
#         dCT_dr=dt_dr / (rho * (omega**2) * (radius**4) * np.pi * dr / radius),
#         dCP_dr=(omega * r_stations * dfx_dr) / (rho * (omega**3) * (radius**5) * np.pi * dr / radius),
#         mach_helical=mach_r
#     )
   
# # ------------------------------------------------------------------------------
# # 3. WIDGET CONTROLS (COMPACT MARGINS & HORIZONTAL MODE SELECTOR)
# # ------------------------------------------------------------------------------

# style_w = {'description_width': '160px'}
# layout_w = widgets.Layout(width='340px', margin='1px 0px')

# full_airfoil_catalog = [
#     "Boeing-Vertol VR-12",
#     "NASA SC(2)-0010", "NASA SC(2)-0012", "NASA SC(2)-0410", "NASA SC(2)-0412",
#     "NACA 0009", "NACA 0012", "NACA 23012", "NACA 23015", "NACA 64-A010", "NACA 64-A012",
#     "ONERA OA209", "ONERA OA212", "Sikorsky SC1095",
#     "NACA 0015", "NACA 0018", "NACA 2412", "NACA 4412", "NACA 4415",
#     "Clark Y", "Clark YH", "USA 35B"
# ]

# w_airfoil = widgets.Dropdown(options=full_airfoil_catalog, value="Boeing-Vertol VR-12", description='Rotor Airfoil:', style=style_w, layout=layout_w)
# w_radius = widgets.FloatSlider(value=4.42, min=2.5, max=6.5, step=0.001, description='Rotor Radius R [m]:', style=style_w, layout=layout_w, continuous_update=False)
# w_nblades = widgets.IntSlider(value=4, min=2, max=6, step=1, description='Number of Blades Nb:', style=style_w, layout=layout_w, continuous_update=False)
# w_chord = widgets.FloatSlider(value=0.75, min=0.15, max=0.8, step=0.01, description='Root Chord c_0 [m]:', style=style_w, layout=layout_w, continuous_update=False)
# w_taper = widgets.FloatSlider(value=0.552, min=0.30, max=1.00, step=0.05, description='Blade Taper (c_tip/c_root):', style=style_w, layout=layout_w, continuous_update=False)
# w_pitch_hov = widgets.FloatSlider(value=6, min=0.0, max=25.0, step=0.01, description=r'Hover Pitch $theta_{0.75}$ [°]:', style=style_w, layout=layout_w, continuous_update=False)
# w_pitch_cr = widgets.FloatSlider(value=70, min=20.0, max=70.0, step=0.01, description=r'Cruise Pitch $theta_{0.75}$ [°]:', style=style_w, layout=layout_w, continuous_update=False)
# w_twist = widgets.FloatSlider(value=-20.0, min=-45.0, max=0.0, step=0.01, description='Blade Twist [°]:', style=style_w, layout=layout_w, continuous_update=False)
# w_rpm_hov = widgets.FloatSlider(value=540.0, min=300.0, max=650.0, step=5.0, description='Hover RPM:', style=style_w, layout=layout_w, continuous_update=False)
# w_rpm_cr = widgets.FloatSlider(value=200.0, min=220.0, max=500.0, step=5.0, description='Cruise RPM:', style=style_w, layout=layout_w, continuous_update=False)

# w_s_wing = widgets.FloatSlider(value=26.0, min=12.0, max=50.0, step=0.5, description='Wing Area [m²]:', style=style_w, layout=layout_w, continuous_update=False)
# w_ar = widgets.FloatSlider(value=7.8, min=4.0, max=14.0, step=0.1, description='Wing Aspect Ratio:', style=style_w, layout=layout_w, continuous_update=False)
# w_sweep = widgets.FloatSlider(value=2.5, min=-5.0, max=25.0, step=0.5, description='Wing Sweep [°]:', style=style_w, layout=layout_w, continuous_update=False)
# w_tc = widgets.FloatSlider(value=22.5, min=14.0, max=28.0, step=0.5, description='Wing Root t/c [%]:', style=style_w, layout=layout_w, continuous_update=False)

# w_v_cruise = widgets.FloatSlider(value=450.0, min=320.0, max=550.0, step=10.0, description='Cruise Speed [km/h]:', style=style_w, layout=layout_w, continuous_update=False)
# w_range = widgets.FloatSlider(value=1000.0, min=500.0, max=2000.0, step=50.0, description='Mission Range [km]:', style=style_w, layout=layout_w, continuous_update=False)
# w_ceiling = widgets.FloatSlider(value=7000.0, min=4000.0, max=9000.0, step=250.0, description='Service Ceiling [m]:', style=style_w, layout=layout_w, continuous_update=False)
# w_p_inst = widgets.FloatSlider(value=5000.0, min=2000.0, max=9000.0, step=100.0, description='Installed Power [kW]:', style=style_w, layout=layout_w, continuous_update=False)
# w_sfc = widgets.FloatSlider(value=0.285, min=0.18, max=0.45, step=0.005, description='Engine SFC [kg/kW/hr]:', style=style_w, layout=layout_w, continuous_update=False)

# # Horizontal Toggle Selector for compact vertical profile
# w_bemt_mode = widgets.ToggleButtons(
#     options=['Hover', 'Cruise', 'Both'],
#     value='Both',
#     description='Plot Mode:',
#     style={'description_width': '80px', 'button_width': '80px'},
#     layout=widgets.Layout(margin='2px 0px 6px 0px')
# )

# out_cad = widgets.Output()
# out_heatmaps = widgets.Output()
# out_blade_3d = widgets.Output()
# out_stats = widgets.Output()

# tabs_designer = widgets.Tab(
#     children=[out_cad, out_heatmaps, out_blade_3d, out_stats],
#     layout=widgets.Layout(flex='1 1 auto', min_width='800px', margin='0 0 0 16px')
# )
# tabs_designer.set_title(0, 'General Arrangement & Card')
# tabs_designer.set_title(1, 'Feasibility Heatmap Matrix')
# tabs_designer.set_title(2, '3D Lofted Blade & BEMT Analysis')
# tabs_designer.set_title(3, 'Statistical Regressions')

# _airfoil_cache = {}

# plt.ioff()
# def update_designer_dashboard(*args):
#     plt.ioff()
#     selected_af = w_airfoil.value
#     if selected_af not in _airfoil_cache:
#         _airfoil_cache[selected_af] = AirfoilModel(airfoil_name=selected_af, ncrit=9)
#     af_model = _airfoil_cache[selected_af]

#     R = float(w_radius.value)
#     Nb = int(w_nblades.value)
#     c_0 = float(w_chord.value)
#     taper = float(w_taper.value)
#     th_hov = float(w_pitch_hov.value)
#     th_cr = float(w_pitch_cr.value)
#     twist = float(w_twist.value)
#     rpm_hov = float(w_rpm_hov.value)
#     rpm_cr = float(w_rpm_cr.value)
#     S_w = float(w_s_wing.value)
#     AR = float(w_ar.value)
#     sweep_deg = float(w_sweep.value)
#     tc_pct = float(w_tc.value)
#     v_cr_kmh = float(w_v_cruise.value)
#     v_cr_ms = v_cr_kmh / 3.6
#     range_m = float(w_range.value) * 1000.0
#     Ceiling_m = float(w_ceiling.value)
#     P_inst_kW = float(w_p_inst.value)
#     sfc_si = (float(w_sfc.value)) / 3.6e6
#     g = 9.80665

#     rho_sl, a_sl, _, mu_sl = isa_atmosphere(0.0)
#     rho_ceil, a_ceil, _, mu_ceil = isa_atmosphere(Ceiling_m)

#     b_wing = np.sqrt(S_w * AR)
#     c_root_w = (2.0 * S_w) / (b_wing * (1.0 + 0.65))
#     c_tip_w = c_root_w * 0.65
#     mac = (2.0 / 3.0) * c_root_w * (1.0 + 0.65 + 0.65**2) / (1.0 + 0.65)
#     l_arm = 0.45 * FUSE_LEN
#     s_htail = (0.85 * S_w * mac) / l_arm
#     fuse_clearance = (0.5 * b_wing - R) - (0.5 * FUSE_W)

#     # High-Fidelity BEMT Solves for active design point
#     bemt_hov = run_bemt_solver(R, 0.45, Nb, c_0, taper, th_hov, twist, rpm_hov, 0.0, af_model, rho_sl, a_sl, mu_sl, num_elements=24)
#     bemt_cr = run_bemt_solver(R, 0.45, Nb, c_0, taper, th_cr, twist, rpm_cr, v_cr_ms, af_model, rho_sl, a_sl, mu_sl, num_elements=24)

#     eta_prop = max(bemt_cr.eta_prop, 0.70)
#     fm_hov = max(bemt_hov.FM, 0.65)

#     C_fe = 0.0030
#     S_wet_total = S_WET_FUSE + 2.05 * S_w + 2.0 * s_htail
#     cd0_dyn = C_fe * (S_wet_total / S_w)

#     W0_cur = 9000.0
#     ld_cr = 12.0
    
#     for _ in range(25):
#         # q_cr = 0.5 * rho_sl * (v_cr_ms**2)
#         # cl_cr = (W0_cur * g) / (q_cr * S_w)
#         # cd_cr = cd0_dyn + (cl_cr**2) / (np.pi * AR * 0.85)
#         #ld_cr = cl_cr / max(cd_cr, 1e-4)

#         m_f = np.exp(-(range_m * sfc_si * g) / (eta_prop * ld_cr))
#         ff = 1.06 * (1.0 - (0.985 * 0.990 * 0.975 * m_f * 0.990 * 0.985))

#         we_base = power_law(W0_cur, *popt_we)
#         w_blade_penalty = 0.035 * (4.20 / R) ** (-0.5)
#         we_total = we_base + w_blade_penalty - 0.035
#         denom = 1.0 - ff - we_total
#         if denom <= 0.05:
#             W0_cur = 35000.0
#             break
#         W_next = PAYLOAD_FIXED / denom
#         if abs(W_next - W0_cur) / W0_cur < 1e-4:
#             W0_cur = W_next
#             break
#         W0_cur = W_next

#     m_empty = W0_cur * power_law(W0_cur, *popt_we)
#     m_fuel = W0_cur * ff
#     A_disc_total = N_ROTORS * np.pi * (R**2)
#     disc_loading = W0_cur / A_disc_total
#     solidity = (Nb * c_0 * (1.0 + taper) * 0.5) / (np.pi * R)
#     cur_pw = P_inst_kW / W0_cur
#     V_flutter = 320.0 * ((tc_pct / 15.0) ** 1.5)

#     v_tip_hov = (rpm_hov * (np.pi / 30.0)) * R
#     v_tip_cr = (rpm_cr * (np.pi / 30.0)) * R
#     m_tip_hov = v_tip_hov / a_sl
#     m_tip_cr_helical = np.sqrt(v_tip_cr**2 + v_cr_ms**2) / a_sl
#     p_hov_total_req = (2.0 * bemt_hov.power_kW) / 0.95
#     p_cr_total_req = ((W0_cur * g / ld_cr) * v_cr_ms) / (eta_prop * 1000.0 * 0.95)

#     global SIZED_VEHICLE
#     SIZED_VEHICLE = {
#         "MTOW": W0_cur, "Empty_Mass": m_empty, "Fuel_Mass": m_fuel,
#         "Payload": PAYLOAD_FIXED, "Range_km": w_range.value, "V_max_kmh": v_cr_kmh,
#         "Ceiling_m": Ceiling_m, "Rotor_Radius": R, "Num_Blades": Nb, "Blade_Chord": c_0,
#         "Twist_deg": twist, "Pitch_Hover": th_hov, "Pitch_Cruise": th_cr, "Wing_Area": S_w,
#         "Aspect_Ratio": AR, "Wingspan": b_wing, "Wing_tc": tc_pct, "Installed_Power": P_inst_kW,
#         "CD0": cd0_dyn, "SFC": w_sfc.value, "Oswald_e": 0.85, "Airfoil": af_model,
#         "BEMT_Hover": bemt_hov, "BEMT_Cruise": bemt_cr
#     }

#     # ==========================================================================
#     # TAB 0: 2D GENERAL ARRANGEMENT & CARD (UNIFORM FIGSIZE 16.5 x 9.6)
#     # ==========================================================================
#     out_cad.clear_output(wait=True)
#     with out_cad:
#         fig_c = plt.figure(figsize=(16.5, 9.6), dpi=100)
#         gs_cad = fig_c.add_gridspec(2, 2, width_ratios=[1.3, 1.0])
#         ax_top = fig_c.add_subplot(gs_cad[0, 0])
#         ax_front = fig_c.add_subplot(gs_cad[1, 0])
#         ax_card = fig_c.add_subplot(gs_cad[:, 1])

#         collision_warn = fuse_clearance < 0.25

#         fx = np.array([0, 0.15 * FUSE_LEN, 0.75 * FUSE_LEN, FUSE_LEN, 0.75 * FUSE_LEN, 0.15 * FUSE_LEN, 0])
#         fy = np.array([0, 0.5 * FUSE_W, 0.5 * FUSE_W, 0, -0.5 * FUSE_W, -0.5 * FUSE_W, 0])
#         ax_top.fill(fx, fy, color='#dc3545' if collision_warn else '#ced4da', alpha=0.85, edgecolor='k', lw=1.3)

#         wx_le = 0.35 * FUSE_LEN
#         tip_x_off = 0.5 * b_wing * np.tan(np.radians(sweep_deg))
#         wx = [wx_le, wx_le + tip_x_off, wx_le + tip_x_off + c_tip_w, wx_le + c_root_w, wx_le + tip_x_off + c_tip_w, wx_le + tip_x_off, wx_le]
#         wy = [0, 0.5 * b_wing, 0.5 * b_wing, 0, -0.5 * b_wing, -0.5 * b_wing, 0]
#         ax_top.fill(wx, wy, color='#9ec5fe', alpha=0.8, edgecolor='blue', lw=1.3)

#         b_ht = np.sqrt(s_htail * 4.2)
#         c_ht = s_htail / b_ht
#         hx_le = wx_le + l_arm
#         ax_top.fill([hx_le, hx_le, hx_le + c_ht, hx_le + c_ht], [-0.5 * b_ht, 0.5 * b_ht, 0.5 * b_ht, -0.5 * b_ht], color='#6c757d', alpha=0.85, edgecolor='k')

#         rcs = [(wx_le + tip_x_off, 0.5 * b_wing), (wx_le + tip_x_off, -0.5 * b_wing)]
#         for rx, ry in rcs:
#             col = 'crimson' if collision_warn else 'darkgreen'
#             ax_top.add_patch(plt.Circle((rx, ry), R, color=col, fill=True, alpha=0.12, linestyle='--', lw=1.4))
#             ax_top.add_patch(plt.Circle((rx, ry), R, color=col, fill=False, linestyle='--', lw=1.4))
#             ax_top.plot(rx, ry, 'o', color=col, ms=4)

#         ax_top.set_aspect('equal')
#         ax_top.set_xlim(-1, FUSE_LEN + 2)
#         ax_top.set_ylim(-0.55 * b_wing - R, 0.55 * b_wing + R)
#         ax_top.set_title('Top View: Helicopter Mode' + (f' [!] TIP CLEARANCE: {fuse_clearance:.2f}m' if collision_warn else f' (Clearance: {fuse_clearance:.2f}m)'), fontsize=10, fontweight='bold')
#         ax_top.set_xlabel('X [m]', fontsize=9); ax_top.set_ylabel('Y [m]', fontsize=9); ax_top.grid(True, linestyle=':', alpha=0.5)

#         th_fuse = np.linspace(0, 2 * np.pi, 60)
#         fx_front = (0.5 * FUSE_W) * np.cos(th_fuse)
#         fz_front = (0.5 * FUSE_H) * np.sin(th_fuse)
#         ax_front.fill(fx_front, fz_front, color='#ced4da', alpha=0.9, edgecolor='k', lw=1.4)

#         wing_z = 0.35 * FUSE_H
#         ax_front.fill([-0.5 * b_wing, 0.5 * b_wing, 0.5 * b_wing, -0.5 * b_wing], [wing_z, wing_z, wing_z + 0.225 * c_root_w, wing_z + 0.225 * c_root_w], color='#9ec5fe', alpha=0.85, edgecolor='blue', lw=1.3)

#         for ny in [-0.5 * b_wing, 0.5 * b_wing]:
#             p_col = 'crimson' if collision_warn else '#198754'
#             ax_front.add_patch(plt.Circle((ny, wing_z), R, color=p_col, fill=True, alpha=0.15, linestyle='-', lw=1.8))
#             ax_front.add_patch(plt.Circle((ny, wing_z), R, color=p_col, fill=False, linestyle='-', lw=1.8))
#             ax_front.plot(ny, wing_z, 'o', color='black', ms=4.5)

#         ax_front.set_aspect('equal')
#         ax_front.set_xlim(-0.55 * b_wing - R, 0.55 * b_wing + R)
#         ax_front.set_ylim(-R - 0.5, R + 2.0)
#         ax_front.set_title(f'Front View: Cruise Mode (R={R:.2f}m)', fontsize=10, fontweight='bold')
#         ax_front.set_xlabel('Y [m]', fontsize=9); ax_front.set_ylabel('Z [m]', fontsize=9); ax_front.grid(True, linestyle=':', alpha=0.5)

#         ax_card.axis('off')
#         power_margin_hov = P_inst_kW - p_hov_total_req
#         power_margin_cr = P_inst_kW - p_cr_total_req
#         is_feasible = (power_margin_hov >= 0.0) and (power_margin_cr >= 0.0) and (not collision_warn)

#         card_str = (
#             "TROOP TRANSPORTER (10 PAX + 2 CREW) SIZING AUDIT\n"
#             "===================================================\n"
#             f"STATUS: {'[+] SIZING FEASIBLE' if is_feasible else '[-] PERFORMANCE / CLEARANCE DEFICIT'}\n"
#             f"ROTOR AIRFOIL:        {af_model.airfoil_name}\n"
#             "---------------------------------------------------\n"
#             "WEIGHT & PACKAGING BREAKDOWN:\n"
#             f"  • Takeoff Weight (MTOW): {W0_cur:8.1f} kg\n"
#             f"  • Operating Empty (We):  {m_empty:8.1f} kg ({m_empty/W0_cur*100:.1f}%)\n"
#             f"  • Mission Fuel (Wf):     {m_fuel:8.1f} kg ({m_fuel/W0_cur*100:.1f}%)\n"
#             f"  • Fixed Payload:         {PAYLOAD_FIXED:8.1f} kg (12 Troops)\n"
#             f"  • Wingspan (b):          {b_wing:8.2f} m\n"
#             f"  • Fuselage Clearance:    {fuse_clearance:8.2f} m\n\n"
#             "ROTOR PERFORMANCE & POWER AUDIT:\n"
#             f"  • Cruise L/D (est):      {ld_cr:8.2f}\n"
#             f"  • Rotor Radius (R):      {R:6.2f} m (Blades: {Nb})\n"
#             f"  • Disc Loading (DL):     {disc_loading:6.1f} kg/m²\n"
#             f"  • Hover Tip Mach:        {m_tip_hov:6.2f} (RPM: {rpm_hov:.0f})\n"
#             f"  • Cruise Helical Mach:   {m_tip_cr_helical:6.2f} (RPM: {rpm_cr:.0f})\n"
#             f"  • Flutter Speed (Est):   {V_flutter:6.1f} km/h (t/c = {tc_pct:.1f}%)\n"
#             f"  • Hover Total Power:     {p_hov_total_req:6.1f} kW\n"
#             f"  • Cruise Prop Efficiency:{eta_prop*100:6.1f} %\n"
#             f"  • Power Margin (Hover):  {power_margin_hov:6.1f} kW\n"
#         )
#         ax_card.text(0.02, 0.98, card_str, fontfamily='monospace', fontsize=9.6, verticalalignment='top',
#                      bbox=dict(boxstyle='round,pad=0.6', facecolor='#f8fff9' if is_feasible else '#fff8f8', edgecolor='#198754' if is_feasible else '#dc3545', lw=1.4))

#         plt.tight_layout()
#         plt.show()
#         plt.close(fig_c)
        

#     # ==========================================================================
#     # TAB 1: 4-PANEL FEASIBILITY HEATMAP MATRIX (UNIFORM FIGSIZE 16.5 x 9.6)
#     # ==========================================================================
#     out_heatmaps.clear_output(wait=True)
#     with out_heatmaps:
#         fig_hm, ((ax_h1, ax_h2), (ax_h3, ax_h4)) = plt.subplots(2, 2, figsize=(16.5, 9.6), dpi=100)

#         # ---------------- Panel 1: Dual-Pitch Stall Margin ----------------
#         n_p = 12
#         p_hov_vec = np.linspace(2.0, 24.0, n_p)
#         p_cr_vec = np.linspace(20.0, 60.0, n_p)
#         P_HOV_G, P_CR_G = np.meshgrid(p_hov_vec, p_cr_vec)
#         STALL_MARGIN_G = np.zeros_like(P_HOV_G)

#         for i in range(n_p):
#             for j in range(n_p):
#                 res_h = run_bemt_solver(R, 0.45, Nb, c_0, taper, P_HOV_G[i, j], twist, rpm_hov, 0.0, af_model, rho_sl, a_sl, num_elements=10)
#                 res_c = run_bemt_solver(R, 0.45, Nb, c_0, taper, P_CR_G[i, j], twist, rpm_cr, v_cr_ms, af_model, rho_sl, a_sl, num_elements=10)
#                 STALL_MARGIN_G[i, j] = 12.0 - max(np.max(np.abs(res_h.alpha_deg)), np.max(np.abs(res_c.alpha_deg)))

#         cp1 = ax_h1.contourf(P_HOV_G, P_CR_G, STALL_MARGIN_G, levels=14, cmap='RdYlGn', alpha=0.85)
#         cbar1 = plt.colorbar(cp1, ax=ax_h1)
#         cbar1.set_label(r'Stall Margin ($12^\circ - \max|\alpha|$) [°]', fontsize=8.5)
#         ax_h1.contour(P_HOV_G, P_CR_G, STALL_MARGIN_G, levels=[0.0], colors='black', linewidths=2.0, linestyles='--')
#         ax_h1.scatter(th_hov, th_cr, color='cyan', edgecolors='black', s=140, marker='*', zorder=10)
        
#         legend_h1 = [
#             Line2D([0], [0], color='black', lw=2.0, ls='--', label=r'Stall Boundary ($|\alpha| = 12^\circ$)'),
#             Patch(facecolor='forestgreen', edgecolor='k', alpha=0.6, label='Clean Flow Regime'),
#             Patch(facecolor='crimson', edgecolor='k', alpha=0.6, label='Stall Incursion'),
#             Line2D([0], [0], marker='*', color='w', markerfacecolor='cyan', markeredgecolor='k', markersize=12, label=f'Current: ({th_hov:.1f}°, {th_cr:.1f}°)')
#         ]
#         ax_h1.legend(handles=legend_h1, loc='lower left', fontsize=8, framealpha=0.92)
#         ax_h1.set_xlabel(r'Hover Pitch $\theta_{0.75\mathrm{,hov}}$ [°]', fontsize=9)
#         ax_h1.set_ylabel(r'Cruise Pitch $\theta_{0.75\mathrm{,cr}}$ [°]', fontsize=9)
#         ax_h1.set_title('A. Rotor Dual-Pitch Stall Feasibility Envelope', fontsize=10.5, fontweight='bold')
#         ax_h1.grid(True, ls=':', alpha=0.6)

#         # ---------------- Panel 2: Sizing Carpet Island ----------------
#         N_grid = 16
#         dl_vec = np.linspace(25.0, 200.0, N_grid)
#         ws_vec = np.linspace(100.0, 700.0, N_grid)
#         DL_g, WS_g = np.meshgrid(dl_vec, ws_vec)

#         P_avail_ceil = P_inst_kW * ((rho_ceil / rho_sl) ** 1.05)
#         w_mat = np.full_like(DL_g, 9000.0)

#         for _ in range(12):
#             s_w_mat = w_mat / WS_g
#             r_mat = np.sqrt((w_mat / DL_g) / (2 * np.pi))
#             q_cr_mat = 0.5 * rho_sl * (v_cr_ms ** 2)
#             cl_mat = (w_mat * g) / (q_cr_mat * s_w_mat)
#             cd_mat = cd0_dyn + (cl_mat ** 2) / (np.pi * AR * 0.85)
#             ld_mat = cl_mat / np.maximum(cd_mat, 1e-4)
#             mf_mat = np.exp(-(range_m * sfc_si * g) / (eta_prop * ld_mat))
#             ff_mat = 1.06 * (1.0 - (0.985 * 0.990 * 0.975 * mf_mat * 0.990 * 0.985))
#             we_mat = power_law(w_mat, *popt_we) + 0.035 * (4.20 / r_mat)**(-0.5) - 0.035
#             denom_mat = 1.0 - ff_mat - we_mat
#             w_mat = np.where(denom_mat <= 0.05, 35000.0, PAYLOAD_FIXED / np.maximum(denom_mat, 0.05))

#         W0_g = w_mat
#         s_w_final = W0_g / WS_g
#         b_w_final = np.sqrt(s_w_final * AR)
#         r_final = np.sqrt((W0_g / DL_g) / (2 * np.pi))
#         Clearance_g = (0.5 * b_w_final - r_final) - (0.5 * FUSE_W)

#         q_ceil = 0.5 * rho_ceil * (v_cr_ms ** 2)
#         cl_ceil = (W0_g * g) / (q_ceil * s_w_final)
#         cd_ceil = cd0_dyn + (cl_ceil ** 2) / (np.pi * AR * 0.85)
#         P_req_ceil_g = (q_ceil * s_w_final * cd_ceil * v_cr_ms) / (eta_prop * 1000.0) + (W0_g * g * 1.5) / 1000.0

#         cp2 = ax_h2.contourf(DL_g, WS_g, W0_g, levels=18, cmap='viridis_r', alpha=0.90)
#         cbar2 = plt.colorbar(cp2, ax=ax_h2)
#         cbar2.set_label(r'Gross Takeoff Weight $W_0$ [kg]', fontsize=8.5)
#         ax_h2.contour(DL_g, WS_g, Clearance_g, levels=[0.25], colors='red', linewidths=2.0)
#         ax_h2.contour(DL_g, WS_g, P_req_ceil_g, levels=[P_avail_ceil], colors='darkorange', linewidths=2.0, linestyles='--')
#         ax_h2.scatter(disc_loading, W0_cur / S_w, color='lime', edgecolors='black', s=140, marker='*', zorder=10)
        
#         legend_h2 = [
#             Line2D([0], [0], color='red', lw=2.0, label=r'Fuselage Clearance ($\geq 0.25$ m)'),
#             Line2D([0], [0], color='darkorange', lw=2.0, ls='--', label=rf'{Ceiling_m:.0f} m Ceiling Limit ($P \leq P_{{\mathrm{{avail}}}}$)'),
#             Line2D([0], [0], marker='*', color='w', markerfacecolor='lime', markeredgecolor='k', markersize=12, label=f'Current: DL={disc_loading:.1f}, W/S={W0_cur/S_w:.1f}')
#         ]
#         ax_h2.legend(handles=legend_h2, loc='upper right', fontsize=8, framealpha=0.92)
#         ax_h2.set_xlabel(r'Disc Loading $DL$ [$\mathrm{kg/m^2}$]', fontsize=9)
#         ax_h2.set_ylabel(r'Wing Loading $W/S$ [$\mathrm{kg/m^2}$]', fontsize=9)
#         ax_h2.set_title(f'B. Sizing Carpet Island ({v_cr_kmh:.0f} km/h | {Ceiling_m:.0f} m)', fontsize=10.5, fontweight='bold')
#         ax_h2.grid(True, ls=':', alpha=0.6)

#         # ---------------- Panel 3: Rotor Geometry & Efficiency ----------------
#         r_grid_vec = np.linspace(3.0, 5.5, n_p)
#         c_grid_vec = np.linspace(0.20, 0.65, n_p)
#         R_G, C_G = np.meshgrid(r_grid_vec, c_grid_vec)
#         SOL_G = (Nb * C_G * (1.0 + taper) * 0.5) / (np.pi * R_G)
#         CLEAR_G = (0.5 * b_wing - R_G) - (0.5 * FUSE_W)
#         FM_G = np.zeros_like(R_G)

#         for i in range(n_p):
#             for j in range(n_p):
#                 res_g = run_bemt_solver(R_G[i, j], 0.45, Nb, C_G[i, j], taper, th_hov, twist, rpm_hov, 0.0, af_model, rho_sl, a_sl, num_elements=10)
#                 FM_G[i, j] = res_g.FM

#         cp3 = ax_h3.contourf(R_G, C_G, FM_G, levels=14, cmap='magma', alpha=0.85)
#         cbar3 = plt.colorbar(cp3, ax=ax_h3)
#         cbar3.set_label('Hover Figure of Merit (FM)', fontsize=8.5)
#         ax_h3.contour(R_G, C_G, CLEAR_G, levels=[0.25], colors='red', linewidths=2.0)
#         ax_h3.contour(R_G, C_G, SOL_G, levels=[0.06, 0.09, 0.12], colors='cyan', linewidths=1.5, linestyles=':')
#         ax_h3.scatter(R, c_0, color='lime', edgecolors='black', s=140, marker='*', zorder=10)
        
#         legend_h3 = [
#             Line2D([0], [0], color='red', lw=2.0, label=r'Tip Clearance Limit ($0.25$ m)'),
#             Line2D([0], [0], color='cyan', lw=1.5, ls=':', label=r'Solidity Contours ($\sigma$)'),
#             Line2D([0], [0], marker='*', color='w', markerfacecolor='lime', markeredgecolor='k', markersize=12, label=f'Current: R={R:.2f}m, c={c_0:.2f}m')
#         ]
#         ax_h3.legend(handles=legend_h3, loc='lower right', fontsize=8, framealpha=0.92)
#         ax_h3.set_xlabel('Rotor Radius R [m]', fontsize=9)
#         ax_h3.set_ylabel(r'Blade Root Chord $c_0$ [m]', fontsize=9)
#         ax_h3.set_title('C. Rotor Geometry vs. Hover Efficiency', fontsize=10.5, fontweight='bold')
#         ax_h3.grid(True, ls=':', alpha=0.6)

#         # ---------------- Panel 4: Cruise Propulsive Matching ----------------
#         v_grid_vec = np.linspace(340.0, 560.0, n_p)
#         rpm_grid_vec = np.linspace(240.0, 480.0, n_p)
#         V_G, RPM_G = np.meshgrid(v_grid_vec, rpm_grid_vec)

#         vg_ms_grid = V_G / 3.6
#         v_t_grid = (RPM_G * (np.pi / 30.0)) * R
#         M_TIP_G = np.sqrt(v_t_grid**2 + vg_ms_grid**2) / a_sl
#         ETA_G = np.zeros_like(V_G)

#         for i in range(n_p):
#             for j in range(n_p):
#                 res_v = run_bemt_solver(R, 0.45, Nb, c_0, taper, th_cr, twist, RPM_G[i, j], vg_ms_grid[i, j], af_model, rho_sl, a_sl, num_elements=10)
#                 ETA_G[i, j] = max(res_v.eta_prop, 0.0)

#         PWR_CR_G = ((W0_cur * g / ld_cr) * vg_ms_grid) / (np.maximum(ETA_G, 0.40) * 1000.0 * 0.95)

#         cp4 = ax_h4.contourf(V_G, RPM_G, ETA_G * 100.0, levels=14, cmap='Blues', alpha=0.85)
#         cbar4 = plt.colorbar(cp4, ax=ax_h4)
#         cbar4.set_label(r'Cruise Prop Efficiency $\eta_{\mathrm{prop}}$ [%]', fontsize=8.5)
#         ax_h4.contour(V_G, RPM_G, M_TIP_G, levels=[0.82], colors='red', linewidths=2.0)
#         ax_h4.contour(V_G, RPM_G, PWR_CR_G, levels=[P_inst_kW], colors='darkorange', linewidths=2.0, linestyles='--')
#         ax_h4.scatter(v_cr_kmh, rpm_cr, color='yellow', edgecolors='black', s=140, marker='*', zorder=10)
        
#         legend_h4 = [
#             Line2D([0], [0], color='red', lw=2.0, label=r'Helical Mach Limit ($M_{\mathrm{tip}} = 0.82$)'),
#             Line2D([0], [0], color='darkorange', lw=2.0, ls='--', label=rf'Installed Power Limit ({P_inst_kW:.0f} kW)'),
#             Line2D([0], [0], marker='*', color='w', markerfacecolor='yellow', markeredgecolor='k', markersize=12, label=f'Current: ({v_cr_kmh:.0f} km/h, {rpm_cr:.0f} RPM)')
#         ]
#         ax_h4.legend(handles=legend_h4, loc='lower right', fontsize=8, framealpha=0.92)
#         ax_h4.set_xlabel(r'Cruise Speed $V_{\mathrm{cruise}}$ [km/h]', fontsize=9)
#         ax_h4.set_ylabel('Cruise Rotor Speed [RPM]', fontsize=9)
#         ax_h4.set_title(r'D. Cruise Matching ($M_{\mathrm{tip}} \leq 0.82$ Limit in Red)', fontsize=10.5, fontweight='bold')
#         ax_h4.grid(True, ls=':', alpha=0.6)

#         plt.tight_layout()
#         plt.show()
#         plt.close(fig_hm)
        

#     # ==========================================================================
#     # TAB 2: 3D LOFTED BLADE + 8-PANEL BEMT DASHBOARD (UNIFORM FIGSIZE 16.5 x 9.6)
#     # ==========================================================================
#     out_blade_3d.clear_output(wait=True)
#     with out_blade_3d:
#         fig_bemt = plt.figure(figsize=(16.5, 9.6), dpi=100)
#         gs = fig_bemt.add_gridspec(4, 4, width_ratios=[1.25, 1.25, 1.0, 1.0])

#         ax_top_rotor = fig_bemt.add_subplot(gs[0:1, 0:2])
#         ax_3d = fig_bemt.add_subplot(gs[1:4, 0:2], projection='3d')

#         ax_aoa  = fig_bemt.add_subplot(gs[0, 2])
#         ax_t    = fig_bemt.add_subplot(gs[0, 3])
#         ax_re   = fig_bemt.add_subplot(gs[1, 2])
#         ax_pq   = fig_bemt.add_subplot(gs[1, 3])
#         ax_coef = fig_bemt.add_subplot(gs[2, 2])
#         ax_ct_th= fig_bemt.add_subplot(gs[2, 3])
#         ax_inf  = fig_bemt.add_subplot(gs[3, 2])
#         ax_fm   = fig_bemt.add_subplot(gs[3, 3])

#         mode = w_bemt_mode.value
#         if mode == 'Hover':
#             a_hov, a_cr = 1.0, 0.08
#         elif mode == 'Cruise':
#             a_hov, a_cr = 0.08, 1.0
#         else:
#             a_hov, a_cr = 0.95, 0.95

#         # 1. 2D Top View
#         angles = np.linspace(0, 2 * np.pi, Nb, endpoint=False)
#         for ang in angles:
#             # Draw blade from root cutout (0.45m) to tip (R)
#             bx = [0.45 * np.cos(ang), R * np.cos(ang)]
#             by = [0.45 * np.sin(ang), R * np.sin(ang)]
#             ax_top_rotor.plot(bx, by, 'b-', lw=3.0)

#         # Rotor disk perimeter & center hub
#         ax_top_rotor.add_patch(plt.Circle((0, 0), R, color='gray', fill=False, linestyle='--', lw=1.2))
#         ax_top_rotor.add_patch(plt.Circle((0, 0), 0.45, color='gray', fill=True, alpha=0.6))
#         ax_top_rotor.set_aspect('equal')
#         ax_top_rotor.set_xlim(-R * 1.15, R * 1.15)
#         ax_top_rotor.set_ylim(-R * 1.15, R * 1.15)
#         ax_top_rotor.set_title(f'Top View (Nb={Nb}, r_root=0.45m)', fontsize=9.5, fontweight='bold')
#         ax_top_rotor.axis('off')

#         # 2. 3D Lofted Blade
#         x_af, y_af = load_airfoil_coords(af_model.airfoil_name)
#         n_elem = 20
#         r_edges = np.linspace(0.45, R, n_elem + 1)
#         r_cen = 0.5 * (r_edges[:-1] + r_edges[1:])
#         alpha_hov_elem = np.interp(r_cen, bemt_hov.r_stations, np.abs(bemt_hov.alpha_deg))
#         alpha_cr_elem = np.interp(r_cen, bemt_cr.r_stations, np.abs(bemt_cr.alpha_deg))
        
#         if mode == 'Hover':
#             is_stalled = alpha_hov_elem >= 12.0
#             blade_pitch = th_hov
#         elif mode == 'Cruise':
#             is_stalled = alpha_cr_elem >= 12.0
#             blade_pitch = th_cr
#         else:
#             is_stalled = (alpha_hov_elem >= 12.0) | (alpha_cr_elem >= 12.0)
#             blade_pitch = th_hov

#         ax_3d.plot([0, 0.45], [0, 0], [0, 0], color='#495057', lw=3.5, label='Root Hub Cutout (0.45m)')
#         ax_3d.plot([0.45, R], [0, 0], [0, 0], 'k--', lw=1.2, label='Pitch Axis (25% Chord)')

#         for i in range(n_elem):
#             r_in, r_out = r_edges[i], r_edges[i + 1]
#             c_in = c_0 + (c_0 * taper - c_0) * ((r_in - 0.45) / max(1e-4, R - 0.45))
#             c_out = c_0 + (c_0 * taper - c_0) * ((r_out - 0.45) / max(1e-4, R - 0.45))
#             th_in = np.radians(blade_pitch + twist * ((r_in / R) - 0.75))
#             th_out = np.radians(blade_pitch + twist * ((r_out / R) - 0.75))

#             x_rot_in = (0.25 - x_af) * c_in * np.cos(th_in) - y_af * c_in * np.sin(th_in)
#             z_rot_in = (0.25 - x_af) * c_in * np.sin(th_in) + y_af * c_in * np.cos(th_in)
#             x_rot_out = (0.25 - x_af) * c_out * np.cos(th_out) - y_af * c_out * np.sin(th_out)
#             z_rot_out = (0.25 - x_af) * c_out * np.sin(th_out) + y_af * c_out * np.cos(th_out)

#             poly_list = [[
#                 [r_in, x_rot_in[j], z_rot_in[j]], [r_in, x_rot_in[j + 1], z_rot_in[j + 1]],
#                 [r_out, x_rot_out[j + 1], z_rot_out[j + 1]], [r_out, x_rot_out[j], z_rot_out[j]],
#             ] for j in range(len(x_af) - 1)]

#             seg_color = (0.95, 0.2, 0.2, 0.8) if is_stalled[i] else (0.2, 0.7, 0.9, 0.65)
#             ax_3d.add_collection3d(art3d.Poly3DCollection(poly_list, facecolors=seg_color, edgecolors=(0, 0, 0, 0.2), linewidths=0.25))

#         ax_3d.set_box_aspect((2.5, 1.2, 0.9))
#         ax_3d.set_xlim(0, R + 0.1); ax_3d.set_ylim(-c_0 * 0.9, c_0 * 0.9); ax_3d.set_zlim(-c_0 * 0.5, c_0 * 0.5)
#         ax_3d.set_xlabel('Radius r [m]', fontsize=8); ax_3d.set_ylabel('Chordwise x [m]', fontsize=8); ax_3d.set_zlabel('Height z [m]', fontsize=8)
#         ax_3d.set_title(f'Lofted 3D Blade ({af_model.airfoil_name} - {mode} Pitch)', fontsize=9.5, fontweight='bold')
#         ax_3d.view_init(elev=22, azim=-60)
#         ax_3d.legend(loc='upper right', fontsize=7.5)

#         r_norm = bemt_hov.r_stations / R

#         # 3. Aerodynamics Grid: AoA
#         ax_aoa.plot(r_norm, bemt_hov.alpha_deg, 'b-', lw=1.8, alpha=a_hov, label=rf'Hover ({th_hov:.0f}°)')
#         ax_aoa.plot(r_norm, bemt_cr.alpha_deg, 'm--', lw=1.8, alpha=a_cr, label=rf'Cruise ({th_cr:.0f}°)')
#         ax_aoa.axhline(14.0, color='r', ls=':', label=r'$\alpha_{\mathrm{stall}}$')
#         ax_aoa.set_ylabel('AoA [deg]', fontsize=8); ax_aoa.set_title('AoA Profile', fontsize=8.5, fontweight='bold')
#         ax_aoa.grid(True, ls=':', alpha=0.6); ax_aoa.legend(fontsize=7, loc='upper right')

#         # 4. Thrust Loading
#         ax_t.plot(r_norm, bemt_hov.dt_dr, 'g-', lw=1.8, alpha=a_hov, label=f'Hover ({bemt_hov.thrust_N:.0f} N)')
#         ax_t.plot(r_norm, bemt_cr.dt_dr, 'g--', lw=1.8, alpha=a_cr, label=f'Cruise ({bemt_cr.thrust_N:.0f} N)')
#         ax_t.set_ylabel('dT [N/m]', fontsize=8); ax_t.set_title('Thrust Loading', fontsize=8.5, fontweight='bold')
#         ax_t.grid(True, ls=':', alpha=0.6); ax_t.legend(fontsize=7, loc='upper right')

#         # 5. Reynolds Distribution
#         ax_re.plot(r_norm, bemt_hov.re_r / 1e5, 'k-', lw=1.8, alpha=a_hov, label=rf'Hover ($Re_{{tip}}={np.max(bemt_hov.re_r)/1e3:.0f}\mathrm{{k}}$)')
#         ax_re.plot(r_norm, bemt_cr.re_r / 1e5, 'orange', ls='--', lw=1.8, alpha=a_cr, label=rf'Cruise ($Re_{{tip}}={np.max(bemt_cr.re_r)/1e3:.0f}\mathrm{{k}}$)')
#         ax_re.set_ylabel(r'$Re \times 10^{-5}$', fontsize=8); ax_re.set_title('Reynolds Distribution', fontsize=8.5, fontweight='bold')
#         ax_re.grid(True, ls=':', alpha=0.6); ax_re.legend(fontsize=7, loc='upper left')

#         # 6. Power Loading
#         ax_pq.plot(r_norm, bemt_hov.dp_dr / 1000.0, 'r-', lw=1.8, alpha=a_hov, label=f'Hover ({bemt_hov.power_kW:.1f} kW)')
#         ax_pq.plot(r_norm, bemt_cr.dp_dr / 1000.0, 'r--', lw=1.8, alpha=a_cr, label=f'Cruise ({bemt_cr.power_kW:.1f} kW)')
#         ax_pq.set_ylabel('dP [kW/m]', color='r', fontsize=8)
#         ax_pq.set_title('Sectional Power Loading', fontsize=8.5, fontweight='bold')
#         ax_pq.grid(True, ls=':', alpha=0.6); ax_pq.legend(fontsize=7, loc='upper left')

#         # 7. Section Coefficients (Cl, Cd)
#         ax_coef.plot(r_norm, bemt_hov.cl, 'b-', lw=1.8, alpha=a_hov, label=r'Hover $C_l$')
#         ax_coef.plot(r_norm, 10.0 * bemt_hov.cd, 'r-', lw=1.3, alpha=a_hov, label=r'Hover $10 \times C_d$')
#         ax_coef.plot(r_norm, bemt_cr.cl, 'b--', lw=1.8, alpha=a_cr, label=r'Cruise $C_l$')
#         ax_coef.plot(r_norm, 10.0 * bemt_cr.cd, 'r--', lw=1.3, alpha=a_cr, label=r'Cruise $10 \times C_d$')
#         ax_coef.set_ylabel(r'$C_l$ and $C_d$', fontsize=8); ax_coef.set_title('Section Coefficients', fontsize=8.5, fontweight='bold')
#         ax_coef.grid(True, ls=':', alpha=0.6); ax_coef.legend(fontsize=6.8, loc='upper left')

#         # 8. Fast CT Sweeps (12 Points)
#         th_sweep_hov = np.linspace(0.0, 25.0, 12)
#         th_sweep_cr = np.linspace(20.0, 60.0, 12)
#         res_sweep_hov = [run_bemt_solver(R, 0.45, Nb, c_0, taper, p, twist, rpm_hov, 0.0, af_model, rho_sl, a_sl, num_elements=10) for p in th_sweep_hov]
#         res_sweep_cr = [run_bemt_solver(R, 0.45, Nb, c_0, taper, p, twist, rpm_cr, v_cr_ms, af_model, rho_sl, a_sl, num_elements=10) for p in th_sweep_cr]
        
#         ct_sweep_hov = [r.CT for r in res_sweep_hov]
#         ct_sweep_cr = [r.CT for r in res_sweep_cr]
#         fm_sweep_hov = [r.FM for r in res_sweep_hov]
#         eta_sweep_cr = [max(r.eta_prop, 0.0) for r in res_sweep_cr]

#         ax_ct_th.plot(th_sweep_hov, ct_sweep_hov, 'b-o', ms=3.0, alpha=a_hov, label=r'Hover $C_T(\theta)$')
#         ax_ct_th.plot(th_hov, bemt_hov.CT, 'ro', ms=6.0, alpha=a_hov)
#         ax_ct_th.plot(th_sweep_cr, ct_sweep_cr, 'm--s', ms=3.0, alpha=a_cr, label=r'Cruise $C_T(\theta)$')
#         ax_ct_th.plot(th_cr, bemt_cr.CT, 'mo', ms=6.0, alpha=a_cr)
#         ax_ct_th.set_xlabel(r'$\theta_{0.75}$ [deg]', fontsize=8); ax_ct_th.set_ylabel(r'$C_T$', fontsize=8); ax_ct_th.set_title(r'$C_T$ vs Pitch Angle $\theta_{0.75}$', fontsize=8.5, fontweight='bold')
#         ax_ct_th.grid(True, ls=':', alpha=0.6); ax_ct_th.legend(fontsize=6.8, loc='upper left')

#         # 9. Inflow Distribution
#         ax_inf.plot(r_norm, bemt_hov.lambda_tot, 'k-', lw=1.8, alpha=a_hov, label=r'Hover $\lambda_{\mathrm{tot}}$')
#         ax_inf.plot(r_norm, bemt_hov.lambda_i, 'c:', lw=1.5, alpha=a_hov, label=r'Hover $\lambda_i$')
#         ax_inf.plot(r_norm, bemt_cr.lambda_tot, 'k--', lw=1.8, alpha=a_cr, label=r'Cruise $\lambda_{\mathrm{tot}}$')
#         ax_inf.plot(r_norm, bemt_cr.lambda_i, 'm:', lw=1.5, alpha=a_cr, label=r'Cruise $\lambda_i$')
#         ax_inf.set_xlabel(r'$r/R$', fontsize=8); ax_inf.set_ylabel('Inflow ' + r'$\lambda$', fontsize=8); ax_inf.set_title('Inflow Distribution', fontsize=8.5, fontweight='bold')
#         ax_inf.grid(True, ls=':', alpha=0.6); ax_inf.legend(fontsize=6.8, loc='upper right')

#         # 10. Efficiency Polars
#         ax_fm.plot(ct_sweep_hov, fm_sweep_hov, 'g-o', ms=3.0, alpha=a_hov, label='Hover FM Curve')
#         ax_fm.plot(bemt_hov.CT, bemt_hov.FM, 'ro', ms=6.0, alpha=a_hov, label=f'FM={bemt_hov.FM:.3f}')
#         ax_fm.plot(ct_sweep_cr, eta_sweep_cr, 'm--s', ms=3.0, alpha=a_cr, label=r'Cruise $\eta_{\mathrm{prop}}$')
#         ax_fm.plot(bemt_cr.CT, bemt_cr.eta_prop, 'mo', ms=6.0, alpha=a_cr, label=rf'$\eta$={bemt_cr.eta_prop*100:.1f}%')
#         ax_fm.set_xlabel(r'$C_T$', fontsize=8); ax_fm.set_ylabel('FM / Prop Efficiency', fontsize=8); ax_fm.set_title('Hover FM & Cruise Efficiency Polars', fontsize=8.5, fontweight='bold')
#         ax_fm.grid(True, ls=':', alpha=0.6); ax_fm.legend(fontsize=6.8, loc='lower right')

#         plt.tight_layout()
#         plt.show()
#         plt.close(fig_bemt)
        

#     # ==========================================================================
#     # TAB 3: STATISTICAL REGRESSIONS (UNIFORM FIGSIZE 16.5 x 9.6)
#     # ==========================================================================
#     out_stats.clear_output(wait=True)
#     with out_stats:
#         fig_reg, axes_reg = plt.subplots(2, 2, figsize=(16.5, 9.6), dpi=100)
#         x_w_grid = np.linspace(1800.0, 26000.0, 150)
#         x_dl_grid = np.linspace(20.0, 220.0, 150)

#         # 1. Empty Weight Fraction
#         ax_r1 = axes_reg[0, 0]
#         ax_r1.scatter(benchmarks['MTOW_kg'], benchmarks['We_W0'], color='crimson', s=65, edgecolors='k', zorder=4)
#         for _, r in benchmarks.iterrows():
#             ax_r1.annotate(str(r['Aircraft']), xy=(float(r['MTOW_kg'] * 1.04), float(r['We_W0'])), fontsize=8)
#         ax_r1.plot(x_w_grid, power_law(x_w_grid, *popt_we), 'b-', lw=1.8, label=rf'Power Fit (${popt_we[0]:.2f} W_0^{{{popt_we[1]:.3f}}}$)')
#         ax_r1.scatter(W0_cur, m_empty / W0_cur, color='lime', s=150, marker='*', edgecolors='k', zorder=5, label=f'Your Design ({m_empty/W0_cur:.3f})')
#         ax_r1.set_xscale('log'); ax_r1.set_xlabel('Takeoff Gross Weight W0 [kg]', fontsize=9); ax_r1.set_ylabel('Empty Fraction We/W0', fontsize=9)
#         ax_r1.set_title('1. Empty Weight Fraction Regression', fontsize=10.5, fontweight='bold'); ax_r1.grid(True, which='both', ls=':', alpha=0.6); ax_r1.legend(fontsize=8)

#         # 2. Installed Power Loading
#         ax_r2 = axes_reg[0, 1]
#         ax_r2.scatter(benchmarks['Disc_Loading'], benchmarks['Installed_PW'], color='crimson', s=65, edgecolors='k', zorder=4)
#         for _, r in benchmarks.iterrows():
#             ax_r2.annotate(str(r['Aircraft']), xy=(float(r['Disc_Loading'] + 2.0), float(r['Installed_PW'])), fontsize=8)
#         ax_r2.plot(x_dl_grid, power_law(x_dl_grid, *popt_pw), 'b-', lw=1.8, label=rf'Power Fit (${popt_pw[0]:.3f} DL^{{{popt_pw[1]:.3f}}}$)')
#         ax_r2.scatter(disc_loading, cur_pw, color='lime', s=150, marker='*', edgecolors='k', zorder=5, label=f'Your Design ({cur_pw:.3f} kW/kg)')
#         ax_r2.set_xlabel('Disc Loading DL [kg/m²]', fontsize=9); ax_r2.set_ylabel('Power Loading P/W0 [kW/kg]', fontsize=9)
#         ax_r2.set_title('2. Installed Power Loading vs Disc Loading', fontsize=10.5, fontweight='bold'); ax_r2.grid(True, ls=':', alpha=0.6); ax_r2.legend(fontsize=8)

#         # 3. Blade Solidity
#         ax_r3 = axes_reg[1, 0]
#         ax_r3.scatter(benchmarks['Disc_Loading'], benchmarks['Solidity'], color='crimson', s=65, edgecolors='k', zorder=4)
#         for _, r in benchmarks.iterrows():
#             ax_r3.annotate(str(r['Aircraft']), xy=(float(r['Disc_Loading'] + 2.0), float(r['Solidity'])), fontsize=8)
#         ax_r3.plot(x_dl_grid, linear_law(x_dl_grid, *popt_sol), 'g-', lw=1.8, label='Linear Fit')
#         ax_r3.scatter(disc_loading, solidity, color='lime', s=150, marker='*', edgecolors='k', zorder=5, label=f'Your Design ({solidity:.4f})')
#         ax_r3.set_xlabel('Disc Loading DL [kg/m²]', fontsize=9); ax_r3.set_ylabel('Blade Solidity σ', fontsize=9)
#         ax_r3.set_title('3. Blade Solidity vs Disc Loading', fontsize=10.5, fontweight='bold'); ax_r3.grid(True, ls=':', alpha=0.6); ax_r3.legend(fontsize=8)

#         # 4. Service Ceiling Capability
#         ax_r4 = axes_reg[1, 1]
#         x_ceil_grid = np.linspace(3000.0, 9500.0, 150)
#         ax_r4.scatter(benchmarks['Service_Ceiling_m'], benchmarks['Installed_PW'], color='crimson', s=65, edgecolors='k', zorder=4)
#         for _, r in benchmarks.iterrows():
#             ax_r4.annotate(str(r['Aircraft']), xy=(float(r['Service_Ceiling_m'] + 80.0), float(r['Installed_PW'])), fontsize=8)
#         ax_r4.plot(x_ceil_grid, linear_law(x_ceil_grid, *popt_ceil), 'm-', lw=1.8, label='Ceiling Fit')
#         ax_r4.scatter(Ceiling_m, cur_pw, color='lime', s=150, marker='*', edgecolors='k', zorder=5, label=f'Your Design ({cur_pw:.3f} kW/kg)')
#         ax_r4.set_xlabel('Service Ceiling [m]', fontsize=9); ax_r4.set_ylabel('Power Loading P/W0 [kW/kg]', fontsize=9)
#         ax_r4.set_title('4. Service Ceiling Capability', fontsize=10.5, fontweight='bold'); ax_r4.grid(True, ls=':', alpha=0.6); ax_r4.legend(fontsize=8)

#         plt.tight_layout()
#         plt.show()
#         plt.close(fig_reg)
        

# for w in [w_airfoil, w_radius, w_nblades, w_chord, w_taper, w_pitch_hov, w_pitch_cr, w_twist,
#           w_rpm_hov, w_rpm_cr, w_s_wing, w_ar, w_sweep, w_tc, w_v_cruise, w_range, w_ceiling, w_p_inst, w_sfc, w_bemt_mode]:
#     w.observe(update_designer_dashboard, names='value')

# def header_lbl(text):
#     return widgets.HTML(f'<b style="font-size:12px; margin:2px 0px 1px 0px; display:inline-block;">{text}</b>')

# designer_sidebar = widgets.VBox(
#     [
#         header_lbl('3D/BEMT Diagnostics Display Mode'),
#         w_bemt_mode,
#         header_lbl('Mission & Flight Envelope'),
#         w_range,
#         w_v_cruise,
#         w_ceiling,
#         header_lbl('Proprotor Blade & Aeromechanics'),
#         w_airfoil,
#         w_radius,
#         w_nblades,
#         w_chord,
#         w_taper,
#         w_pitch_hov,
#         w_pitch_cr,
#         w_twist,
#         header_lbl('Dual-Speed RPM Schedules'),
#         w_rpm_hov,
#         w_rpm_cr,
#         header_lbl('Wing Aerodynamics & Structural Sizing'),
#         w_s_wing,
#         w_ar,
#         w_sweep,
#         w_tc,
#         header_lbl('Installed Powerplant & Drag'),
#         w_p_inst,
#         w_sfc,
#     ],
#     layout=widgets.Layout(
#         padding='6px 8px',
#         border='1px solid #ced4da',
#         border_radius='6px',
#         width='365px',
#         min_width='365px',
#         flex='0 0 365px'
#     ),
# )

# display(
#     widgets.HBox(
#         [designer_sidebar, tabs_designer],
#         layout=widgets.Layout(
#             width='100%',
#             align_items='flex-start',
#             overflow='auto'
#         ),
#     )
# )
# update_designer_dashboard()

In [46]:
# ==============================================================================
# CELL 2: TILTROTOR DESIGNER, 4-PANEL HEATMAPS, 3D BEMT ANALYZER & REGRESSIONS
# (AUTO-TRIMMED HOVER & CRUISE COLLECTIVE EQUILIBRIUM)
# ==============================================================================
import os
import glob
import urllib.request
from dataclasses import dataclass
from typing import Dict, List, Optional, Tuple

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import mpl_toolkits.mplot3d.art3d as art3d
import ipywidgets as widgets
from IPython.display import display, HTML
from scipy.interpolate import RectBivariateSpline
from scipy.optimize import curve_fit, brentq
from matplotlib.lines import Line2D
from matplotlib.patches import Patch

# ------------------------------------------------------------------------------
# 1. ATMOSPHERE & HISTORICAL BENCHMARK REGRESSIONS
# ------------------------------------------------------------------------------
FUSE_LEN = 15.20
FUSE_W = 2.15
FUSE_H = 2.05
PAYLOAD_FIXED = 1400.0
N_ROTORS = 2

d_eq = np.sqrt((4.0 / np.pi) * FUSE_W * FUSE_H)
fineness = max(FUSE_LEN / d_eq, 3.0)
S_WET_FUSE = np.pi * d_eq * FUSE_LEN * (1.0 - 2.0 / fineness) ** (2 / 3) * (1.0 + 1.0 / (fineness**2))

def isa_atmosphere(altitude_m: float) -> tuple[float, float, float, float]:
    alt = np.clip(altitude_m, 0.0, 11000.0)
    T = 288.15 - 0.0065 * alt
    P = 101325.0 * ((T / 288.15) ** 5.2561)
    rho = P / (287.058 * T)
    a = np.sqrt(1.4 * 287.058 * T)
    mu = 1.789e-5 * ((T / 288.15) ** 1.5) * ((288.15 + 110.4) / (T + 110.4))
    return rho, a, T, mu

benchmarks = pd.DataFrame({
    "Aircraft": ["Bell XV-3", "Curtiss-Wright X-19", "CL-84 Dynavert", "Bell XV-15", "Leonardo AW609", "Bell V-280 Valor", "LTV XC-142", "Bell Boeing V-22"],
    "MTOW_kg": [2177.0, 6196.0, 5715.0, 6000.0, 8165.0, 14000.0, 20185.0, 23982.0],
    "Empty_kg": [1648.0, 4527.0, 3818.0, 4570.0, 4765.0, 8200.0, 10250.0, 15032.0],
    "Disc_Loading": [32.5, 145.0, 102.0, 73.2, 88.4, 95.0, 135.0, 102.5],
    "Solidity": [0.053, 0.115, 0.102, 0.089, 0.096, 0.100, 0.110, 0.105],
    "Installed_PW": [0.154, 0.410, 0.440, 0.385, 0.354, 0.533, 0.450, 0.380],
    "Service_Ceiling_m": [3600.0, 5330.0, 4880.0, 8840.0, 7620.0, 8500.0, 7600.0, 7620.0],
    "Wing_tc": [15.0, 19.0, 16.0, 23.0, 21.0, 20.0, 18.0, 23.0]
})
benchmarks["We_W0"] = benchmarks["Empty_kg"] / benchmarks["MTOW_kg"]

def power_law(x, a, b):
    return a * (x ** b)

def linear_law(x, a, b):
    return a * x + b

popt_we, _ = curve_fit(power_law, benchmarks["MTOW_kg"].values, benchmarks["We_W0"].values, p0=[1.5, -0.09])
popt_pw, _ = curve_fit(power_law, benchmarks["Disc_Loading"].values, benchmarks["Installed_PW"].values, p0=[0.05, 0.5])
popt_sol, _ = curve_fit(linear_law, benchmarks["Disc_Loading"].values, benchmarks["Solidity"].values, p0=[0.001, 0.02])
popt_ceil, _ = curve_fit(linear_law, benchmarks["Service_Ceiling_m"].values, benchmarks["Installed_PW"].values, p0=[5e-5, 0.1])

# ------------------------------------------------------------------------------
# 2. AIRFOIL SCRAPER, BEMT SOLVER & TRIM EQUILIBRIUM ENGINE
# ------------------------------------------------------------------------------
KNOWN_MAPPINGS = {
    'clarky': 'clarky-il',
    'clarkyh': 'clarkyh-il',
    'goe398': 'goe398-il',
    'goe535': 'goe535-il',
    'usa35b': 'usa35b-il',
    'vr12': 'vr12-il',
    'boeingvertolvr12': 'vr12-il',
    'boeingvertolvr12airfoil': 'vr12-il',
    'nasasc20010': 'sc20010-il',
    'nasasc20012': 'sc20012-il',
    'nasasc20410': 'sc20410-il',
    'nasasc20412': 'sc20412-il',
    'naca64a010': 'naca64a010-il',
    '64a010': 'naca64a010-il',
    'naca64a012': 'n64012a-il',
    '64a012': 'n64012a-il',
    'naca23012': 'n23012-il',
    'oneraoa209': 'oa209-il',
    'oneraoa212': 'oa212-il',
    'sikorskysc1095': 'sc1095-il',
}

def to_airfoiltools_slug(name: str) -> str:
    clean = name.lower().strip().replace(" ", "").replace("-", "").replace("_", "").replace("(", "").replace(")", "")
    if clean in KNOWN_MAPPINGS:
        return KNOWN_MAPPINGS[clean]
    if clean.startswith("naca"):
        return f"n{clean.replace('naca', '')}-il"
    return f"{clean}-il" if not (clean.endswith("il") or clean.endswith("sa")) else clean

def ensure_airfoil_data(airfoil_name: str, airfoil_dir: str = "airfoils"):
    os.makedirs(airfoil_dir, exist_ok=True)
    slug = to_airfoiltools_slug(airfoil_name)
    target_dir = os.path.join(airfoil_dir, slug)
    os.makedirs(target_dir, exist_ok=True)
    dat_path = os.path.join(target_dir, f"{slug}.dat")
    if not os.path.exists(dat_path):
        try:
            url_dat = f"http://airfoiltools.com/airfoil/seligdatfile?airfoil={slug}"
            req = urllib.request.Request(url_dat, headers={'User-Agent': 'Mozilla/5.0'})
            with urllib.request.urlopen(req, timeout=3) as resp:
                content = resp.read().decode('utf-8')
                if len(content) > 100 and "<html>" not in content.lower():
                    with open(dat_path, 'w') as f:
                        f.write(content)
        except Exception:
            pass

    for re_v in [50000, 100000, 200000, 500000, 1000000]:
        for nc_str, nc_tag in [("", ""), ("-n5", "-n5")]:
            csv_path = os.path.join(target_dir, f"xf-{slug}-{re_v}{nc_tag}.csv")
            if not os.path.exists(csv_path):
                try:
                    url_csv = f"http://airfoiltools.com/polar/csv?polar=xf-{slug}-{re_v}{nc_str}"
                    req = urllib.request.Request(url_csv, headers={'User-Agent': 'Mozilla/5.0'})
                    with urllib.request.urlopen(req, timeout=3) as resp:
                        content = resp.read().decode('utf-8')
                        if "alpha" in content.lower() and "cl" in content.lower():
                            with open(csv_path, 'w') as f:
                                f.write(content)
                except Exception:
                    pass

def load_airfoil_coords(airfoil_name: str) -> Tuple[np.ndarray, np.ndarray]:
    ensure_airfoil_data(airfoil_name)
    slug = to_airfoiltools_slug(airfoil_name)
    for d in ["airfoils", "."]:
        for fpath in glob.glob(os.path.join(d, "**", "*.dat"), recursive=True):
            if slug.replace("-il", "") in os.path.basename(fpath).lower():
                raw_x, raw_y = [], []
                with open(fpath, 'r') as f:
                    for line in f.readlines()[1:]:
                        p = line.strip().split()
                        if len(p) == 2:
                            try:
                                raw_x.append(float(p[0]))
                                raw_y.append(float(p[1]))
                            except ValueError:
                                continue
                if len(raw_x) > 10:
                    return np.array(raw_x), np.array(raw_y)
    beta = np.linspace(0, np.pi, 30)
    x = 0.5 * (1.0 - np.cos(beta))
    yt = 5.0 * 0.12 * (0.2969 * np.sqrt(x) - 0.1260 * x - 0.3516 * (x**2) + 0.2843 * (x**3) - 0.1015 * (x**4))
    return np.concatenate([x[::-1], x[1:]]), np.concatenate([yt[::-1], -yt[1:]])

class AirfoilModel:
    def __init__(self, airfoil_name: str = "NACA 0012", ncrit: int = 9):
        self.airfoil_name = airfoil_name
        self.ncrit = ncrit
        self.has_polars = False
        self.raw_data = {}
        ensure_airfoil_data(airfoil_name)
        self._build_polar_interpolators()

    def _build_polar_interpolators(self):
        slug = to_airfoiltools_slug(self.airfoil_name).replace("-il", "")
        for fpath in glob.glob(os.path.join("airfoils", "**", "*.csv"), recursive=True):
            if slug in os.path.basename(fpath).lower():
                try:
                    df = pd.read_csv(fpath, skiprows=10)
                    df.columns = [c.strip().lower() for c in df.columns]
                    if 'alpha' in df.columns and 'cl' in df.columns and 'cd' in df.columns:
                        re_val = float(os.path.basename(fpath).split("-")[-1].replace(".csv", "").replace("n5", ""))
                        self.raw_data[re_val] = df[['alpha', 'cl', 'cd']].rename(columns={'cl': 'CL', 'cd': 'CD'})
                except Exception:
                    pass

        if self.raw_data:
            self.re_list = sorted(self.raw_data.keys())
            alpha_common = np.linspace(np.radians(-15.0), np.radians(20.0), 71)
            cl_grid = np.zeros((len(alpha_common), len(self.re_list)))
            cd_grid = np.zeros((len(alpha_common), len(self.re_list)))
            for j, re_v in enumerate(self.re_list):
                df = self.raw_data[re_v]
                a_rad = np.radians(df['alpha'].values.astype(float))
                cl_grid[:, j] = np.interp(alpha_common, a_rad, df['CL'].values.astype(float))
                cd_grid[:, j] = np.interp(alpha_common, a_rad, df['CD'].values.astype(float))
            self.spline_cl = RectBivariateSpline(
                alpha_common, np.log10(self.re_list), cl_grid,
                kx=2, ky=min(len(self.re_list) - 1, 2)
            )
            self.spline_cd = RectBivariateSpline(
                alpha_common, np.log10(self.re_list), cd_grid,
                kx=2, ky=min(len(self.re_list) - 1, 2)
            )
            self.has_polars = True

    def get_cl_slope(self, re_val: float = 500000.0) -> float:
        if self.has_polars:
            log_re = np.log10(np.clip(re_val, 1e4, 1e7))
            a1, a2 = np.radians(-2.0), np.radians(4.0)
            cl1 = float(self.spline_cl.ev(a1, log_re))
            cl2 = float(self.spline_cl.ev(a2, log_re))
            slope = (cl2 - cl1) / (a2 - a1)
            return float(np.clip(slope, 4.5, 6.8))
        return 5.73

    def evaluate(self, alpha_rad: np.ndarray, re_arr: np.ndarray, mach_arr: np.ndarray) -> Tuple[np.ndarray, np.ndarray, np.ndarray]:
        if self.has_polars:
            log_re = np.log10(np.clip(re_arr, 1e4, 1e7))
            cl = self.spline_cl.ev(alpha_rad, log_re)
            cd = self.spline_cd.ev(alpha_rad, log_re)
        else:
            cl = 5.75 * alpha_rad
            cd = 0.011 + 1.25 * (alpha_rad ** 2)
        beta = np.sqrt(np.maximum(1e-4, 1.0 - np.clip(mach_arr, 0.0, 0.95)**2))
        return cl / beta, cd / beta, np.abs(alpha_rad) > np.radians(14.0)

@dataclass
class BEMTResult:
    thrust_N: float
    torque_Nm: float
    power_kW: float
    CT: float
    CP: float
    FM: float
    eta_prop: float
    r_stations: np.ndarray
    chords: np.ndarray
    thetas_deg: np.ndarray
    phi_deg: np.ndarray
    alpha_deg: np.ndarray
    lambda_i: np.ndarray
    lambda_tot: np.ndarray
    cl: np.ndarray
    cd: np.ndarray
    re_r: np.ndarray
    dt_dr: np.ndarray
    dp_dr: np.ndarray
    dq_dr: np.ndarray
    dCT_dr: np.ndarray
    dCP_dr: np.ndarray
    mach_helical: np.ndarray

def run_bemt_solver(radius: float, root_cutout: float, num_blades: int, c_root: float, taper: float,
                    pitch_75_deg: float, twist_deg: float, rpm: float, v_axial_ms: float,
                    airfoil: AirfoilModel, rho: float = 1.225, a_sound: float = 340.3, mu: float = 1.789e-5, num_elements: int = 24) -> BEMTResult:
    r_edges = np.linspace(root_cutout, radius, num_elements + 1)
    r_stations = 0.5 * (r_edges[:-1] + r_edges[1:])
    dr = np.diff(r_edges)
    r_norm = r_stations / radius

    chords = c_root + (c_root * taper - c_root) * ((r_stations - root_cutout) / max(1e-4, radius - root_cutout))
    thetas = np.radians(pitch_75_deg + twist_deg * (r_norm - 0.75))

    omega = rpm * (np.pi / 30.0)
    v_tip = max(omega * radius, 1e-4)
    lambda_c = v_axial_ms / v_tip
    sigma_r = (num_blades * chords) / (np.pi * radius)

    re_mid = (
        rho
        * (omega * 0.75 * radius)
        * (c_root * (1.0 + taper) * 0.5)
        / max(mu, 1e-6)
    )
    cl_slope = airfoil.get_cl_slope(re_mid) if hasattr(airfoil, 'get_cl_slope') else 5.73
    term_ind = sigma_r * cl_slope / 16.0
    lambda_i = np.sign(thetas) * np.sqrt(np.abs(term_ind**2 + (sigma_r * cl_slope * thetas * r_norm / 8.0))) - term_ind
    lambda_tot = lambda_c + np.clip(lambda_i, -0.2, 0.5)

    u_t = omega * r_stations
    for _ in range(16):
        f = (num_blades * 0.5) * (1.0 - r_norm) / np.maximum(np.abs(lambda_tot), 1e-4)
        f_loss = np.maximum((2.0 / np.pi) * np.arccos(np.exp(-np.clip(f, 0.0, 18.0))), 1e-3)
        t1 = (sigma_r * cl_slope) / (16.0 * f_loss)
        t2 = (sigma_r * cl_slope * thetas * r_norm) / (8.0 * f_loss)
        lambda_new = np.sign(thetas) * (np.sqrt(np.maximum(1e-6, t1**2 + np.abs(t2))) - t1) + lambda_c
        lambda_tot = 0.6 * lambda_tot + 0.4 * lambda_new

    u_p = lambda_tot * v_tip
    phi = np.arctan2(u_p, u_t)
    alpha = thetas - phi

    u_res_sq = u_t**2 + u_p**2
    mach_r = np.sqrt(u_res_sq) / a_sound
    re_r = (rho * np.sqrt(u_res_sq) * chords) / mu
    cl, cd, _ = airfoil.evaluate(alpha, re_r, mach_r)

    q_blade = num_blades * 0.5 * rho * u_res_sq * chords
    cos_p, sin_p = np.cos(phi), np.sin(phi)
    dt_dr = q_blade * (cl * cos_p - cd * sin_p)
    dfx_dr = q_blade * (cd * cos_p + cl * sin_p)
    thrust = np.sum(dt_dr * dr)
    torque = np.sum(r_stations * dfx_dr * dr)
    power = omega * torque

    disk_area = np.pi * (radius ** 2)
    ct = thrust / max(rho * disk_area * (v_tip ** 2), 1e-6)
    cp = power / max(rho * disk_area * (v_tip ** 3), 1e-6)
    
    if ct > 1e-4 and cp > 1e-5 and v_axial_ms == 0.0:
        p_ideal_w = (max(thrust, 0.0) ** 1.5) / np.sqrt(2.0 * rho * disk_area)
        fm_raw = p_ideal_w / max(power, 1e-3)
        fm = float(np.clip(fm_raw, 0.0, 0.85))
    else:
        fm = 0.0

    eta = float(np.clip((thrust * v_axial_ms) / (power + 1e-6), 0.0, 0.92)) if v_axial_ms > 5.0 else 0.0

    return BEMTResult(
        thrust_N=thrust, torque_Nm=torque, power_kW=power * 1e-3, CT=ct, CP=cp, FM=fm, eta_prop=eta,
        r_stations=r_stations, chords=chords, thetas_deg=np.degrees(thetas), phi_deg=np.degrees(phi),
        alpha_deg=np.degrees(alpha), lambda_i=(lambda_tot - lambda_c), lambda_tot=lambda_tot,
        cl=cl, cd=cd, re_r=re_r, dt_dr=dt_dr, dp_dr=(omega * r_stations * dfx_dr), dq_dr=(r_stations * dfx_dr),
        dCT_dr=dt_dr / (rho * (omega**2) * (radius**4) * np.pi * dr / radius),
        dCP_dr=(omega * r_stations * dfx_dr) / (rho * (omega**3) * (radius**5) * np.pi * dr / radius),
        mach_helical=mach_r
    )

def trim_rotor_collective(target_thrust_n: float, radius: float, root_cutout: float, num_blades: int,
                          c_root: float, taper: float, twist_deg: float, rpm: float, v_axial_ms: float,
                          airfoil: AirfoilModel, rho: float, a_sound: float, mu: float,
                          pitch_bounds: Tuple[float, float] = (-5.0, 75.0)) -> Tuple[float, BEMTResult]:
    """Finds exact collective pitch theta_0.75 where BEMT thrust equals vehicle demand."""
    def residual(th_guess):
        res = run_bemt_solver(radius, root_cutout, num_blades, c_root, taper, th_guess, twist_deg, rpm, v_axial_ms, airfoil, rho, a_sound, mu, num_elements=16)
        return res.thrust_N - target_thrust_n

    try:
        th_opt = brentq(residual, pitch_bounds[0], pitch_bounds[1], xtol=0.02, maxiter=35)
    except Exception:
        # Fallback to coarse sweep minimum residual if brentq bracketing fails
        coarse_th = np.linspace(pitch_bounds[0], pitch_bounds[1], 40)
        coarse_err = [abs(residual(t)) for t in coarse_th]
        th_opt = float(coarse_th[np.argmin(coarse_err)])

    trimmed_bemt = run_bemt_solver(radius, root_cutout, num_blades, c_root, taper, th_opt, twist_deg, rpm, v_axial_ms, airfoil, rho, a_sound, mu, num_elements=24)
    return th_opt, trimmed_bemt

# ------------------------------------------------------------------------------
# 3. WIDGET CONTROLS (TRIMMED VEHICLE SIZING INTERFACE)
# ------------------------------------------------------------------------------
style_w = {'description_width': '160px'}
layout_w = widgets.Layout(width='340px', margin='1px 0px')

full_airfoil_catalog = [
    "Boeing-Vertol VR-12",
    "NASA SC(2)-0010", "NASA SC(2)-0012", "NASA SC(2)-0410", "NASA SC(2)-0412",
    "NACA 0009", "NACA 0012", "NACA 23012", "NACA 23015", "NACA 64-A010", "NACA 64-A012",
    "ONERA OA209", "ONERA OA212", "Sikorsky SC1095",
    "NACA 0015", "NACA 0018", "NACA 2412", "NACA 4412", "NACA 4415",
    "Clark Y", "Clark YH", "USA 35B"
]

w_airfoil = widgets.Dropdown(options=full_airfoil_catalog, value="Boeing-Vertol VR-12", description='Rotor Airfoil:', style=style_w, layout=layout_w)
w_radius = widgets.FloatSlider(value=4.42, min=2.5, max=6.5, step=0.01, description='Rotor Radius R [m]:', style=style_w, layout=layout_w, continuous_update=False)
w_nblades = widgets.IntSlider(value=4, min=2, max=6, step=1, description='Number of Blades Nb:', style=style_w, layout=layout_w, continuous_update=False)
w_chord = widgets.FloatSlider(value=0.55, min=0.15, max=0.85, step=0.01, description='Root Chord c_0 [m]:', style=style_w, layout=layout_w, continuous_update=False)
w_taper = widgets.FloatSlider(value=0.55, min=0.30, max=1.00, step=0.05, description='Blade Taper (c_tip/c_root):', style=style_w, layout=layout_w, continuous_update=False)
w_twist = widgets.FloatSlider(value=-22.0, min=-45.0, max=0.0, step=0.5, description='Blade Twist [°]:', style=style_w, layout=layout_w, continuous_update=False)
w_rpm_hov = widgets.FloatSlider(value=540.0, min=300.0, max=650.0, step=5.0, description='Hover RPM:', style=style_w, layout=layout_w, continuous_update=False)
w_rpm_cr = widgets.FloatSlider(value=220.0, min=150.0, max=450.0, step=5.0, description='Cruise RPM:', style=style_w, layout=layout_w, continuous_update=False)

w_s_wing = widgets.FloatSlider(value=26.0, min=12.0, max=50.0, step=0.5, description='Wing Area [m²]:', style=style_w, layout=layout_w, continuous_update=False)
w_ar = widgets.FloatSlider(value=7.8, min=4.0, max=14.0, step=0.1, description='Wing Aspect Ratio:', style=style_w, layout=layout_w, continuous_update=False)
w_sweep = widgets.FloatSlider(value=2.5, min=-5.0, max=25.0, step=0.5, description='Wing Sweep [°]:', style=style_w, layout=layout_w, continuous_update=False)
w_tc = widgets.FloatSlider(value=22.5, min=14.0, max=28.0, step=0.5, description='Wing Root t/c [%]:', style=style_w, layout=layout_w, continuous_update=False)

w_v_cruise = widgets.FloatSlider(value=450.0, min=320.0, max=550.0, step=10.0, description='Cruise Speed [km/h]:', style=style_w, layout=layout_w, continuous_update=False)
w_range = widgets.FloatSlider(value=1000.0, min=500.0, max=2000.0, step=50.0, description='Mission Range [km]:', style=style_w, layout=layout_w, continuous_update=False)
w_ceiling = widgets.FloatSlider(value=7000.0, min=4000.0, max=9000.0, step=250.0, description='Service Ceiling [m]:', style=style_w, layout=layout_w, continuous_update=False)
w_p_inst = widgets.FloatSlider(value=5000.0, min=2000.0, max=9000.0, step=100.0, description='Installed Power [kW]:', style=style_w, layout=layout_w, continuous_update=False)
w_sfc = widgets.FloatSlider(value=0.285, min=0.18, max=0.45, step=0.005, description='Engine SFC [kg/kW/hr]:', style=style_w, layout=layout_w, continuous_update=False)

w_bemt_mode = widgets.ToggleButtons(
    options=['Hover', 'Cruise', 'Both'],
    value='Both',
    description='Plot Mode:',
    style={'description_width': '80px', 'button_width': '80px'},
    layout=widgets.Layout(margin='2px 0px 6px 0px')
)

out_cad = widgets.Output()
out_heatmaps = widgets.Output()
out_blade_3d = widgets.Output()
out_stats = widgets.Output()
out_hover_maps = widgets.Output()
out_cruise_maps = widgets.Output()
out_comparisons = widgets.Output()

tabs_designer = widgets.Tab(
    children=[out_cad, out_heatmaps, out_blade_3d, out_stats, out_hover_maps, out_cruise_maps, out_comparisons],
    layout=widgets.Layout(flex='1 1 auto', min_width='800px', margin='0 0 0 16px')
)
tabs_designer.set_title(0, 'General Arrangement & Card')
tabs_designer.set_title(1, 'Feasibility Heatmap Matrix')
tabs_designer.set_title(2, '3D Lofted Blade & BEMT Analysis')
tabs_designer.set_title(3, 'Statistical Regressions')
tabs_designer.set_title(4, '6.1 Hover Performance Maps')
tabs_designer.set_title(5, '6.2 Forward-Flight Propeller Maps')
tabs_designer.set_title(6, '6.3 Comparable Rotor Benchmarks')

_airfoil_cache = {}

plt.ioff()
def update_designer_dashboard(*args):
    plt.ioff()
    selected_af = w_airfoil.value
    if selected_af not in _airfoil_cache:
        _airfoil_cache[selected_af] = AirfoilModel(airfoil_name=selected_af, ncrit=9)
    af_model = _airfoil_cache[selected_af]

    R = float(w_radius.value)
    Nb = int(w_nblades.value)
    c_0 = float(w_chord.value)
    taper = float(w_taper.value)
    twist = float(w_twist.value)
    rpm_hov = float(w_rpm_hov.value)
    rpm_cr = float(w_rpm_cr.value)
    S_w = float(w_s_wing.value)
    AR = float(w_ar.value)
    sweep_deg = float(w_sweep.value)
    tc_pct = float(w_tc.value)
    v_cr_kmh = float(w_v_cruise.value)
    v_cr_ms = v_cr_kmh / 3.6
    range_m = float(w_range.value) * 1000.0
    Ceiling_m = float(w_ceiling.value)
    P_inst_kW = float(w_p_inst.value)
    sfc_si = (float(w_sfc.value)) / 3.6e6
    g = 9.80665

    rho_sl, a_sl, _, mu_sl = isa_atmosphere(0.0)
    rho_ceil, a_ceil, _, mu_ceil = isa_atmosphere(Ceiling_m)
    P_avail_ceil_kW = P_inst_kW * ((rho_ceil / rho_sl) ** 1.05)

    b_wing = np.sqrt(S_w * AR)
    c_root_w = (2.0 * S_w) / (b_wing * (1.0 + 0.65))
    c_tip_w = c_root_w * 0.65
    mac = (2.0 / 3.0) * c_root_w * (1.0 + 0.65 + 0.65**2) / (1.0 + 0.65)
    l_arm = 0.45 * FUSE_LEN
    s_htail = (0.85 * S_w * mac) / l_arm
    fuse_clearance = (0.5 * b_wing - R) - (0.5 * FUSE_W)

    # 1. Aircraft Sizing Convergence Loop (Baseline L/D = 12.0)
    W0_cur = 9000.0
    ld_cr = 12.0
    eta_prop_est = 0.75

    for _ in range(25):
        m_f = np.exp(-(range_m * sfc_si * g) / (eta_prop_est * ld_cr))
        ff = 1.06 * (1.0 - (0.985 * 0.990 * 0.975 * m_f * 0.990 * 0.985))

        we_base = power_law(W0_cur, *popt_we)
        w_blade_penalty = 0.035 * (4.20 / R) ** (-0.5)
        we_total = we_base + w_blade_penalty - 0.035
        denom = 1.0 - ff - we_total
        if denom <= 0.05:
            W0_cur = 35000.0
            break
        W_next = PAYLOAD_FIXED / denom
        if abs(W_next - W0_cur) / W0_cur < 1e-4:
            W0_cur = W_next
            break
        W0_cur = W_next

    # 2. Physics-Based Thrust Demands for Vehicle Equilibrium
    # Hover: must balance weight + 8% vertical download
    thrust_hover_target = (W0_cur * g * 1.08) / N_ROTORS
    # Cruise: must balance total airframe drag at heavy cruise weight
    total_drag_cruise = (W0_cur * g) / ld_cr
    thrust_cruise_target = total_drag_cruise / N_ROTORS

    # 3. Dual-Trim BEMT Execution
    th_hov_trimmed, bemt_hov = trim_rotor_collective(
        thrust_hover_target, R, 0.45, Nb, c_0, taper, twist, rpm_hov, 0.0,
        af_model, rho_sl, a_sl, mu_sl, pitch_bounds=(-2.0, 24.0)
    )
    th_cr_trimmed, bemt_cr = trim_rotor_collective(
        thrust_cruise_target, R, 0.45, Nb, c_0, taper, twist, rpm_cr, v_cr_ms,
        af_model, rho_sl, a_sl, mu_sl, pitch_bounds=(30.0, 75.0)
    )

    eta_prop = max(bemt_cr.eta_prop, 0.70)
    fm_hov = max(bemt_hov.FM, 0.65)

    C_fe = 0.0030
    S_wet_total = S_WET_FUSE + 2.05 * S_w + 2.0 * s_htail
    cd0_dyn = C_fe * (S_wet_total / S_w)

    m_empty = W0_cur * power_law(W0_cur, *popt_we)
    m_fuel = W0_cur * ff
    A_disc_total = N_ROTORS * np.pi * (R**2)
    disc_loading = W0_cur / A_disc_total
    solidity = (Nb * c_0 * (1.0 + taper) * 0.5) / (np.pi * R)
    cur_pw = P_inst_kW / W0_cur
    V_flutter = 320.0 * ((tc_pct / 15.0) ** 1.5)

    v_tip_hov = (rpm_hov * (np.pi / 30.0)) * R
    v_tip_cr = (rpm_cr * (np.pi / 30.0)) * R
    m_tip_hov = v_tip_hov / a_sl
    m_tip_cr_helical = np.sqrt(v_tip_cr**2 + v_cr_ms**2) / a_sl
    p_hov_total_req = (2.0 * bemt_hov.power_kW) / 0.95
    p_cr_total_req = (2.0 * bemt_cr.power_kW) / 0.95

    global SIZED_VEHICLE
    SIZED_VEHICLE = {
        "MTOW": W0_cur, "Empty_Mass": m_empty, "Fuel_Mass": m_fuel,
        "Payload": PAYLOAD_FIXED, "Range_km": w_range.value, "V_max_kmh": v_cr_kmh,
        "Ceiling_m": Ceiling_m, "Rotor_Radius": R, "Num_Blades": Nb, "Blade_Chord": c_0,
        "Twist_deg": twist, "Pitch_Hover": th_hov_trimmed, "Pitch_Cruise": th_cr_trimmed, "Wing_Area": S_w,
        "Aspect_Ratio": AR, "Wingspan": b_wing, "Wing_tc": tc_pct, "Installed_Power": P_inst_kW,
        "CD0": cd0_dyn, "SFC": w_sfc.value, "Oswald_e": 0.85, "Airfoil": af_model,
        "BEMT_Hover": bemt_hov, "BEMT_Cruise": bemt_cr
    }

    # ==========================================================================
    # TAB 0: 2D GENERAL ARRANGEMENT & CARD
    # ==========================================================================
    out_cad.clear_output(wait=True)
    with out_cad:
        fig_c = plt.figure(figsize=(16.5, 9.6), dpi=100)
        gs_cad = fig_c.add_gridspec(2, 2, width_ratios=[1.3, 1.0])
        ax_top = fig_c.add_subplot(gs_cad[0, 0])
        ax_front = fig_c.add_subplot(gs_cad[1, 0])
        ax_card = fig_c.add_subplot(gs_cad[:, 1])

        collision_warn = fuse_clearance < 0.25

        fx = np.array([0, 0.15 * FUSE_LEN, 0.75 * FUSE_LEN, FUSE_LEN, 0.75 * FUSE_LEN, 0.15 * FUSE_LEN, 0])
        fy = np.array([0, 0.5 * FUSE_W, 0.5 * FUSE_W, 0, -0.5 * FUSE_W, -0.5 * FUSE_W, 0])
        ax_top.fill(fx, fy, color='#dc3545' if collision_warn else '#ced4da', alpha=0.85, edgecolor='k', lw=1.3)

        wx_le = 0.35 * FUSE_LEN
        tip_x_off = 0.5 * b_wing * np.tan(np.radians(sweep_deg))
        wx = [wx_le, wx_le + tip_x_off, wx_le + tip_x_off + c_tip_w, wx_le + c_root_w, wx_le + tip_x_off + c_tip_w, wx_le + tip_x_off, wx_le]
        wy = [0, 0.5 * b_wing, 0.5 * b_wing, 0, -0.5 * b_wing, -0.5 * b_wing, 0]
        ax_top.fill(wx, wy, color='#9ec5fe', alpha=0.8, edgecolor='blue', lw=1.3)

        b_ht = np.sqrt(s_htail * 4.2)
        c_ht = s_htail / b_ht
        hx_le = wx_le + l_arm
        ax_top.fill([hx_le, hx_le, hx_le + c_ht, hx_le + c_ht], [-0.5 * b_ht, 0.5 * b_ht, 0.5 * b_ht, -0.5 * b_ht], color='#6c757d', alpha=0.85, edgecolor='k')

        rcs = [(wx_le + tip_x_off, 0.5 * b_wing), (wx_le + tip_x_off, -0.5 * b_wing)]
        for rx, ry in rcs:
            col = 'crimson' if collision_warn else 'darkgreen'
            ax_top.add_patch(plt.Circle((rx, ry), R, color=col, fill=True, alpha=0.12, linestyle='--', lw=1.4))
            ax_top.add_patch(plt.Circle((rx, ry), R, color=col, fill=False, linestyle='--', lw=1.4))
            ax_top.plot(rx, ry, 'o', color=col, ms=4)

        ax_top.set_aspect('equal')
        ax_top.set_xlim(-1, FUSE_LEN + 2)
        ax_top.set_ylim(-0.55 * b_wing - R, 0.55 * b_wing + R)
        ax_top.set_title('Top View: Helicopter Mode' + (f' [!] TIP CLEARANCE: {fuse_clearance:.2f}m' if collision_warn else f' (Clearance: {fuse_clearance:.2f}m)'), fontsize=10, fontweight='bold')
        ax_top.set_xlabel('X [m]', fontsize=9); ax_top.set_ylabel('Y [m]', fontsize=9); ax_top.grid(True, linestyle=':', alpha=0.5)

        th_fuse = np.linspace(0, 2 * np.pi, 60)
        fx_front = (0.5 * FUSE_W) * np.cos(th_fuse)
        fz_front = (0.5 * FUSE_H) * np.sin(th_fuse)
        ax_front.fill(fx_front, fz_front, color='#ced4da', alpha=0.9, edgecolor='k', lw=1.4)

        wing_z = 0.35 * FUSE_H
        ax_front.fill([-0.5 * b_wing, 0.5 * b_wing, 0.5 * b_wing, -0.5 * b_wing], [wing_z, wing_z, wing_z + 0.225 * c_root_w, wing_z + 0.225 * c_root_w], color='#9ec5fe', alpha=0.85, edgecolor='blue', lw=1.3)

        for ny in [-0.5 * b_wing, 0.5 * b_wing]:
            p_col = 'crimson' if collision_warn else '#198754'
            ax_front.add_patch(plt.Circle((ny, wing_z), R, color=p_col, fill=True, alpha=0.15, linestyle='-', lw=1.8))
            ax_front.add_patch(plt.Circle((ny, wing_z), R, color=p_col, fill=False, linestyle='-', lw=1.8))
            ax_front.plot(ny, wing_z, 'o', color='black', ms=4.5)

        ax_front.set_aspect('equal')
        ax_front.set_xlim(-0.55 * b_wing - R, 0.55 * b_wing + R)
        ax_front.set_ylim(-R - 0.5, R + 2.0)
        ax_front.set_title(f'Front View: Cruise Mode (R={R:.2f}m)', fontsize=10, fontweight='bold')
        ax_front.set_xlabel('Y [m]', fontsize=9); ax_front.set_ylabel('Z [m]', fontsize=9); ax_front.grid(True, linestyle=':', alpha=0.5)

        ax_card.axis('off')
        power_margin_hov = P_inst_kW - p_hov_total_req
        power_margin_cr = P_inst_kW - p_cr_total_req
        is_feasible = (power_margin_hov >= 0.0) and (power_margin_cr >= 0.0) and (not collision_warn)

        card_str = (
            "TROOP TRANSPORTER (10 PAX + 2 CREW) SIZING AUDIT\n"
            "===================================================\n"
            f"STATUS: {'[+] SIZING FEASIBLE' if is_feasible else '[-] PERFORMANCE / CLEARANCE DEFICIT'}\n"
            f"ROTOR AIRFOIL:        {af_model.airfoil_name}\n"
            "---------------------------------------------------\n"
            "WEIGHT & PACKAGING BREAKDOWN:\n"
            f"  • Takeoff Weight (MTOW): {W0_cur:8.1f} kg\n"
            f"  • Operating Empty (We):  {m_empty:8.1f} kg ({m_empty/W0_cur*100:.1f}%)\n"
            f"  • Mission Fuel (Wf):     {m_fuel:8.1f} kg ({m_fuel/W0_cur*100:.1f}%)\n"
            f"  • Fixed Payload:         {PAYLOAD_FIXED:8.1f} kg (12 Troops)\n"
            f"  • Wingspan (b):          {b_wing:8.2f} m\n"
            f"  • Fuselage Clearance:    {fuse_clearance:8.2f} m\n\n"
            "AUTO-TRIMMED ROTOR & PROPULSION AUDIT:\n"
            f"  • Cruise L/D (Fixed):    {ld_cr:8.2f}\n"
            f"  • Rotor Radius (R):      {R:6.2f} m (Blades: {Nb})\n"
            f"  • Disc Loading (DL):     {disc_loading:6.1f} kg/m²\n"
            f"  • Trim Hover Pitch:      {th_hov_trimmed:6.2f}° (Target: {thrust_hover_target:.0f} N)\n"
            f"  • Trim Cruise Pitch:     {th_cr_trimmed:6.2f}° (Target: {thrust_cruise_target:.0f} N)\n"
            f"  • Hover Tip Mach:        {m_tip_hov:6.2f} (RPM: {rpm_hov:.0f})\n"
            f"  • Cruise Helical Mach:   {m_tip_cr_helical:6.2f} (RPM: {rpm_cr:.0f})\n"
            f"  • Hover Total Power:     {p_hov_total_req:6.1f} kW\n"
            f"  • Cruise Total Power:    {p_cr_total_req:6.1f} kW\n"
            f"  • Cruise Prop Efficiency:{eta_prop*100:6.1f} %\n"
            f"  • Power Margin (Hover):  {power_margin_hov:6.1f} kW\n"
            f"  • Power Margin (Cruise): {power_margin_cr:6.1f} kW\n"
        )
        ax_card.text(0.02, 0.98, card_str, fontfamily='monospace', fontsize=9.6, verticalalignment='top',
                     bbox=dict(boxstyle='round,pad=0.6', facecolor='#f8fff9' if is_feasible else '#fff8f8', edgecolor='#198754' if is_feasible else '#dc3545', lw=1.4))

        plt.tight_layout()
        plt.show()
        plt.close(fig_c)

    # ==========================================================================
    # TAB 1: 4-PANEL FEASIBILITY HEATMAP MATRIX
    # ==========================================================================
    out_heatmaps.clear_output(wait=True)
    with out_heatmaps:
        fig_hm, ((ax_h1, ax_h2), (ax_h3, ax_h4)) = plt.subplots(2, 2, figsize=(16.5, 9.6), dpi=100)

        # ---------------- Panel 1: Dual-Pitch Stall Margin ----------------
        n_p = 25
        p_hov_vec = np.linspace(2.0, 24.0, n_p)
        p_cr_vec = np.linspace(20.0, 70.0, n_p)
        P_HOV_G, P_CR_G = np.meshgrid(p_hov_vec, p_cr_vec)
        STALL_MARGIN_G = np.zeros_like(P_HOV_G)

        for i in range(n_p):
            for j in range(n_p):
                res_h = run_bemt_solver(R, 0.45, Nb, c_0, taper, P_HOV_G[i, j], twist, rpm_hov, 0.0, af_model, rho_sl, a_sl, num_elements=10)
                res_c = run_bemt_solver(R, 0.45, Nb, c_0, taper, P_CR_G[i, j], twist, rpm_cr, v_cr_ms, af_model, rho_sl, a_sl, num_elements=10)
                STALL_MARGIN_G[i, j] = 12.0 - max(np.max(np.abs(res_h.alpha_deg)), np.max(np.abs(res_c.alpha_deg)))

        cp1 = ax_h1.contourf(P_HOV_G, P_CR_G, STALL_MARGIN_G, levels=14, cmap='RdYlGn', alpha=0.85)
        cbar1 = plt.colorbar(cp1, ax=ax_h1)
        cbar1.set_label(r'Stall Margin ($12^\circ - \max|\alpha|$) [°]', fontsize=8.5)
        ax_h1.contour(P_HOV_G, P_CR_G, STALL_MARGIN_G, levels=[0.0], colors='black', linewidths=2.0, linestyles='--')
        ax_h1.scatter(th_hov_trimmed, th_cr_trimmed, color='cyan', edgecolors='black', s=140, marker='*', zorder=10)
        
        legend_h1 = [
            Line2D([0], [0], color='black', lw=2.0, ls='--', label=r'Stall Boundary ($|\alpha| = 12^\circ$)'),
            Patch(facecolor='forestgreen', edgecolor='k', alpha=0.6, label='Clean Flow Regime'),
            Patch(facecolor='crimson', edgecolor='k', alpha=0.6, label='Stall Incursion'),
            Line2D([0], [0], marker='*', color='w', markerfacecolor='cyan', markeredgecolor='k', markersize=12, label=f'Trimmed: ({th_hov_trimmed:.1f}°, {th_cr_trimmed:.1f}°)')
        ]
        ax_h1.legend(handles=legend_h1, loc='lower left', fontsize=8, framealpha=0.92)
        ax_h1.set_xlabel(r'Hover Pitch $\theta_{0.75\mathrm{,hov}}$ [°]', fontsize=9)
        ax_h1.set_ylabel(r'Cruise Pitch $\theta_{0.75\mathrm{,cr}}$ [°]', fontsize=9)
        ax_h1.set_title('A. Rotor Dual-Pitch Stall Feasibility Envelope', fontsize=10.5, fontweight='bold')
        ax_h1.grid(True, ls=':', alpha=0.6)

        # ---------------- Panel 2: Sizing Carpet Island ----------------
        N_grid = 16
        dl_vec = np.linspace(25.0, 200.0, N_grid)
        ws_vec = np.linspace(100.0, 700.0, N_grid)
        DL_g, WS_g = np.meshgrid(dl_vec, ws_vec)

        P_avail_ceil = P_inst_kW * ((rho_ceil / rho_sl) ** 1.05)
        w_mat = np.full_like(DL_g, 9000.0)

        for _ in range(12):
            s_w_mat = w_mat / WS_g
            r_mat = np.sqrt((w_mat / DL_g) / (2 * np.pi))
            mf_mat = np.exp(-(range_m * sfc_si * g) / (eta_prop * ld_cr))
            ff_mat = 1.06 * (1.0 - (0.985 * 0.990 * 0.975 * mf_mat * 0.990 * 0.985))
            we_mat = power_law(w_mat, *popt_we) + 0.035 * (4.20 / r_mat)**(-0.5) - 0.035
            denom_mat = 1.0 - ff_mat - we_mat
            w_mat = np.where(denom_mat <= 0.05, 35000.0, PAYLOAD_FIXED / np.maximum(denom_mat, 0.05))

        W0_g = w_mat
        s_w_final = W0_g / WS_g
        b_w_final = np.sqrt(s_w_final * AR)
        r_final = np.sqrt((W0_g / DL_g) / (2 * np.pi))
        Clearance_g = (0.5 * b_w_final - r_final) - (0.5 * FUSE_W)
        P_req_ceil_g = ((W0_g * g / ld_cr) * v_cr_ms) / (eta_prop * 1000.0) + (W0_g * g * 1.5) / 1000.0

        cp2 = ax_h2.contourf(DL_g, WS_g, W0_g, levels=18, cmap='viridis_r', alpha=0.90)
        cbar2 = plt.colorbar(cp2, ax=ax_h2)
        cbar2.set_label(r'Gross Takeoff Weight $W_0$ [kg]', fontsize=8.5)
        ax_h2.contour(DL_g, WS_g, Clearance_g, levels=[0.25], colors='red', linewidths=2.0)
        ax_h2.contour(DL_g, WS_g, P_req_ceil_g, levels=[P_avail_ceil], colors='darkorange', linewidths=2.0, linestyles='--')
        ax_h2.scatter(disc_loading, W0_cur / S_w, color='lime', edgecolors='black', s=140, marker='*', zorder=10)
        
        legend_h2 = [
            Line2D([0], [0], color='red', lw=2.0, label=r'Fuselage Clearance ($\geq 0.25$ m)'),
            Line2D([0], [0], color='darkorange', lw=2.0, ls='--', label=rf'{Ceiling_m:.0f} m Ceiling Limit ($P \leq P_{{\mathrm{{avail}}}}$)'),
            Line2D([0], [0], marker='*', color='w', markerfacecolor='lime', markeredgecolor='k', markersize=12, label=f'Current: DL={disc_loading:.1f}, W/S={W0_cur/S_w:.1f}')
        ]
        ax_h2.legend(handles=legend_h2, loc='upper right', fontsize=8, framealpha=0.92)
        ax_h2.set_xlabel(r'Disc Loading $DL$ [$\mathrm{kg/m^2}$]', fontsize=9)
        ax_h2.set_ylabel(r'Wing Loading $W/S$ [$\mathrm{kg/m^2}$]', fontsize=9)
        ax_h2.set_title(f'B. Sizing Carpet Island ({v_cr_kmh:.0f} km/h | {Ceiling_m:.0f} m)', fontsize=10.5, fontweight='bold')
        ax_h2.grid(True, ls=':', alpha=0.6)

        # ---------------- Panel 3: Rotor Geometry & Efficiency ----------------
        r_grid_vec = np.linspace(3.0, 5.5, n_p)
        c_grid_vec = np.linspace(0.20, 0.65, n_p)
        R_G, C_G = np.meshgrid(r_grid_vec, c_grid_vec)
        SOL_G = (Nb * C_G * (1.0 + taper) * 0.5) / (np.pi * R_G)
        CLEAR_G = (0.5 * b_wing - R_G) - (0.5 * FUSE_W)
        FM_G = np.zeros_like(R_G)

        for i in range(n_p):
            for j in range(n_p):
                res_g = run_bemt_solver(R_G[i, j], 0.45, Nb, C_G[i, j], taper, th_hov_trimmed, twist, rpm_hov, 0.0, af_model, rho_sl, a_sl, num_elements=10)
                FM_G[i, j] = res_g.FM

        cp3 = ax_h3.contourf(R_G, C_G, FM_G, levels=14, cmap='magma', alpha=0.85)
        cbar3 = plt.colorbar(cp3, ax=ax_h3)
        cbar3.set_label('Hover Figure of Merit (FM)', fontsize=8.5)
        ax_h3.contour(R_G, C_G, CLEAR_G, levels=[0.25], colors='red', linewidths=2.0)
        ax_h3.contour(R_G, C_G, SOL_G, levels=[0.06, 0.09, 0.12], colors='cyan', linewidths=1.5, linestyles=':')
        ax_h3.scatter(R, c_0, color='lime', edgecolors='black', s=140, marker='*', zorder=10)
        
        legend_h3 = [
            Line2D([0], [0], color='red', lw=2.0, label=r'Tip Clearance Limit ($0.25$ m)'),
            Line2D([0], [0], color='cyan', lw=1.5, ls=':', label=r'Solidity Contours ($\sigma$)'),
            Line2D([0], [0], marker='*', color='w', markerfacecolor='lime', markeredgecolor='k', markersize=12, label=f'Current: R={R:.2f}m, c={c_0:.2f}m')
        ]
        ax_h3.legend(handles=legend_h3, loc='lower right', fontsize=8, framealpha=0.92)
        ax_h3.set_xlabel('Rotor Radius R [m]', fontsize=9)
        ax_h3.set_ylabel(r'Blade Root Chord $c_0$ [m]', fontsize=9)
        ax_h3.set_title('C. Rotor Geometry vs. Hover Efficiency', fontsize=10.5, fontweight='bold')
        ax_h3.grid(True, ls=':', alpha=0.6)

        # ---------------- Panel 4: Cruise Propulsive Matching ----------------
        v_grid_vec = np.linspace(340.0, 560.0, n_p)
        rpm_grid_vec = np.linspace(150.0, 420.0, n_p)
        V_G, RPM_G = np.meshgrid(v_grid_vec, rpm_grid_vec)

        vg_ms_grid = V_G / 3.6
        v_t_grid = (RPM_G * (np.pi / 30.0)) * R
        M_TIP_G = np.sqrt(v_t_grid**2 + vg_ms_grid**2) / a_sl
        ETA_G = np.zeros_like(V_G)

        for i in range(n_p):
            for j in range(n_p):
                res_v = run_bemt_solver(R, 0.45, Nb, c_0, taper, th_cr_trimmed, twist, RPM_G[i, j], vg_ms_grid[i, j], af_model, rho_sl, a_sl, num_elements=10)
                ETA_G[i, j] = max(res_v.eta_prop, 0.0)

        PWR_CR_G = ((W0_cur * g / ld_cr) * vg_ms_grid) / (np.maximum(ETA_G, 0.40) * 1000.0 * 0.95)

        cp4 = ax_h4.contourf(V_G, RPM_G, ETA_G * 100.0, levels=14, cmap='Blues', alpha=0.85)
        cbar4 = plt.colorbar(cp4, ax=ax_h4)
        cbar4.set_label(r'Cruise Prop Efficiency $\eta_{\mathrm{prop}}$ [%]', fontsize=8.5)
        ax_h4.contour(V_G, RPM_G, M_TIP_G, levels=[0.82], colors='red', linewidths=2.0)
        ax_h4.contour(V_G, RPM_G, PWR_CR_G, levels=[P_inst_kW], colors='darkorange', linewidths=2.0, linestyles='--')
        ax_h4.scatter(v_cr_kmh, rpm_cr, color='yellow', edgecolors='black', s=140, marker='*', zorder=10)
        
        legend_h4 = [
            Line2D([0], [0], color='red', lw=2.0, label=r'Helical Mach Limit ($M_{\mathrm{tip}} = 0.82$)'),
            Line2D([0], [0], color='darkorange', lw=2.0, ls='--', label=rf'Installed Power Limit ({P_inst_kW:.0f} kW)'),
            Line2D([0], [0], marker='*', color='w', markerfacecolor='yellow', markeredgecolor='k', markersize=12, label=f'Current: ({v_cr_kmh:.0f} km/h, {rpm_cr:.0f} RPM)')
        ]
        ax_h4.legend(handles=legend_h4, loc='lower right', fontsize=8, framealpha=0.92)
        ax_h4.set_xlabel(r'Cruise Speed $V_{\mathrm{cruise}}$ [km/h]', fontsize=9)
        ax_h4.set_ylabel('Cruise Rotor Speed [RPM]', fontsize=9)
        ax_h4.set_title(r'D. Cruise Matching ($M_{\mathrm{tip}} \leq 0.82$ Limit in Red)', fontsize=10.5, fontweight='bold')
        ax_h4.grid(True, ls=':', alpha=0.6)

        plt.tight_layout()
        plt.show()
        plt.close(fig_hm)

    # ==========================================================================
    # TAB 2: 3D LOFTED BLADE + 8-PANEL BEMT DASHBOARD
    # ==========================================================================
    out_blade_3d.clear_output(wait=True)
    with out_blade_3d:
        fig_bemt = plt.figure(figsize=(16.5, 9.6), dpi=100)
        gs = fig_bemt.add_gridspec(4, 4, width_ratios=[1.25, 1.25, 1.0, 1.0])

        ax_top_rotor = fig_bemt.add_subplot(gs[0:1, 0:2])
        ax_3d = fig_bemt.add_subplot(gs[1:4, 0:2], projection='3d')

        ax_aoa  = fig_bemt.add_subplot(gs[0, 2])
        ax_t    = fig_bemt.add_subplot(gs[0, 3])
        ax_re   = fig_bemt.add_subplot(gs[1, 2])
        ax_pq   = fig_bemt.add_subplot(gs[1, 3])
        ax_coef = fig_bemt.add_subplot(gs[2, 2])
        ax_ct_th= fig_bemt.add_subplot(gs[2, 3])
        ax_inf  = fig_bemt.add_subplot(gs[3, 2])
        ax_fm   = fig_bemt.add_subplot(gs[3, 3])

        mode = w_bemt_mode.value
        if mode == 'Hover':
            a_hov, a_cr = 1.0, 0.08
        elif mode == 'Cruise':
            a_hov, a_cr = 0.08, 1.0
        else:
            a_hov, a_cr = 0.95, 0.95

        # 1. 2D Top View
        angles = np.linspace(0, 2 * np.pi, Nb, endpoint=False)
        for ang in angles:
            bx = [0.45 * np.cos(ang), R * np.cos(ang)]
            by = [0.45 * np.sin(ang), R * np.sin(ang)]
            ax_top_rotor.plot(bx, by, 'b-', lw=3.0)

        ax_top_rotor.add_patch(plt.Circle((0, 0), R, color='gray', fill=False, linestyle='--', lw=1.2))
        ax_top_rotor.add_patch(plt.Circle((0, 0), 0.45, color='gray', fill=True, alpha=0.6))
        ax_top_rotor.set_aspect('equal')
        ax_top_rotor.set_xlim(-R * 1.15, R * 1.15)
        ax_top_rotor.set_ylim(-R * 1.15, R * 1.15)
        ax_top_rotor.set_title(f'Top View (Nb={Nb}, r_root=0.45m)', fontsize=9.5, fontweight='bold')
        ax_top_rotor.axis('off')

        # 2. 3D Lofted Blade
        x_af, y_af = load_airfoil_coords(af_model.airfoil_name)
        n_elem = 20
        r_edges = np.linspace(0.45, R, n_elem + 1)
        r_cen = 0.5 * (r_edges[:-1] + r_edges[1:])
        alpha_hov_elem = np.interp(r_cen, bemt_hov.r_stations, np.abs(bemt_hov.alpha_deg))
        alpha_cr_elem = np.interp(r_cen, bemt_cr.r_stations, np.abs(bemt_cr.alpha_deg))
        
        if mode == 'Hover':
            is_stalled = alpha_hov_elem >= 12.0
            blade_pitch = th_hov_trimmed
        elif mode == 'Cruise':
            is_stalled = alpha_cr_elem >= 12.0
            blade_pitch = th_cr_trimmed
        else:
            is_stalled = (alpha_hov_elem >= 12.0) | (alpha_cr_elem >= 12.0)
            blade_pitch = th_hov_trimmed

        ax_3d.plot([0, 0.45], [0, 0], [0, 0], color='#495057', lw=3.5, label='Root Hub Cutout (0.45m)')
        ax_3d.plot([0.45, R], [0, 0], [0, 0], 'k--', lw=1.2, label='Pitch Axis (25% Chord)')

        for i in range(n_elem):
            r_in, r_out = r_edges[i], r_edges[i + 1]
            c_in = c_0 + (c_0 * taper - c_0) * ((r_in - 0.45) / max(1e-4, R - 0.45))
            c_out = c_0 + (c_0 * taper - c_0) * ((r_out - 0.45) / max(1e-4, R - 0.45))
            th_in = np.radians(blade_pitch + twist * ((r_in / R) - 0.75))
            th_out = np.radians(blade_pitch + twist * ((r_out / R) - 0.75))

            x_rot_in = (0.25 - x_af) * c_in * np.cos(th_in) - y_af * c_in * np.sin(th_in)
            z_rot_in = (0.25 - x_af) * c_in * np.sin(th_in) + y_af * c_in * np.cos(th_in)
            x_rot_out = (0.25 - x_af) * c_out * np.cos(th_out) - y_af * c_out * np.sin(th_out)
            z_rot_out = (0.25 - x_af) * c_out * np.sin(th_out) + y_af * c_out * np.cos(th_out)

            poly_list = [[
                [r_in, x_rot_in[j], z_rot_in[j]], [r_in, x_rot_in[j + 1], z_rot_in[j + 1]],
                [r_out, x_rot_out[j + 1], z_rot_out[j + 1]], [r_out, x_rot_out[j], z_rot_out[j]],
            ] for j in range(len(x_af) - 1)]

            seg_color = (0.95, 0.2, 0.2, 0.8) if is_stalled[i] else (0.2, 0.7, 0.9, 0.65)
            ax_3d.add_collection3d(art3d.Poly3DCollection(poly_list, facecolors=seg_color, edgecolors=(0, 0, 0, 0.2), linewidths=0.25))

        ax_3d.set_box_aspect((2.5, 1.2, 0.9))
        ax_3d.set_xlim(0, R + 0.1); ax_3d.set_ylim(-c_0 * 0.9, c_0 * 0.9); ax_3d.set_zlim(-c_0 * 0.5, c_0 * 0.5)
        ax_3d.set_xlabel('Radius r [m]', fontsize=8); ax_3d.set_ylabel('Chordwise x [m]', fontsize=8); ax_3d.set_zlabel('Height z [m]', fontsize=8)
        ax_3d.set_title(f'Lofted 3D Blade ({af_model.airfoil_name} - {mode} Pitch)', fontsize=9.5, fontweight='bold')
        ax_3d.view_init(elev=22, azim=-60)
        ax_3d.legend(loc='upper right', fontsize=7.5)

        r_norm = bemt_hov.r_stations / R

        # 3. Aerodynamics Grid: AoA
        ax_aoa.plot(r_norm, bemt_hov.alpha_deg, 'b-', lw=1.8, alpha=a_hov, label=rf'Hover Trim ({th_hov_trimmed:.1f}°)')
        ax_aoa.plot(r_norm, bemt_cr.alpha_deg, 'm--', lw=1.8, alpha=a_cr, label=rf'Cruise Trim ({th_cr_trimmed:.1f}°)')
        ax_aoa.axhline(14.0, color='r', ls=':', label=r'$\alpha_{\mathrm{stall}}$')
        ax_aoa.set_ylabel('AoA [deg]', fontsize=8); ax_aoa.set_title('AoA Profile', fontsize=8.5, fontweight='bold')
        ax_aoa.grid(True, ls=':', alpha=0.6); ax_aoa.legend(fontsize=7, loc='upper right')

        # 4. Thrust Loading
        ax_t.plot(r_norm, bemt_hov.dt_dr, 'g-', lw=1.8, alpha=a_hov, label=f'Hover ({bemt_hov.thrust_N:.0f} N)')
        ax_t.plot(r_norm, bemt_cr.dt_dr, 'g--', lw=1.8, alpha=a_cr, label=f'Cruise ({bemt_cr.thrust_N:.0f} N)')
        ax_t.set_ylabel('dT [N/m]', fontsize=8); ax_t.set_title('Thrust Loading', fontsize=8.5, fontweight='bold')
        ax_t.grid(True, ls=':', alpha=0.6); ax_t.legend(fontsize=7, loc='upper right')

        # 5. Reynolds Distribution
        ax_re.plot(r_norm, bemt_hov.re_r / 1e5, 'k-', lw=1.8, alpha=a_hov, label=rf'Hover ($Re_{{tip}}={np.max(bemt_hov.re_r)/1e3:.0f}\mathrm{{k}}$)')
        ax_re.plot(r_norm, bemt_cr.re_r / 1e5, 'orange', ls='--', lw=1.8, alpha=a_cr, label=rf'Cruise ($Re_{{tip}}={np.max(bemt_cr.re_r)/1e3:.0f}\mathrm{{k}}$)')
        ax_re.set_ylabel(r'$Re \times 10^{5}$', fontsize=8); ax_re.set_title('Reynolds Distribution', fontsize=8.5, fontweight='bold')
        ax_re.grid(True, ls=':', alpha=0.6); ax_re.legend(fontsize=7, loc='upper left')

        # 6. Sectional Power Loading
        ax_pq.plot(r_norm, bemt_hov.dp_dr / 1000.0, 'r-', lw=1.8, alpha=a_hov, label=f'Hover ({bemt_hov.power_kW:.1f} kW)')
        ax_pq.plot(r_norm, bemt_cr.dp_dr / 1000.0, 'r--', lw=1.8, alpha=a_cr, label=f'Cruise ({bemt_cr.power_kW:.1f} kW)')
        ax_pq.set_ylabel('dP [kW/m]', color='r', fontsize=8)
        ax_pq.set_title('Sectional Power Loading', fontsize=8.5, fontweight='bold')
        ax_pq.grid(True, ls=':', alpha=0.6); ax_pq.legend(fontsize=7, loc='upper left')

        # 7. Section Coefficients (Cl, Cd)
        ax_coef.plot(r_norm, bemt_hov.cl, 'b-', lw=1.8, alpha=a_hov, label=r'Hover $C_l$')
        ax_coef.plot(r_norm, 10.0 * bemt_hov.cd, 'r-', lw=1.3, alpha=a_hov, label=r'Hover $10 \times C_d$')
        ax_coef.plot(r_norm, bemt_cr.cl, 'b--', lw=1.8, alpha=a_cr, label=r'Cruise $C_l$')
        ax_coef.plot(r_norm, 10.0 * bemt_cr.cd, 'r--', lw=1.3, alpha=a_cr, label=r'Cruise $10 \times C_d$')
        ax_coef.set_ylabel(r'$C_l$ and $C_d$', fontsize=8); ax_coef.set_title('Section Coefficients', fontsize=8.5, fontweight='bold')
        ax_coef.grid(True, ls=':', alpha=0.6); ax_coef.legend(fontsize=6.8, loc='upper left')

        # 8. Fast CT Sweeps
        th_sweep_hov = np.linspace(0.0, 25.0, 12)
        th_sweep_cr = np.linspace(20.0, 70.0, 12)
        res_sweep_hov = [run_bemt_solver(R, 0.45, Nb, c_0, taper, p, twist, rpm_hov, 0.0, af_model, rho_sl, a_sl, num_elements=10) for p in th_sweep_hov]
        res_sweep_cr = [run_bemt_solver(R, 0.45, Nb, c_0, taper, p, twist, rpm_cr, v_cr_ms, af_model, rho_sl, a_sl, num_elements=10) for p in th_sweep_cr]
        
        ct_sweep_hov = [r.CT for r in res_sweep_hov]
        ct_sweep_cr = [r.CT for r in res_sweep_cr]
        fm_sweep_hov = [r.FM for r in res_sweep_hov]
        eta_sweep_cr = [max(r.eta_prop, 0.0) for r in res_sweep_cr]

        ax_ct_th.plot(th_sweep_hov, ct_sweep_hov, 'b-o', ms=3.0, alpha=a_hov, label=r'Hover $C_T(\theta)$')
        ax_ct_th.plot(th_hov_trimmed, bemt_hov.CT, 'ro', ms=6.0, alpha=a_hov)
        ax_ct_th.plot(th_sweep_cr, ct_sweep_cr, 'm--s', ms=3.0, alpha=a_cr, label=r'Cruise $C_T(\theta)$')
        ax_ct_th.plot(th_cr_trimmed, bemt_cr.CT, 'mo', ms=6.0, alpha=a_cr)
        ax_ct_th.set_xlabel(r'$\theta_{0.75}$ [deg]', fontsize=8); ax_ct_th.set_ylabel(r'$C_T$', fontsize=8); ax_ct_th.set_title(r'$C_T$ vs Pitch Angle $\theta_{0.75}$', fontsize=8.5, fontweight='bold')
        ax_ct_th.grid(True, ls=':', alpha=0.6); ax_ct_th.legend(fontsize=6.8, loc='upper left')

        # 9. Inflow Distribution
        ax_inf.plot(r_norm, bemt_hov.lambda_tot, 'k-', lw=1.8, alpha=a_hov, label=r'Hover $\lambda_{\mathrm{tot}}$')
        ax_inf.plot(r_norm, bemt_hov.lambda_i, 'c:', lw=1.5, alpha=a_hov, label=r'Hover $\lambda_i$')
        ax_inf.plot(r_norm, bemt_cr.lambda_tot, 'k--', lw=1.8, alpha=a_cr, label=r'Cruise $\lambda_{\mathrm{tot}}$')
        ax_inf.plot(r_norm, bemt_cr.lambda_i, 'm:', lw=1.5, alpha=a_cr, label=r'Cruise $\lambda_i$')
        ax_inf.set_xlabel(r'$r/R$', fontsize=8); ax_inf.set_ylabel('Inflow ' + r'$\lambda$', fontsize=8); ax_inf.set_title('Inflow Distribution', fontsize=8.5, fontweight='bold')
        ax_inf.grid(True, ls=':', alpha=0.6); ax_inf.legend(fontsize=6.8, loc='upper right')

        # 10. Efficiency Polars
        ax_fm.plot(ct_sweep_hov, fm_sweep_hov, 'g-o', ms=3.0, alpha=a_hov, label='Hover FM Curve')
        ax_fm.plot(bemt_hov.CT, bemt_hov.FM, 'ro', ms=6.0, alpha=a_hov, label=f'FM={bemt_hov.FM:.3f}')
        ax_fm.plot(ct_sweep_cr, eta_sweep_cr, 'm--s', ms=3.0, alpha=a_cr, label=r'Cruise $\eta_{\mathrm{prop}}$')
        ax_fm.plot(bemt_cr.CT, bemt_cr.eta_prop, 'mo', ms=6.0, alpha=a_cr, label=rf'$\eta$={bemt_cr.eta_prop*100:.1f}%')
        ax_fm.set_xlabel(r'$C_T$', fontsize=8); ax_fm.set_ylabel('FM / Prop Efficiency', fontsize=8); ax_fm.set_title('Hover FM & Cruise Efficiency Polars', fontsize=8.5, fontweight='bold')
        ax_fm.grid(True, ls=':', alpha=0.6); ax_fm.legend(fontsize=6.8, loc='lower right')

        plt.tight_layout()
        plt.show()
        plt.close(fig_bemt)

    # ==========================================================================
    # TAB 3: STATISTICAL REGRESSIONS
    # ==========================================================================
    out_stats.clear_output(wait=True)
    with out_stats:
        fig_reg, axes_reg = plt.subplots(2, 2, figsize=(16.5, 9.6), dpi=100)
        x_w_grid = np.linspace(1800.0, 26000.0, 150)
        x_dl_grid = np.linspace(20.0, 220.0, 150)

        # 1. Empty Weight Fraction
        ax_r1 = axes_reg[0, 0]
        ax_r1.scatter(benchmarks['MTOW_kg'], benchmarks['We_W0'], color='crimson', s=65, edgecolors='k', zorder=4)
        for _, r in benchmarks.iterrows():
            ax_r1.annotate(str(r['Aircraft']), xy=(float(r['MTOW_kg'] * 1.04), float(r['We_W0'])), fontsize=8)
        ax_r1.plot(x_w_grid, power_law(x_w_grid, *popt_we), 'b-', lw=1.8, label=rf'Power Fit (${popt_we[0]:.2f} W_0^{{{popt_we[1]:.3f}}}$)')
        ax_r1.scatter(W0_cur, m_empty / W0_cur, color='lime', s=150, marker='*', edgecolors='k', zorder=5, label=f'Your Design ({m_empty/W0_cur:.3f})')
        ax_r1.set_xscale('log'); ax_r1.set_xlabel('Takeoff Gross Weight W0 [kg]', fontsize=9); ax_r1.set_ylabel('Empty Fraction We/W0', fontsize=9)
        ax_r1.set_title('1. Empty Weight Fraction Regression', fontsize=10.5, fontweight='bold'); ax_r1.grid(True, which='both', ls=':', alpha=0.6); ax_r1.legend(fontsize=8)

        # 2. Installed Power Loading
        ax_r2 = axes_reg[0, 1]
        ax_r2.scatter(benchmarks['Disc_Loading'], benchmarks['Installed_PW'], color='crimson', s=65, edgecolors='k', zorder=4)
        for _, r in benchmarks.iterrows():
            ax_r2.annotate(str(r['Aircraft']), xy=(float(r['Disc_Loading'] + 2.0), float(r['Installed_PW'])), fontsize=8)
        ax_r2.plot(x_dl_grid, power_law(x_dl_grid, *popt_pw), 'b-', lw=1.8, label=rf'Power Fit (${popt_pw[0]:.3f} DL^{{{popt_pw[1]:.3f}}}$)')
        ax_r2.scatter(disc_loading, cur_pw, color='lime', s=150, marker='*', edgecolors='k', zorder=5, label=f'Your Design ({cur_pw:.3f} kW/kg)')
        ax_r2.set_xlabel('Disc Loading DL [kg/m²]', fontsize=9); ax_r2.set_ylabel('Power Loading P/W0 [kW/kg]', fontsize=9)
        ax_r2.set_title('2. Installed Power Loading vs Disc Loading', fontsize=10.5, fontweight='bold'); ax_r2.grid(True, ls=':', alpha=0.6); ax_r2.legend(fontsize=8)

        # 3. Blade Solidity
        ax_r3 = axes_reg[1, 0]
        ax_r3.scatter(benchmarks['Disc_Loading'], benchmarks['Solidity'], color='crimson', s=65, edgecolors='k', zorder=4)
        for _, r in benchmarks.iterrows():
            ax_r3.annotate(str(r['Aircraft']), xy=(float(r['Disc_Loading'] + 2.0), float(r['Solidity'])), fontsize=8)
        ax_r3.plot(x_dl_grid, linear_law(x_dl_grid, *popt_sol), 'g-', lw=1.8, label='Linear Fit')
        ax_r3.scatter(disc_loading, solidity, color='lime', s=150, marker='*', edgecolors='k', zorder=5, label=f'Your Design ({solidity:.4f})')
        ax_r3.set_xlabel('Disc Loading DL [kg/m²]', fontsize=9); ax_r3.set_ylabel('Blade Solidity σ', fontsize=9)
        ax_r3.set_title('3. Blade Solidity vs Disc Loading', fontsize=10.5, fontweight='bold'); ax_r3.grid(True, ls=':', alpha=0.6); ax_r3.legend(fontsize=8)

        # 4. Service Ceiling Capability
        ax_r4 = axes_reg[1, 1]
        x_ceil_grid = np.linspace(3000.0, 9500.0, 150)
        ax_r4.scatter(benchmarks['Service_Ceiling_m'], benchmarks['Installed_PW'], color='crimson', s=65, edgecolors='k', zorder=4)
        for _, r in benchmarks.iterrows():
            ax_r4.annotate(str(r['Aircraft']), xy=(float(r['Service_Ceiling_m'] + 80.0), float(r['Installed_PW'])), fontsize=8)
        ax_r4.plot(x_ceil_grid, linear_law(x_ceil_grid, *popt_ceil), 'm-', lw=1.8, label='Ceiling Fit')
        ax_r4.scatter(Ceiling_m, cur_pw, color='lime', s=150, marker='*', edgecolors='k', zorder=5, label=f'Your Design ({cur_pw:.3f} kW/kg)')
        ax_r4.set_xlabel('Service Ceiling [m]', fontsize=9); ax_r4.set_ylabel('Power Loading P/W0 [kW/kg]', fontsize=9)
        ax_r4.set_title('4. Service Ceiling Capability', fontsize=10.5, fontweight='bold'); ax_r4.grid(True, ls=':', alpha=0.6); ax_r4.legend(fontsize=8)

        plt.tight_layout()
        plt.show()
        plt.close(fig_reg)

    # ==========================================================================
    # SECTION 6.1: HOVER PERFORMANCE MAPS (SLIDE 26)
    # ==========================================================================
    out_hover_maps.clear_output(wait=True)
    with out_hover_maps:
        fig_h_maps, axs_hm = plt.subplots(2, 2, figsize=(15.5, 9.2), dpi=100)
        
        # 1. Sweep collective pitch from 0 to 22 deg at Hover RPM
        th_sweep = np.linspace(0.0, 22.0, 22)
        t_sl_arr, p_sl_arr, q_sl_arr, aoa_root_sl, aoa_75_sl = [], [], [], [], []
        t_ceil_arr, p_ceil_arr, q_ceil_arr, aoa_root_ceil, aoa_75_ceil = [], [], [], [], []
        
        for th_val in th_sweep:
            # Sea Level
            bemt_s = run_bemt_solver(R, 0.45, Nb, c_0, taper, th_val, twist, rpm_hov, 0.0, af_model, rho=rho_sl, a_sound=a_sl, num_elements=18)
            t_sl_arr.append(bemt_s.thrust_N * 2.0) # 2 rotors
            p_sl_arr.append((bemt_s.power_kW * 2.0) / 0.94) # total shaft kW
            q_sl_arr.append(bemt_s.torque_Nm * 2.0)
            aoa_root_sl.append(bemt_s.alpha_deg[0])
            aoa_75_sl.append(bemt_s.alpha_deg[int(len(bemt_s.alpha_deg)*0.75)])
            
            # Ceiling
            bemt_c = run_bemt_solver(R, 0.45, Nb, c_0, taper, th_val, twist, rpm_hov, 0.0, af_model, rho=rho_ceil, a_sound=a_ceil, num_elements=18)
            t_ceil_arr.append(bemt_c.thrust_N * 2.0)
            p_ceil_arr.append((bemt_c.power_kW * 2.0) / 0.94)
            q_ceil_arr.append(bemt_c.torque_Nm * 2.0)
            aoa_root_ceil.append(bemt_c.alpha_deg[0])
            aoa_75_ceil.append(bemt_c.alpha_deg[int(len(bemt_c.alpha_deg)*0.75)])
            
        t_sl_arr, p_sl_arr, q_sl_arr = np.array(t_sl_arr), np.array(p_sl_arr), np.array(q_sl_arr)
        t_ceil_arr, p_ceil_arr, q_ceil_arr = np.array(t_ceil_arr), np.array(p_ceil_arr), np.array(q_ceil_arr)
        w_hover_target = W0_cur * 9.80665 * 1.08 # with 8% download

        # [0, 0] Thrust vs Collective
        ax_h1 = axs_hm[0, 0]
        ax_h1.plot(th_sweep, t_sl_arr / 1000.0, "b-", lw=2.2, label="Sea Level (ISA 0m)")
        ax_h1.plot(th_sweep, t_ceil_arr / 1000.0, "b--", lw=1.8, label=f"Hover Ceiling ({Ceiling_m:.0f}m)")
        ax_h1.axhline(w_hover_target / 1000.0, color="crimson", ls=":", lw=1.8, label=f"Target Thrust ({w_hover_target/1000.0:.1f} kN)")
        ax_h1.scatter([th_hov_trimmed], [w_hover_target/1000.0], color="lime", s=140, marker="*", edgecolors="k", zorder=5, label=f"Trim θ_0.75 = {th_hov_trimmed:.1f}°")
        ax_h1.set_xlabel("Collective Pitch θ_0.75 [deg]", fontsize=9)
        ax_h1.set_ylabel("Total Proprotor Thrust [kN]", fontsize=9)
        ax_h1.set_title("1. Thrust vs. Collective Pitch", fontsize=10.5, fontweight="bold")
        ax_h1.grid(True, ls=":", alpha=0.6); ax_h1.legend(fontsize=7.8)

        # [0, 1] AoA & Stall Margin vs Collective
        ax_h2 = axs_hm[0, 1]
        ax_h2.plot(th_sweep, aoa_root_sl, "r-", lw=2.0, label="Root AoA α_root (SL)")
        ax_h2.plot(th_sweep, aoa_75_sl, "m-", lw=1.8, label="75% AoA α_0.75 (SL)")
        ax_h2.plot(th_sweep, aoa_root_ceil, "r--", lw=1.5, label="Root AoA α_root (Ceiling)")
        ax_h2.axhline(12.0, color="crimson", ls="--", lw=1.5, label="Static Stall Limit (α = 12°)")
        ax_h2.axhspan(12.0, 25.0, color="red", alpha=0.12, label="Stall-Limited Region")
        ax_h2.set_xlabel("Collective Pitch θ_0.75 [deg]", fontsize=9)
        ax_h2.set_ylabel("Sectional Angle of Attack [deg]", fontsize=9)
        ax_h2.set_title("2. Blade AoA & Stall Margin vs. Collective", fontsize=10.5, fontweight="bold")
        ax_h2.grid(True, ls=":", alpha=0.6); ax_h2.legend(fontsize=7.8)

        # [1, 0] Torque & Power vs Collective
        ax_h3 = axs_hm[1, 0]
        ax_h3_tw = ax_h3.twinx()
        l_p1, = ax_h3.plot(th_sweep, p_sl_arr, "r-", lw=2.2, label="Shaft Power (SL)")
        l_p2, = ax_h3.plot(th_sweep, p_ceil_arr, "r--", lw=1.8, label="Shaft Power (Ceiling)")
        l_pmax = ax_h3.axhline(P_inst_kW, color="black", ls=":", lw=1.8, label=f"Installed ({P_inst_kW:.0f} kW)")
        l_pmax_c = ax_h3.axhline(P_avail_ceil_kW, color="gray", ls=":", lw=1.5, label=f"Avail at Ceiling ({P_avail_ceil_kW:.0f} kW)")
        ax_h3.axhspan(P_inst_kW, max(P_inst_kW*1.3, float(np.max(p_sl_arr))), color="red", alpha=0.10)
        l_q, = ax_h3_tw.plot(th_sweep, q_sl_arr / 1000.0, "g-.", lw=1.5, label="Total Torque (SL) [kN·m]")
        ax_h3.set_xlabel("Collective Pitch θ_0.75 [deg]", fontsize=9)
        ax_h3.set_ylabel("Required Shaft Power [kW]", color="r", fontsize=9)
        ax_h3_tw.set_ylabel("Rotor Torque [kN·m]", color="g", fontsize=9)
        ax_h3.set_title("3. Torque & Power vs. Collective", fontsize=10.5, fontweight="bold")
        ax_h3.grid(True, ls=":", alpha=0.6)
        ax_h3.legend([l_p1, l_p2, l_pmax, l_pmax_c, l_q], [l_p1.get_label(), l_p2.get_label(), l_pmax.get_label(), l_pmax_c.get_label(), l_q.get_label()], fontsize=7.2, loc="upper left")

        # [1, 1] Operating Envelope & Hover Ceiling vs Gross Weight
        ax_h4 = axs_hm[1, 1]
        gw_sweep = np.linspace(1500.0, 24000.0, 40)
        ceil_pwr_lim, ceil_stall_lim = [], []
        
        for gw in gw_sweep:
            t_req = gw * 9.80665 * 1.08
            p_hov_sl = ((t_req ** 1.5) / np.sqrt(2.0 * 1.225 * A_disc_total)) / (0.74 * 0.94 * 1000.0)
            sigma_lim = (p_hov_sl / max(1.0, P_inst_kW)) ** (1.0 / 1.55)
            if sigma_lim < 1.0:
                T_ratio = sigma_lim ** (1.0 / 4.2561)
                h_pwr = max(0.0, (1.0 - T_ratio) * 288.15 / 0.0065)
            else:
                h_pwr = 0.0
            ceil_pwr_lim.append(min(9000.0, h_pwr))
            
            solidity = (Nb * c_0) / (np.pi * R)
            ct_max = 0.135 * solidity
            vtip = (rpm_hov * 2.0 * np.pi / 60.0) * R
            rho_min = t_req / max(1.0, ct_max * A_disc_total * (vtip ** 2))
            sigma_stall = rho_min / 1.225
            if sigma_stall < 1.0:
                T_ratio_s = sigma_stall ** (1.0 / 4.2561)
                h_stall = max(0.0, (1.0 - T_ratio_s) * 288.15 / 0.0065)
            else:
                h_stall = 0.0
            ceil_stall_lim.append(min(9000.0, h_stall))
            
        ax_h4.plot(gw_sweep / 1000.0, ceil_pwr_lim, "r-", lw=2.0, label="Engine Power-Limited Boundary")
        ax_h4.plot(gw_sweep / 1000.0, ceil_stall_lim, "m--", lw=1.8, label="Blade Stall-Limited Boundary")
        ax_h4.fill_between(gw_sweep / 1000.0, np.minimum(ceil_pwr_lim, ceil_stall_lim), color="lightgreen", alpha=0.25, label="Feasible Hover Region")
        ax_h4.scatter([W0_cur/1000.0], [Ceiling_m], color="lime", s=150, marker="*", edgecolors="k", zorder=5, label=f"Design Point ({W0_cur/1000.0:.1f}t @ {Ceiling_m:.0f}m)")
        ax_h4.set_xlabel("Takeoff Gross Weight [Metric Tons]", fontsize=9)
        ax_h4.set_ylabel("Hover Ceiling Altitude [m]", fontsize=9)
        ax_h4.set_title("4. Hover Operating Envelope & Ceiling Limits", fontsize=10.5, fontweight="bold")
        ax_h4.grid(True, ls=":", alpha=0.6); ax_h4.legend(fontsize=7.8)

        plt.tight_layout()
        plt.show()
        plt.close(fig_h_maps)

    # ==========================================================================
    # SECTION 6.2: AXIAL FORWARD-FLIGHT / PROPELLER MAPS (SLIDE 27)
    # ==========================================================================
    out_cruise_maps.clear_output(wait=True)
    with out_cruise_maps:
        fig_cr_maps, axs_cm = plt.subplots(2, 2, figsize=(15.5, 9.2), dpi=100)
        
        j_grid = np.linspace(0.5, 4.5, 25)
        thetas_cr_family = [30.0, 35.0, 40.0, 45.0, 50.0, 55.0]
        colors_cr = plt.cm.viridis(np.linspace(0.1, 0.9, len(thetas_cr_family)))
        
        ax_c1 = axs_cm[0, 0]
        ax_c1_tw = ax_c1.twinx()
        ax_c2 = axs_cm[0, 1]
        
        n_rps = rpm_cr / 60.0
        D_prop = 2.0 * R
        eta_envelope = np.zeros_like(j_grid)
        
        for idx_th, th_p in enumerate(thetas_cr_family):
            ct_j_list, cp_j_list, eta_j_list = [], [], []
            for j_val in j_grid:
                v_axial_j = j_val * n_rps * D_prop
                bemt_j = run_bemt_solver(R, 0.45, Nb, c_0, taper, th_p, twist, rpm_cr, v_axial_j, af_model, rho=rho_sl, a_sound=a_sl, num_elements=18)
                ct_j = bemt_j.CT
                cp_j = max(bemt_j.CP, 1e-6)
                eta_j = (ct_j * j_val) / cp_j if ct_j > 0 else 0.0
                ct_j_list.append(ct_j)
                cp_j_list.append(cp_j)
                eta_j_list.append(np.clip(eta_j, 0.0, 0.92))
                
            ct_j_list = np.array(ct_j_list)
            cp_j_list = np.array(cp_j_list)
            eta_j_list = np.array(eta_j_list)
            eta_envelope = np.maximum(eta_envelope, eta_j_list)
            
            ax_c1.plot(j_grid, ct_j_list, color=colors_cr[idx_th], lw=1.6, label=f"θ={th_p:.0f}°")
            ax_c1_tw.plot(j_grid, cp_j_list, color=colors_cr[idx_th], ls="--", lw=1.3)
            ax_c2.plot(j_grid, eta_j_list, color=colors_cr[idx_th], lw=1.6, label=rf"$	heta_{{0.75}}={th_p:.0f}^\circ$")
            
        ax_c1.axhline(0, color="k", ls=":", lw=1.0)
        ax_c1.set_xlabel("Advance Ratio J = V_inf / (n D)", fontsize=9)
        ax_c1.set_ylabel("Thrust Coefficient CT (Solid)", fontsize=9)
        ax_c1_tw.set_ylabel("Power Coefficient CP (Dashed)", color="gray", fontsize=9)
        ax_c1.set_title("1. Propeller Performance: $C_T$ and $C_P$ vs. $J$", fontsize=10.5, fontweight="bold")
        ax_c1.grid(True, ls=":", alpha=0.6); ax_c1.legend(fontsize=7.2, loc="upper right")
        
        # Design cruise point
        J_cr = v_cr_ms / (n_rps * D_prop)
        ax_c2.plot(j_grid, eta_envelope, "k-", lw=2.2, label="Max Efficiency Envelope")
        ax_c2.scatter([J_cr], [eta_prop], color="lime", s=160, marker="*", edgecolors="k", zorder=5, label=f"Design Cruise Point (J={J_cr:.2f}, η={eta_prop*100:.1f}%)")
        ax_c2.set_xlabel("Advance Ratio J = V_inf / (n D)", fontsize=9)
        ax_c2.set_ylabel("Propulsive Efficiency η_p = CT*J / CP", fontsize=9)
        ax_c2.set_title("2. Propulsive Efficiency Map vs. Advance Ratio", fontsize=10.5, fontweight="bold")
        ax_c2.set_ylim(0.0, 1.0)
        ax_c2.grid(True, ls=":", alpha=0.6); ax_c2.legend(fontsize=7.5, loc="lower right")
        
        # [1, 0] Spanwise AoA Distribution across speeds
        ax_c3 = axs_cm[1, 0]
        r_norm_pts = bemt_cr.r_stations / R
        test_j_speeds = [1.5, 2.5, 3.5, 4.2]
        colors_j = ["blue", "green", "orange", "crimson"]
        for j_t, col_j in zip(test_j_speeds, colors_j):
            v_ax_t = j_t * n_rps * D_prop
            bemt_t = run_bemt_solver(R, 0.45, Nb, c_0, taper, th_cr_trimmed, twist, rpm_cr, v_ax_t, af_model, rho=rho_sl, a_sound=a_sl, num_elements=len(r_norm_pts))
            ax_c3.plot(r_norm_pts, bemt_t.alpha_deg, color=col_j, lw=1.8, label=f"J={j_t:.1f} ({v_ax_t*3.6:.0f} km/h)")
        ax_c3.axhline(0, color="k", ls=":", lw=1.2)
        ax_c3.axhline(12.0, color="red", ls="--", lw=1.2, label="Stall Limit (12°)")
        ax_c3.axhspan(-20.0, 0.0, color="orange", alpha=0.10, label="Windmilling / Drag Region")
        ax_c3.set_xlabel("Radial Station r/R", fontsize=9)
        ax_c3.set_ylabel("Sectional AoA [deg]", fontsize=9)
        ax_c3.set_title("3. Spanwise AoA Distribution across Advance Ratios", fontsize=10.5, fontweight="bold")
        ax_c3.grid(True, ls=":", alpha=0.6); ax_c3.legend(fontsize=7.5, loc="upper right")
        
        # [1, 1] Feasibility Card
        ax_c4 = axs_cm[1, 1]
        ax_c4.axis("off")
        
        p_req_450_kw = (total_drag_cruise * v_cr_ms) / (max(0.01, eta_prop) * 1000.0)
        tip_mach_cr = np.sqrt(((rpm_cr*2*np.pi/60.0)*R)**2 + v_cr_ms**2) / a_sl
        cr_status = "FEASIBLE & VERIFIED" if (p_req_450_kw <= P_inst_kW and tip_mach_cr <= 0.82 and eta_prop >= 0.70) else "CONSTRAINTS EXCEEDED"
        
        card_cruise_str = """PROPELLER CRUISE PERFORMANCE AUDIT
===============================================
STATUS: {}
-----------------------------------------------
DESIGN OPERATING CRUISE CONDITION:
  • True Airspeed V_inf:     {:6.1f} km/h ({:.1f} m/s)
  • Advance Ratio J:         {:6.2f}
  • Cruise Rotational Speed: {:6.0f} RPM
  • Helical Tip Mach M_tip:  {:6.3f} (Limit: <= 0.80)
  • Trimmed Cruise Pitch:    {:6.1f} deg

PROPULSIVE POWER & THRUST MATCHING:
  • Total Airplane Drag:     {:6.2f} kN
  • Rotor Thrust Delivered:  {:6.2f} kN
  • Propulsive Efficiency η: {:6.1f} %
  • Cruise Shaft Power Req:  {:6.0f} kW
  • Installed Engine Rating: {:6.0f} kW
  • Power Margin at 450 km/h: +{:6.0f} kW
""".format(cr_status, v_cr_kmh, v_cr_ms, J_cr, rpm_cr, tip_mach_cr, th_cr_trimmed, total_drag_cruise/1000.0, (bemt_cr.thrust_N*2)/1000.0, eta_prop*100, p_req_450_kw, P_inst_kW, P_inst_kW - p_req_450_kw)
        border_c = "#28a745" if cr_status == "FEASIBLE & VERIFIED" else "#dc3545"
        ax_c4.text(0.02, 0.98, card_cruise_str, fontfamily="monospace", fontsize=9.0, verticalalignment="top",
                   bbox=dict(boxstyle="round,pad=0.6", facecolor="#f8f9fa", edgecolor=border_c, lw=1.5))
                   
        plt.tight_layout()
        plt.show()
        plt.close(fig_cr_maps)

    # ==========================================================================
    # SECTION 6.3: COMPARISON WITH COMPARABLE ROTORS (SLIDE 28)
    # ==========================================================================
    out_comparisons.clear_output(wait=True)
    with out_comparisons:
        fig_comp = plt.figure(figsize=(15.5, 9.2), dpi=100)
        gs_comp = fig_comp.add_gridspec(2, 2, height_ratios=[1.0, 1.0], hspace=0.35, wspace=0.25)
        
        # 1. Multi-Aircraft Comparative Benchmark Table
        ax_ctbl = fig_comp.add_subplot(gs_comp[0, :])
        ax_ctbl.axis("off")
        
        comp_df = pd.DataFrame([
            {"Aircraft": "Bell XV-15 (NASA/Army)", "R [m]": 3.81, "Nb": 3, "Solidity": 0.089, "Twist [°]": -38.0, "DL [kg/m²]": 73.2, "Vtip_hov [m/s]": 225.0, "Vtip_cr [m/s]": 170.0, "Max FM": 0.74, "Cruise η": 0.82, "P/W0 [kW/kg]": 0.385},
            {"Aircraft": "Bell Boeing V-22 Osprey", "R [m]": 5.80, "Nb": 3, "Solidity": 0.105, "Twist [°]": -47.0, "DL [kg/m²]": 102.5, "Vtip_hov [m/s]": 240.0, "Vtip_cr [m/s]": 180.0, "Max FM": 0.72, "Cruise η": 0.79, "P/W0 [kW/kg]": 0.380},
            {"Aircraft": "Leonardo AW609 (Civil)", "R [m]": 4.05, "Nb": 3, "Solidity": 0.096, "Twist [°]": -35.0, "DL [kg/m²]": 88.4, "Vtip_hov [m/s]": 228.0, "Vtip_cr [m/s]": 175.0, "Max FM": 0.75, "Cruise η": 0.84, "P/W0 [kW/kg]": 0.354},
            {"Aircraft": "Bell V-280 Valor (FVL)", "R [m]": 5.33, "Nb": 4, "Solidity": 0.100, "Twist [°]": -32.0, "DL [kg/m²]": 95.0, "Vtip_hov [m/s]": 235.0, "Vtip_cr [m/s]": 165.0, "Max FM": 0.76, "Cruise η": 0.85, "P/W0 [kW/kg]": 0.533},
            {"Aircraft": ">>> YOUR DESIGNED TILTROTOR", "R [m]": round(R, 2), "Nb": Nb, "Solidity": round(solidity, 3), "Twist [°]": round(twist, 1), "DL [kg/m²]": round(disc_loading, 1), "Vtip_hov [m/s]": round(v_tip_hov, 1), "Vtip_cr [m/s]": round(v_tip_cr, 1), "Max FM": round(bemt_hov.FM, 2), "Cruise η": round(eta_prop, 2), "P/W0 [kW/kg]": round(cur_pw, 3)}
        ])
        
        table_data = [comp_df.columns.tolist()] + comp_df.values.tolist()
        t_elem = ax_ctbl.table(cellText=table_data, loc="center", cellLoc="center")
        t_elem.auto_set_font_size(False)
        t_elem.set_fontsize(8.5)
        t_elem.scale(1.0, 1.5)
        
        for col_i in range(len(comp_df.columns)):
            t_elem[(0, col_i)].set_facecolor("#1f77b4")
            t_elem[(0, col_i)].set_text_props(color="white", fontweight="bold")
            t_elem[(5, col_i)].set_facecolor("#d4edda")
            t_elem[(5, col_i)].set_text_props(fontweight="bold")
            
        ax_ctbl.set_title("COMPARISON WITH COMPARABLE PROPROTOR BENCHMARKS (NORMALIZED NONDIMENSIONAL METRICS)", fontsize=11, fontweight="bold", pad=12)
        
        # 2. Nondimensional Metrics Grouped Bar Comparison
        ax_cbar = fig_comp.add_subplot(gs_comp[1, 0])
        labels_m = ["Solidity (σ)", "Disc Load (DL/80)", "FM (Hover)", "Prop Eff (η_cr)", "P/W (kW/kg)"]
        xv15_vals = [0.089, 73.2/80.0, 0.74, 0.82, 0.385]
        aw609_vals = [0.096, 88.4/80.0, 0.75, 0.84, 0.354]
        your_vals = [solidity, disc_loading/80.0, bemt_hov.FM, eta_prop, cur_pw]
        
        x_idx = np.arange(len(labels_m))
        bar_w = 0.25
        ax_cbar.bar(x_idx - bar_w, xv15_vals, width=bar_w, color="#4575b4", label="Bell XV-15")
        ax_cbar.bar(x_idx, aw609_vals, width=bar_w, color="#f46d43", label="Leonardo AW609")
        ax_cbar.bar(x_idx + bar_w, your_vals, width=bar_w, color="#2ca02c", label="Your Aircraft")
        ax_cbar.set_xticks(x_idx); ax_cbar.set_xticklabels(labels_m, fontsize=8)
        ax_cbar.set_ylabel("Normalized Metric Value", fontsize=8.5)
        ax_cbar.set_title("Normalized Aerodynamic & Sizing Comparisons", fontsize=10, fontweight="bold")
        ax_cbar.grid(True, ls=":", alpha=0.6); ax_cbar.legend(fontsize=7.8)
        
        # 3. Technical Discussion Card
        ax_ctext = fig_comp.add_subplot(gs_comp[1, 1])
        ax_ctext.axis("off")
        
        accept_verdict = "ACCEPTABLE & BALANCED" if (0.07 <= solidity <= 0.12 and 50 <= disc_loading <= 120 and bemt_hov.FM >= 0.65 and eta_prop >= 0.72) else "NEEDS GEOMETRIC TUNING"
        
        disc_text = """DESIGN ACCEPTABILITY & COMPARATIVE EVALUATION
=================================================
OVERALL DESIGN VERDICT: {}
-------------------------------------------------
1. DISC LOADING (DL = {:.1f} kg/m²):
   • Sits squarely between XV-15 (73.2) and AW609 (88.4).
   • Compact proprotor diameter prevents fuselage collision
     while providing reasonable hover downwash velocity.

2. ROTOR SOLIDITY (σ = {:.3f}):
   • {}-bladed proprotor provides optimal blade loading
     (Ct/σ ≈ 0.08) avoiding root stall in hover.

3. TWIST RATE (θ_tw = {:.1f}°):
   • Provides attached high-speed cruise thrust while
     mitigating extreme negative root windmilling.

4. TIP SPEED SCALING:
   • Hover Vtip ({:.0f} m/s) -> Cruise Vtip ({:.0f} m/s)
   • ~{:.0f}% RPM reduction keeps cruise Mtip <= 0.80.
""".format(accept_verdict, disc_loading, solidity, Nb, twist, v_tip_hov, v_tip_cr, (1 - v_tip_cr/v_tip_hov)*100)
        border_d = "#28a745" if accept_verdict == "ACCEPTABLE & BALANCED" else "#ffc107"
        ax_ctext.text(0.02, 0.98, disc_text, fontfamily="monospace", fontsize=8.8, verticalalignment="top",
                      bbox=dict(boxstyle="round,pad=0.6", facecolor="#f8f9fa", edgecolor=border_d, lw=1.5))
                      
        fig_comp.subplots_adjust(top=0.95, bottom=0.07, left=0.06, right=0.97, hspace=0.38, wspace=0.27)
        plt.show()
        plt.close(fig_comp)
        
for w in [w_airfoil, w_radius, w_nblades, w_chord, w_taper, w_twist,
          w_rpm_hov, w_rpm_cr, w_s_wing, w_ar, w_sweep, w_tc, w_v_cruise, w_range, w_ceiling, w_p_inst, w_sfc, w_bemt_mode]:
    w.observe(update_designer_dashboard, names='value')

def header_lbl(text):
    return widgets.HTML(f'<b style="font-size:12px; margin:2px 0px 1px 0px; display:inline-block;">{text}</b>')

designer_sidebar = widgets.VBox(
    [
        header_lbl('3D/BEMT Diagnostics Display Mode'),
        w_bemt_mode,
        header_lbl('Mission & Flight Envelope'),
        w_range,
        w_v_cruise,
        w_ceiling,
        header_lbl('Proprotor Blade Geometry'),
        w_airfoil,
        w_radius,
        w_nblades,
        w_chord,
        w_taper,
        w_twist,
        header_lbl('Dual-Speed RPM Schedules'),
        w_rpm_hov,
        w_rpm_cr,
        header_lbl('Wing Aerodynamics & Structural Sizing'),
        w_s_wing,
        w_ar,
        w_sweep,
        w_tc,
        header_lbl('Installed Powerplant & Fuel Burn'),
        w_p_inst,
        w_sfc,
    ],
    layout=widgets.Layout(
        padding='6px 8px',
        border='1px solid #ced4da',
        border_radius='6px',
        width='365px',
        min_width='365px',
        flex='0 0 365px'
    ),
)

display(
    widgets.HBox(
        [designer_sidebar, tabs_designer],
        layout=widgets.Layout(
            width='100%',
            align_items='flex-start',
            overflow='auto'
        ),
    )
)
update_designer_dashboard()

In [ ]:
# =============================================================================
# CELL 3: TWO-STAGE COARSE-TO-FINE DETERMINISTIC GRID OPTIMIZER
# (AUTO-TRIMMED HOVER & CRUISE EQUILIBRIUM CO-OPTIMIZATION)
# =============================================================================
import itertools
import time
import numpy as np
import pandas as pd
from IPython.display import display, HTML

# ------------------------------------------------------------------------------
# 1. RETRIEVE ACTIVE MISSION & SIZING PARAMETERS FROM CELL 2
# ------------------------------------------------------------------------------
req_range_km = float(w_range.value) if 'w_range' in globals() else 1000.0
req_v_cr_kmh = float(w_v_cruise.value) if 'w_v_cruise' in globals() else 450.0
req_ceiling_m = float(w_ceiling.value) if 'w_ceiling' in globals() else 7000.0
req_p_inst_kw = float(w_p_inst.value) if 'w_p_inst' in globals() else 5000.0
req_sfc = float(w_sfc.value) if 'w_sfc' in globals() else 0.285
req_s_wing = float(w_s_wing.value) if 'w_s_wing' in globals() else 26.0
req_ar = float(w_ar.value) if 'w_ar' in globals() else 7.8
req_tc = float(w_tc.value) if 'w_tc' in globals() else 22.5

v_cr_ms = req_v_cr_kmh / 3.6
range_m = req_range_km * 1000.0
sfc_si = req_sfc / 3.6e6
g = 9.80665
rho_sl, a_sl, _, mu_sl = isa_atmosphere(0.0)

b_wing = np.sqrt(req_s_wing * req_ar)
c_root_w = (2.0 * req_s_wing) / (b_wing * (1.0 + 0.65))
mac = (2.0 / 3.0) * c_root_w * (1.0 + 0.65 + 0.65**2) / (1.0 + 0.65)
s_htail = (0.85 * req_s_wing * mac) / (0.45 * FUSE_LEN)
s_wet_total = S_WET_FUSE + 2.05 * req_s_wing + 2.0 * s_htail
cd0_dyn = 0.0030 * (s_wet_total / req_s_wing)

candidate_airfoils = [
    # Baseline Proprotor / Transonic
    "Boeing-Vertol VR-12",
    # NASA Supercritical
    "NASA SC(2)-0010", "NASA SC(2)-0012", "NASA SC(2)-0410", "NASA SC(2)-0412",
    # Classic NACA Workhorses
    "NACA 0009", "NACA 0012", "NACA 23012", "NACA 23015",
    "NACA 64-A010", "NACA 64-A012",
    # Modern Rotorcraft
    "ONERA OA209", "ONERA OA212", "Sikorsky SC1095"
]

blade_counts = [3, 4, 5, 6]

for af_name in candidate_airfoils:
    if af_name not in _airfoil_cache:
        try:
            _airfoil_cache[af_name] = AirfoilModel(airfoil_name=af_name, ncrit=9)
        except Exception as e:
            print(f"Skipping {af_name}: {e}")

# ------------------------------------------------------------------------------
# 2. CONTINUOUS EVALUATOR (AUTO-TRIMMED 6-PARAMETER VECTOR)
# Vector: [R, c_0, taper, twist, rpm_hov, rpm_cr]
# ------------------------------------------------------------------------------
def evaluate_continuous_vector(x, Nb, af_name):
    R, c_0, taper, twist, rpm_hov, rpm_cr = x
    af_model = _airfoil_cache[af_name]

    # 1. Fuselage Clearance Constraint (>= 0.25 m)
    fuse_clearance = (0.5 * b_wing - R) - (0.5 * FUSE_W)
    if fuse_clearance < 0.25:
        return -1e6 - (0.25 - fuse_clearance) * 1e4, None

    # 2. Helical Tip Mach Constraints
    v_tip_hov = (rpm_hov * (np.pi / 30.0)) * R
    v_tip_cr = (rpm_cr * (np.pi / 30.0)) * R
    m_tip_hov = v_tip_hov / a_sl
    m_tip_cr_helical = np.sqrt(v_tip_cr**2 + v_cr_ms**2) / a_sl
    if m_tip_hov > 0.78 or m_tip_cr_helical > 0.83:
        return -1e6, None

    # 3. Solidity Check (0.055 <= sigma <= 0.17)
    solidity = (Nb * c_0 * (1.0 + taper) * 0.5) / (np.pi * R)
    if solidity < 0.055 or solidity > 0.17:
        return -1e6, None

    # 4. Aircraft Sizing Convergence Loop (Baseline Fixed L/D = 12.0)
    eta_prop_est = 0.75
    W0_cur = 9000.0
    ld_cr = 12.0
    converged = False
    
    for _ in range(25):
        m_f = np.exp(-(range_m * sfc_si * g) / (eta_prop_est * ld_cr))
        ff = 1.06 * (1.0 - (0.985 * 0.990 * 0.975 * m_f * 0.990 * 0.985))

        we_base = power_law(W0_cur, *popt_we)
        w_blade_penalty = 0.035 * (4.20 / R) ** (-0.5)
        we_total = we_base + w_blade_penalty - 0.035
        denom = 1.0 - ff - we_total
        if denom <= 0.05:
            break
        W_next = PAYLOAD_FIXED / denom
        if abs(W_next - W0_cur) / W0_cur < 1e-4:
            W0_cur = W_next
            converged = True
            break
        W0_cur = W_next

    if not converged or W0_cur > 25000.0:
        return -1e6, None

    # 5. Required Vehicle Thrust Targets
    thrust_hover_target = (W0_cur * g * 1.08) / N_ROTORS
    thrust_cruise_target = ((W0_cur * g) / ld_cr) / N_ROTORS

    # 6. Auto-Trimmed BEMT Solves
    th_hov, res_h = trim_rotor_collective(
        thrust_hover_target, R, 0.45, Nb, c_0, taper, twist, rpm_hov, 0.0,
        af_model, rho_sl, a_sl, mu_sl, pitch_bounds=(-2.0, 24.0)
    )
    th_cr, res_c = trim_rotor_collective(
        thrust_cruise_target, R, 0.45, Nb, c_0, taper, twist, rpm_cr, v_cr_ms,
        af_model, rho_sl, a_sl, mu_sl, pitch_bounds=(30.0, 75.0)
    )

    max_alpha_h = np.max(np.abs(res_h.alpha_deg))
    max_alpha_c = np.max(np.abs(res_c.alpha_deg))
    min_alpha_c = np.min(res_c.alpha_deg)
    stall_margin = 12.0 - max(max_alpha_h, max_alpha_c)

    # Inboard negative-thrust / windmilling check
    if stall_margin <= 0.0 or min_alpha_c < -1.5 or res_h.thrust_N <= 0.0 or res_c.thrust_N <= 0.0:
        return -1e6, None

    # 7. Installed Power Check
    p_hov_req = (2.0 * res_h.power_kW) / 0.95
    p_cr_req = (2.0 * res_c.power_kW) / 0.95

    if (req_p_inst_kw - p_hov_req) < 0.0 or (req_p_inst_kw - p_cr_req) < 0.0:
        return -1e6, None

    # Aeromechanical Fitness Function
    fitness = (
        (res_c.eta_prop * 35.0) +
        (res_h.FM * 25.0) +
        (stall_margin * 2.0) -
        (W0_cur / 3000.0)
    )

    disc_loading = W0_cur / (N_ROTORS * np.pi * (R**2))
    diag = {
        "Airfoil": af_name,
        "Radius R [m]": round(float(R), 3),
        "Diameter [m]": round(float(2 * R), 3),
        "Nb": int(Nb),
        "Root Chord c0 [m]": round(float(c_0), 3),
        "Tip Chord [m]": round(float(c_0 * taper), 3),
        "Taper": round(float(taper), 3),
        "Twist [°]": round(float(twist), 2),
        "Hover Pitch [°]": round(float(th_hov), 2),
        "Cruise Pitch [°]": round(float(th_cr), 2),
        "Hover RPM": int(round(rpm_hov)),
        "Cruise RPM": int(round(rpm_cr)),
        "Hover FM": round(float(res_h.FM), 3),
        "Cruise η [%]": round(float(res_c.eta_prop * 100.0), 1),
        "Max α_hov [°]": round(float(max_alpha_h), 1),
        "Max α_cr [°]": round(float(max_alpha_c), 1),
        "Heatmap Stall Margin [°]": round(float(stall_margin), 2),
        "MTOW [kg]": round(float(W0_cur), 1),
        "Solidity σ": round(float(solidity), 4),
        "Disc Loading [kg/m²]": round(float(disc_loading), 1),
        "Hover Req Pwr [kW]": round(float(p_hov_req), 1),
        "Cruise Req Pwr [kW]": round(float(p_cr_req), 1),
        "Fuselage Clearance [m]": round(float(fuse_clearance), 2),
        "Fitness": round(float(fitness), 2)
    }
    return fitness, diag

# ------------------------------------------------------------------------------
# 3. STAGE 1: COARSE FACTORIAL SWEEP ACROSS TOPOLOGICAL CASES
# ------------------------------------------------------------------------------
t_start_grid = time.time()
print("Starting Two-Stage Coarse-to-Fine Grid Search...")

coarse_grid_1d = {
    'R': np.linspace(3.5, 5.2, 3),
    'c_0': np.linspace(0.35, 0.75, 3),
    'taper': np.linspace(0.40, 0.85, 3),
    'twist': np.linspace(-38.0, -20.0, 3),
    'rpm_hov': np.linspace(420.0, 580.0, 3),
    'rpm_cr': np.linspace(200.0, 360.0, 3)
}

coarse_vectors = list(itertools.product(
    coarse_grid_1d['R'], coarse_grid_1d['c_0'], coarse_grid_1d['taper'],
    coarse_grid_1d['twist'], coarse_grid_1d['rpm_hov'], coarse_grid_1d['rpm_cr']
))

total_cases = len(candidate_airfoils) * len(blade_counts)
print(f"Stage 1: Sweeping {total_cases} topological cases x {len(coarse_vectors)} points = {total_cases * len(coarse_vectors)} evaluations...")

stage1_candidates = []
for af_name in candidate_airfoils:
    if af_name not in _airfoil_cache:
        continue
    for Nb in blade_counts:
        best_fit_case = -1e9
        best_diag_case = None
        for vec in coarse_vectors:
            fit, diag = evaluate_continuous_vector(list(vec), Nb, af_name)
            if diag is not None and fit > best_fit_case:
                best_fit_case = fit
                best_diag_case = diag
        if best_diag_case is not None:
            stage1_candidates.append((best_fit_case, best_diag_case))

print(f"Stage 1 completed in {time.time() - t_start_grid:.1f}s. Survived topological cases: {len(stage1_candidates)}")

# ------------------------------------------------------------------------------
# 4. STAGE 2: FINE LOCAL REFINEMENT SWEEP (TOP 4 CANDIDATES)
# ------------------------------------------------------------------------------
if stage1_candidates:
    stage1_sorted = sorted(stage1_candidates, key=lambda x: x[0], reverse=True)
    top_stage1 = [x[1] for x in stage1_sorted[:4]]
    
    stage2_final_results = []
    print("Stage 2: Running fine local resolution sweep around top configurations...")
    
    for cand in top_stage1:
        af_name = cand["Airfoil"]
        Nb = cand["Nb"]
        
        # Local search window (narrowed around candidate parameters)
        fine_r = np.linspace(max(3.0, cand["Radius R [m]"] - 0.35), min(5.4, cand["Radius R [m]"] + 0.35), 4)
        fine_c0 = np.linspace(max(0.25, cand["Root Chord c0 [m]"] - 0.08), min(0.85, cand["Root Chord c0 [m]"] + 0.08), 4)
        fine_taper = np.linspace(max(0.35, cand["Taper"] - 0.12), min(1.0, cand["Taper"] + 0.12), 3)
        fine_twist = np.linspace(max(-44.0, cand["Twist [°]"] - 4.0), min(-18.0, cand["Twist [°]"] + 4.0), 3)
        fine_rpm_hov = np.linspace(cand["Hover RPM"] - 35.0, cand["Hover RPM"] + 35.0, 3)
        fine_rpm_cr = np.linspace(cand["Cruise RPM"] - 30.0, cand["Cruise RPM"] + 30.0, 3)
        
        fine_vectors = itertools.product(
            fine_r, fine_c0, fine_taper, fine_twist,
            fine_rpm_hov, fine_rpm_cr
        )
        
        best_fine_fit = -1e9
        best_fine_diag = None
        for vec in fine_vectors:
            fit, diag = evaluate_continuous_vector(list(vec), Nb, af_name)
            if diag is not None and fit > best_fine_fit:
                best_fine_fit = fit
                best_fine_diag = diag
                
        if best_fine_diag is not None:
            stage2_final_results.append(best_fine_diag)

    t_elapsed_grid = time.time() - t_start_grid
    print(f"Two-Stage Grid Search completed in {t_elapsed_grid:.1f}s.")

    # ------------------------------------------------------------------------------
    # 5. RANKING & PRESENTATION
    # ------------------------------------------------------------------------------
    df_grid = pd.DataFrame(stage2_final_results).sort_values("Fitness", ascending=False).reset_index(drop=True)
    df_grid.index = [f"Rank #{i+1}" for i in range(len(df_grid))]

    rotor_cols = [
        "Airfoil", "Radius R [m]", "Diameter [m]", "Nb", "Root Chord c0 [m]",
        "Tip Chord [m]", "Taper", "Twist [°]", "Hover Pitch [°]", "Cruise Pitch [°]",
        "Hover RPM", "Cruise RPM"
    ]
    perf_cols = [
        "Hover FM", "Cruise η [%]", "Max α_hov [°]", "Max α_cr [°]", "Heatmap Stall Margin [°]",
        "MTOW [kg]", "Solidity σ", "Disc Loading [kg/m²]", "Hover Req Pwr [kW]", "Cruise Req Pwr [kW]", "Fuselage Clearance [m]"
    ]

    display(HTML("<h3 style='color:#0d6efd;'>Deterministic Grid-Optimized Rotor Designs</h3>"))
    display(HTML(df_grid[rotor_cols].to_html(classes="table table-striped table-hover table-bordered text-center", justify="center")))

    display(HTML("<h3 style='color:#198754; margin-top:14px;'>Aerodynamic & Mission Audit Metrics</h3>"))
    display(HTML(df_grid[perf_cols].to_html(classes="table table-striped table-hover table-bordered text-center", justify="center")))
else:
    print("Zero designs survived Stage 1. Ensure your power or wing area sliders in Cell 2 are not set to under-sized conditions.")

In [ ]:
# ==============================================================================
# CELL 4: MISSION PLANNER & TELEMETRY SIMULATOR
# ==============================================================================
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, HTML
from enum import Enum
from dataclasses import dataclass
from typing import Dict, List

class SegmentType(Enum):
    HOVER = "Hover"
    VERTICAL_CLIMB = "Vertical Climb"
    VERTICAL_DESCENT = "Vertical Descent"
    CRUISE = "Cruise (Axial)"
    LOITER = "Loiter"

@dataclass
class MissionLeg:
    name: str
    seg_type: SegmentType
    duration_min: float = 0.0
    distance_km: float = 0.0
    speed_kmh: float = 0.0
    target_alt_m: float = 0.0
    climb_rate_ms: float = 0.0
    rpm: float = 440.0
    wind_kmh: float = 0.0

MISSION_PRESETS: Dict[str, List[MissionLeg]] = {
    "-- Custom / Blank --": [],
    "Standard 10-Troop Mission (1000 km | 450 km/h)": [
        MissionLeg("1. Takeoff Hover (OGE)", SegmentType.HOVER, duration_min=1.0, target_alt_m=0.0, rpm=440.0),
        MissionLeg("2. Vertical Climb", SegmentType.VERTICAL_CLIMB, target_alt_m=2500.0, climb_rate_ms=5.0, rpm=440.0),
        MissionLeg("3. Outbound Cruise", SegmentType.CRUISE, distance_km=450.0, speed_kmh=450.0, target_alt_m=2500.0, rpm=365.0, wind_kmh=0.0),
        MissionLeg("4. On-Station Loiter", SegmentType.LOITER, duration_min=15.0, speed_kmh=260.0, target_alt_m=2500.0, rpm=365.0),
        MissionLeg("5. Inbound Cruise", SegmentType.CRUISE, distance_km=450.0, speed_kmh=450.0, target_alt_m=2500.0, rpm=365.0, wind_kmh=0.0),
        MissionLeg("6. Descent to Field", SegmentType.VERTICAL_DESCENT, target_alt_m=0.0, climb_rate_ms=-3.5, rpm=440.0),
        MissionLeg("7. Touchdown Hover", SegmentType.HOVER, duration_min=1.0, target_alt_m=0.0, rpm=440.0),
    ],
    "High Altitude Infiltration (7000m Service Ceiling)": [
        MissionLeg("1. Takeoff Hover", SegmentType.HOVER, duration_min=1.0, target_alt_m=0.0, rpm=440.0),
        MissionLeg("2. Climb to Ceiling", SegmentType.VERTICAL_CLIMB, target_alt_m=7000.0, climb_rate_ms=6.0, rpm=440.0),
        MissionLeg("3. High-Altitude Transit", SegmentType.CRUISE, distance_km=400.0, speed_kmh=450.0, target_alt_m=7000.0, rpm=365.0, wind_kmh=15.0),
        MissionLeg("4. Descent to LZ", SegmentType.VERTICAL_DESCENT, target_alt_m=1500.0, climb_rate_ms=-5.0, rpm=440.0),
        MissionLeg("5. LZ Hover & Deploy", SegmentType.HOVER, duration_min=2.0, target_alt_m=1500.0, rpm=440.0),
    ]
}

mission_schedule: List[MissionLeg] = []

w_p_preset = widgets.Dropdown(options=list(MISSION_PRESETS.keys()), value="Standard 10-Troop Mission (1000 km | 450 km/h)", description='Preset:', style={'description_width': '75px'}, layout=widgets.Layout(width='100%'))
w_p_new_type = widgets.Dropdown(options=[(t.value, t) for t in SegmentType], value=SegmentType.CRUISE, description='New Leg:', style={'description_width': '75px'}, layout=widgets.Layout(width='58%'))
btn_p_add = widgets.Button(description="Add Leg", button_style='success', icon='plus', layout=widgets.Layout(width='40%'))

w_p_list = widgets.Select(options=[], layout=widgets.Layout(width='100%', height='115px'))
btn_p_del = widgets.Button(description="Delete Leg", button_style='danger', icon='trash', layout=widgets.Layout(width='48%'))
btn_p_clear = widgets.Button(description="Clear All", button_style='warning', icon='times', layout=widgets.Layout(width='48%'))

# Inspector Controls
w_p_edit_type = widgets.Dropdown(options=[(t.value, t) for t in SegmentType], value=SegmentType.CRUISE, description='Type:', style={'description_width': '85px'}, layout=widgets.Layout(width='100%'))
w_p_name = widgets.Text(value="Cruise Segment", description='Name:', style={'description_width': '85px'}, layout=widgets.Layout(width='100%'))
w_p_dist = widgets.FloatText(value=150.0, description='Dist [km]:', style={'description_width': '85px'}, layout=widgets.Layout(width='49%'))
w_p_dur = widgets.FloatText(value=10.0, description='Dur [min]:', style={'description_width': '85px'}, layout=widgets.Layout(width='49%'))
w_p_speed = widgets.FloatText(value=450.0, description='Speed [km/h]:', style={'description_width': '85px'}, layout=widgets.Layout(width='49%'))
w_p_alt = widgets.FloatText(value=2500.0, description='Alt [m]:', style={'description_width': '85px'}, layout=widgets.Layout(width='49%'))
w_p_climb = widgets.FloatText(value=5.0, description='Climb [m/s]:', style={'description_width': '85px'}, layout=widgets.Layout(width='49%'))
w_p_rpm = widgets.FloatText(value=365.0, description='RPM:', style={'description_width': '85px'}, layout=widgets.Layout(width='49%'))
w_p_wind = widgets.FloatText(value=0.0, description='Wind [km/h]:', style={'description_width': '85px'}, layout=widgets.Layout(width='49%'))

_ui_m_locked = False

def format_leg_str(idx: int, leg: MissionLeg) -> str:
    if leg.seg_type == SegmentType.HOVER:
        return f"{idx+1}. {leg.name} ({leg.duration_min:.1f}m @ {leg.rpm:.0f} RPM)"
    elif leg.seg_type in [SegmentType.VERTICAL_CLIMB, SegmentType.VERTICAL_DESCENT]:
        return f"{idx+1}. {leg.name} (to {leg.target_alt_m:.0f}m @ {leg.climb_rate_ms:.1f}m/s)"
    elif leg.seg_type == SegmentType.CRUISE:
        return f"{idx+1}. {leg.name} ({leg.distance_km:.0f}km @ {leg.speed_kmh:.0f}km/h)"
    elif leg.seg_type == SegmentType.LOITER:
        return f"{idx+1}. {leg.name} ({leg.duration_min:.1f}m @ {leg.speed_kmh:.0f}km/h)"
    return f"{idx+1}. {leg.name}"

def refresh_planner_list(select_idx=None):
    global _ui_m_locked
    _ui_m_locked = True
    w_p_list.options = [format_leg_str(i, leg) for i, leg in enumerate(mission_schedule)]
    if mission_schedule:
        if select_idx is not None and 0 <= select_idx < len(mission_schedule):
            w_p_list.index = select_idx
        elif w_p_list.index is None or w_p_list.index >= len(mission_schedule):
            w_p_list.index = len(mission_schedule) - 1
    else:
        w_p_list.index = None
    _ui_m_locked = False
    load_selected_m_leg()

def load_selected_m_leg(change=None):
    global _ui_m_locked
    if _ui_m_locked or w_p_list.index is None or not (0 <= w_p_list.index < len(mission_schedule)):
        return
    _ui_m_locked = True
    leg = mission_schedule[w_p_list.index]
    w_p_edit_type.value = leg.seg_type
    w_p_name.value = leg.name
    w_p_dist.value = leg.distance_km
    w_p_dur.value = leg.duration_min
    w_p_speed.value = leg.speed_kmh
    w_p_alt.value = leg.target_alt_m
    w_p_climb.value = leg.climb_rate_ms
    w_p_rpm.value = leg.rpm
    w_p_wind.value = leg.wind_kmh
    _ui_m_locked = False

w_p_list.observe(load_selected_m_leg, names='index')

def on_m_field_edited(change=None):
    global _ui_m_locked
    if _ui_m_locked or w_p_list.index is None or not (0 <= w_p_list.index < len(mission_schedule)):
        return
    idx = w_p_list.index
    leg = mission_schedule[idx]
    leg.seg_type = w_p_edit_type.value
    leg.name = w_p_name.value.strip() or f"Leg {idx+1}"
    leg.distance_km = float(w_p_dist.value)
    leg.duration_min = float(w_p_dur.value)
    leg.speed_kmh = float(w_p_speed.value)
    leg.target_alt_m = float(w_p_alt.value)
    leg.climb_rate_ms = float(w_p_climb.value)
    leg.rpm = float(w_p_rpm.value)
    leg.wind_kmh = float(w_p_wind.value)

    _ui_m_locked = True
    current_opts = list(w_p_list.options)
    current_opts[idx] = format_leg_str(idx, leg)
    w_p_list.options = current_opts
    _ui_m_locked = False
    run_mission_simulation()

for w in [w_p_edit_type, w_p_name, w_p_dist, w_p_dur, w_p_speed, w_p_alt, w_p_climb, w_p_rpm, w_p_wind]:
    w.observe(on_m_field_edited, names='value')

def on_add_m_clicked(b):
    st = w_p_new_type.value
    idx = len(mission_schedule) + 1
    new_leg = MissionLeg(
        name=f"Leg {idx}", seg_type=st,
        duration_min=10.0 if st in [SegmentType.HOVER, SegmentType.LOITER] else 0.0,
        distance_km=200.0 if st == SegmentType.CRUISE else 0.0,
        speed_kmh=450.0 if st == SegmentType.CRUISE else (260.0 if st == SegmentType.LOITER else 0.0),
        target_alt_m=2500.0 if st != SegmentType.HOVER else 0.0,
        climb_rate_ms=5.0 if st == SegmentType.VERTICAL_CLIMB else (-3.5 if st == SegmentType.VERTICAL_DESCENT else 0.0),
        rpm=365.0 if st in [SegmentType.CRUISE, SegmentType.LOITER] else 440.0,
        wind_kmh=0.0
    )
    mission_schedule.append(new_leg)
    refresh_planner_list(select_idx=len(mission_schedule) - 1)
    run_mission_simulation()

def on_del_m_clicked(b):
    if w_p_list.index is not None and len(mission_schedule) > 0:
        cur = w_p_list.index
        mission_schedule.pop(cur)
        refresh_planner_list(select_idx=max(0, cur - 1) if len(mission_schedule) > 0 else None)
        run_mission_simulation()

def on_clear_m_clicked(b):
    mission_schedule.clear()
    refresh_planner_list()
    run_mission_simulation()

def on_preset_m_chosen(change):
    chosen = w_p_preset.value
    mission_schedule.clear()
    for leg in MISSION_PRESETS[chosen]:
        mission_schedule.append(MissionLeg(
            name=leg.name, seg_type=leg.seg_type, duration_min=leg.duration_min,
            distance_km=leg.distance_km, speed_kmh=leg.speed_kmh, target_alt_m=leg.target_alt_m,
            climb_rate_ms=leg.climb_rate_ms, rpm=leg.rpm, wind_kmh=leg.wind_kmh
        ))
    refresh_planner_list(select_idx=0 if len(mission_schedule) > 0 else None)
    run_mission_simulation()

btn_p_add.on_click(on_add_m_clicked)
btn_p_del.on_click(on_del_m_clicked)
btn_p_clear.on_click(on_clear_m_clicked)
w_p_preset.observe(on_preset_m_chosen, names='value')

out_m_profile = widgets.Output()
out_m_table = widgets.Output()
m_tabs = widgets.Tab(children=[out_m_profile, out_m_table])
m_tabs.set_title(0, "Mission Flight Profile & L/D")
m_tabs.set_title(1, "Mission Telemetry Table")

def run_mission_simulation(*args):
    # Fallback default values if SIZED_VEHICLE is not yet generated
    veh = globals().get("SIZED_VEHICLE", {
        "MTOW": 9500.0, "Payload": 1400.0, "Wing_Area": 26.0,
        "Aspect_Ratio": 7.8, "Rotor_Radius": 4.20, "Installed_Power": 5000.0,
        "CD0": 0.019, "Oswald_e": 0.85, "SFC": 0.285
    })
    
    mtow = veh["MTOW"]
    m_empty = mtow * 0.58
    m_fuel_init = mtow * 0.28
    payload = veh["Payload"]
    reserve_fuel = 200.0
    s_wing = veh["Wing_Area"]
    ar_wing = veh["Aspect_Ratio"]
    r_rotor = veh["Rotor_Radius"]
    p_inst_kw = veh["Installed_Power"]
    cd0 = veh["CD0"]
    oswald_e = veh["Oswald_e"]
    sfc_hr = veh["SFC"]

    curr_t, curr_alt, curr_fuel, curr_dist = 0.0, 0.0, m_fuel_init, 0.0
    telemetry = []
    failed, fail_msg = False, ""
    dt = 2.0

    for leg in mission_schedule:
        start_alt = curr_alt
        if leg.seg_type in [SegmentType.HOVER, SegmentType.LOITER]:
            dur_s = max(leg.duration_min * 60.0, 1.0)
            v_tas = (leg.speed_kmh / 3.6) if leg.seg_type == SegmentType.LOITER else 0.0
            v_ground = v_tas
            alt_target = curr_alt if leg.seg_type == SegmentType.HOVER else leg.target_alt_m
        elif leg.seg_type in [SegmentType.VERTICAL_CLIMB, SegmentType.VERTICAL_DESCENT]:
            climb_speed = abs(leg.climb_rate_ms) if abs(leg.climb_rate_ms) > 0.1 else 3.0
            dur_s = max(abs(leg.target_alt_m - curr_alt) / climb_speed, 1.0)
            v_tas = climb_speed
            v_ground = 0.0
            alt_target = leg.target_alt_m
        elif leg.seg_type == SegmentType.CRUISE:
            v_tas = max(leg.speed_kmh / 3.6, 15.0)
            v_ground = max(v_tas - (leg.wind_kmh / 3.6), 5.0)
            dur_s = max(leg.distance_km * 1000.0, 100.0) / v_ground
            alt_target = leg.target_alt_m

        n_steps = max(int(np.ceil(dur_s / dt)), 1)
        step_dt = dur_s / n_steps
        alt_rate = (alt_target - start_alt) / dur_s if dur_s > 0 else 0.0

        for _ in range(n_steps):
            gross_m = m_empty + curr_fuel + payload
            w_newtons = gross_m * 9.80665
            rho, a_sound, _, mu_curr = isa_atmosphere(curr_alt)
            omega = (leg.rpm * 2.0 * np.pi) / 60.0
            v_tip = omega * r_rotor
            m_tip = np.sqrt(v_tip**2 + v_tas**2) / a_sound

            if leg.seg_type in [SegmentType.HOVER, SegmentType.VERTICAL_CLIMB, SegmentType.VERTICAL_DESCENT]:
                download_factor = 1.08
                T_hover_total_N = gross_m * 9.80665 * download_factor
                P_induced_W = (T_hover_total_N ** 1.5) / np.sqrt(2.0 * rho * (2 * np.pi * (r_rotor**2)))
                p_req_kw = (P_induced_W / 0.74) / (0.94 * 1000.0)
                current_LD = 0.0
            else:
                q_dyn = 0.5 * rho * (v_tas ** 2)
                cl = w_newtons / (q_dyn * s_wing)
                cd_ind = (cl ** 2) / (np.pi * ar_wing * oswald_e)
                cd_total = cd0 + cd_ind
                p_req_kw = (q_dyn * s_wing * cd_total * v_tas) / (0.83 * 1000.0)
                current_LD = cl / cd_total

            p_avail_kw = p_inst_kw * ((rho / 1.225) ** 1.05)
            fuel_burn = p_req_kw * (sfc_hr / 3600.0) * step_dt

            if p_req_kw > p_avail_kw:
                failed, fail_msg = True, f"Power limit reached in {leg.name}"
                break
            if curr_fuel <= reserve_fuel:
                failed, fail_msg = True, f"Reserve fuel breached in {leg.name}"
                break

            curr_fuel -= fuel_burn
            curr_alt += alt_rate * step_dt
            curr_dist += (v_ground * step_dt) / 1000.0
            curr_t += step_dt

            telemetry.append({
                "time_min": curr_t / 60.0, "leg": leg.name, "type": leg.seg_type.value, "alt_m": curr_alt,
                "gross_kg": gross_m, "fuel_kg": curr_fuel, "payload_kg": payload,
                "p_req_kw": p_req_kw, "p_avail_kw": p_avail_kw, "speed_kmh": v_tas * 3.6,
                "dist_km": curr_dist, "mach_tip": m_tip, "L_over_D": current_LD
            })

        if failed:
            break

    df_res = pd.DataFrame(telemetry)

    out_m_profile.clear_output(wait=True)
    with out_m_profile:
        if not df_res.empty:
            fig, (ax1, ax2, ax3) = plt.subplots(3, 1, figsize=(9.5, 6.0), dpi=100, sharex=True)
            ax1.plot(df_res["time_min"], df_res["alt_m"], 'b-', lw=2, label='Altitude [m]')
            ax1.set_ylabel("Altitude [m]", color='b')
            ax1.set_title("Mission Telemetry Profile & Aerodynamic Efficiency", fontsize=10, fontweight='bold')
            ax1.grid(True, ls=":", alpha=0.6)

            ax1_twin = ax1.twinx()
            ax1_twin.plot(df_res["time_min"], df_res["speed_kmh"], 'orange', ls='--', lw=1.5)
            ax1_twin.set_ylabel("Airspeed [km/h]", color='orange')

            ax2.plot(df_res["time_min"], df_res["L_over_D"], 'purple', lw=2, label='L/D Ratio')
            ax2.set_ylabel("L/D Ratio")
            ax2.grid(True, ls=":", alpha=0.6)
            ax2.legend(loc='upper right', fontsize=8)

            ax3.plot(df_res["time_min"], df_res["fuel_kg"], 'r--', lw=1.8, label='Fuel Left [kg]')
            ax3.axhline(reserve_fuel, color='crimson', ls=':', lw=1.5, label='Reserve Limit')
            ax3.set_ylabel("Fuel [kg]")
            ax3.set_xlabel("Time [minutes]")
            ax3.grid(True, ls=":", alpha=0.6)
            ax3.legend(loc='center right', fontsize=8)

            plt.tight_layout()
            plt.show()
        else:
            print("No mission data available. Select a preset or add legs.")

    out_m_table.clear_output(wait=True)
    with out_m_table:
        if not df_res.empty:
            sample_df = df_res.iloc[::max(1, len(df_res)//12)].copy()
            display(HTML(sample_df.to_html(index=False, classes="table table-striped table-bordered", float_format="%.1f")))

m_inspector = widgets.VBox([
    w_p_edit_type, w_p_name,
    widgets.HBox([w_p_alt, w_p_rpm]),
    widgets.HBox([w_p_dist, w_p_dur]),
    widgets.HBox([w_p_speed, w_p_climb]),
    widgets.HBox([w_p_wind])
], layout=widgets.Layout(margin='4px 0'))

m_sidebar = widgets.VBox([
    widgets.HTML("<b>Mission Planner Controls</b>"),
    w_p_preset,
    widgets.HBox([w_p_new_type, btn_p_add]),
    m_inspector,
    w_p_list,
    widgets.HBox([btn_p_del, btn_p_clear])
], layout=widgets.Layout(width='36%', padding='6px', border='1px solid #ced4da', border_radius='6px'))

display(widgets.HBox([m_sidebar, m_tabs], layout=widgets.Layout(width='100%', align_items='flex-start')))
refresh_planner_list()
run_mission_simulation()